# Quality, patient aggregation, and domain-shift audit

**Public role:** Core quality/patient/source-shift and shortcut-forensics notebook.

**Relation to the manuscript:** Supports quality strata, patient aggregation, duplicate forensics, source transportability, and outside-OD analyses.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `14_hygd_smdg_domain_shift_audit_reviewer_ready_v36_extensions_cell11fix.ipynb`


In [ ]:
import os
# Fast resume for Notebook 14 v3.6 extensions
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive already mounted.")
except Exception as e:
    print(f"Drive mount warning: {e}")

PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
RESULTS_ROOT = PROJECT_ROOT / "results"

NOTEBOOK_NAME = "14_hygd_smdg_domain_shift_audit"
RESULTS_14_DIR = RESULTS_ROOT / NOTEBOOK_NAME
TABLE_DIR = RESULTS_14_DIR / "tables"
FIGURE_DATA_DIR = RESULTS_14_DIR / "figure_data"
HANDOFF_DIR = RESULTS_14_DIR / "handoff"

for path in [RESULTS_14_DIR, TABLE_DIR, FIGURE_DATA_DIR, HANDOFF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PRIMARY_MODEL_NAME = "EfficientNetB0"
AQPR_PANEL_MODEL_NAMES = ["EfficientNetB0", "DenseNet121", "SwinTiny", "ConvNeXtTiny"]

REVIEWER_SOURCE_MAP_CSV = TABLE_DIR / "notebook14_reviewer_required_source_map.csv"
HYGD_UNIFIED_BASE_CSV = TABLE_DIR / "notebook14_hygd_unified_aqpr_base_manifest.csv"

required_resume_files = [
    REVIEWER_SOURCE_MAP_CSV,
    HYGD_UNIFIED_BASE_CSV,
]

missing = [str(p) for p in required_resume_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Fast resume is not possible because these previous Notebook 14 outputs are missing:\n"
        + "\n".join(missing)
        + "\n\nIn that case, you must run Cells 1–10 once."
    )

print("Fast resume ready.")
print(f"Reviewer source map: {REVIEWER_SOURCE_MAP_CSV}")
print(f"HYGD unified base:   {HYGD_UNIFIED_BASE_CSV}")
print(f"Tables dir:          {TABLE_DIR}")
print(f"Figure data dir:     {FIGURE_DATA_DIR}")

In [ ]:
# Cell 2: Select HYGD and SMDG source-of-truth artifacts

import re

required_vars = [
    "TABLE_DIR",
    "RESULTS_14_DIR",
    "RESULTS_ROOT",
    "PRIMARY_MODEL_NAME",
    "AQPR_PANEL_MODEL_NAMES",
    "SMDG_ARTIFACT_REGISTRY_CSV",
    "HYGD_ARTIFACT_DISCOVERY_CSV",
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run Cell 1 first. Missing variables: {missing_vars}")


HYGD_ROLE_SCORING_CSV = TABLE_DIR / "notebook14_hygd_role_scoring.csv"
HYGD_SELECTED_SOURCES_CSV = TABLE_DIR / "notebook14_hygd_selected_source_of_truth_artifacts.csv"
SMDG_SELECTED_SOURCES_CSV = TABLE_DIR / "notebook14_smdg_selected_source_of_truth_artifacts.csv"
CROSS_EXTERNAL_REGISTRY_CSV = TABLE_DIR / "notebook14_cross_external_source_of_truth_registry.csv"
CELL2_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell02_source_of_truth_selection_summary.json"
CELL2_SUMMARY_TXT = RESULTS_14_DIR / "notebook14_CELL02_SOURCE_OF_TRUTH_SELECTION_SUMMARY.txt"


SMDG_ROLE_MAP = {
    "smdg_prediction_source": "smdg_predictions_all_models",
    "smdg_global_metric_source": "smdg_global_metrics",
    "smdg_global_ci_source": "smdg_global_ci",
    "smdg_native_metric_source": "smdg_native_metrics",
    "smdg_native_ci_source": "smdg_native_ci",
    "smdg_source_metric_source": "smdg_source_metrics",
    "smdg_source_ci_source": "smdg_source_ci",
    "smdg_domain_shift_source": "smdg_domain_shift_audit",
    "smdg_model_ranking_source": "smdg_model_ranking",
    "smdg_paired_comparison_source": "smdg_paired_comparison",
    "smdg_calibration_table_source": "smdg_table_calibration",
}


HYGD_ROLE_THRESHOLDS = {
    "hygd_prediction_source": 100,
    "hygd_global_metric_source": 90,
    "hygd_patient_metric_source": 90,
    "hygd_calibration_source": 90,
    "hygd_anatomical_xai_source": 100,
    "hygd_ablation_source": 100,
}


def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def csv_shape(path):
    try:
        preview = pd.read_csv(path, nrows=5, low_memory=False)
        n_cols = len(preview.columns)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            n_rows = max(sum(1 for _ in f) - 1, 0)
        return int(n_rows), int(n_cols)
    except Exception:
        return None, None


def read_header(path):
    try:
        return list(pd.read_csv(path, nrows=0, low_memory=False).columns)
    except Exception:
        return []


def read_preview(path, nrows=3000):
    try:
        return pd.read_csv(path, nrows=nrows, low_memory=False)
    except Exception:
        return pd.DataFrame()


def has_token(text, tokens):
    return any(t in text for t in tokens)


def detect_features(columns):
    cols = [norm_text(c) for c in columns]
    text = " ".join(cols)

    return {
        "model": has_token(text, ["model", "architecture", "backbone"]),
        "label": has_token(text, ["label", "target", "y_true", "ground_truth", "gon", "glaucoma"]),
        "probability": has_token(text, ["prob", "probability", "p_positive", "score", "confidence", "logit"]),
        "prediction": has_token(text, ["pred", "prediction", "y_pred"]),
        "patient": has_token(text, ["patient", "subject"]),
        "metric": has_token(text, ["auroc", "auc", "auprc", "accuracy", "balanced", "sensitivity", "specificity", "precision", "f1"]),
        "calibration": has_token(text, ["ece", "brier", "calibration", "reliability"]),
        "anatomy": has_token(text, ["anatom", "od", "oc", "disc", "cup", "heatmap", "enrichment", "xai", "gradcam", "integrated"]),
        "ablation": has_token(text, ["ablation", "occlusion", "shortcut", "masked", "outside", "inside"]),
    }


def panel_models_in_file(path):
    df = read_preview(path)

    if df.empty:
        return []

    model_cols = [
        c for c in df.columns
        if has_token(norm_text(c), ["model", "architecture", "backbone"])
    ]

    found = set()

    for col in model_cols:
        values = set(df[col].dropna().astype(str).unique())

        for model_name in AQPR_PANEL_MODEL_NAMES:
            if model_name in values:
                found.add(model_name)

    return sorted(found)


def role_scores(path, n_rows, features, found_models):
    name = norm_text(path.name)
    full = norm_text(str(path))
    scores = {role: -999 for role in HYGD_ROLE_THRESHOLDS}

    hard_exclude = [
        "papila",
        "smdg",
        "model_registry",
        "model_panel",
        "manifest",
        "case_log",
        "visual",
        "progress",
        "plan",
        "candidate",
        "path_audit",
    ]

    if has_token(full, hard_exclude):
        return scores

    panel_bonus = 20 if PRIMARY_MODEL_NAME in found_models else 0
    panel_bonus += 10 * len(found_models)
    panel_bonus += 20 if len(found_models) == len(AQPR_PANEL_MODEL_NAMES) else 0

    if (
        n_rows is not None
        and n_rows >= 100
        and features["model"]
        and features["label"]
        and features["probability"]
        and not features["anatomy"]
        and not features["ablation"]
        and has_token(name, ["prediction", "predictions", "inference"])
    ):
        scores["hygd_prediction_source"] = 100 + panel_bonus

    if (
        features["model"]
        and features["metric"]
        and not features["anatomy"]
        and not features["ablation"]
        and has_token(name, ["metric", "metrics", "performance", "evaluation"])
        and not has_token(name, ["screening", "registry", "panel"])
    ):
        scores["hygd_global_metric_source"] = 90 + panel_bonus

    if (
        features["patient"]
        and features["metric"]
        and not features["anatomy"]
        and not features["ablation"]
        and has_token(name, ["patient", "metrics", "performance"])
        and not has_token(name, ["screening", "registry", "panel"])
    ):
        scores["hygd_patient_metric_source"] = 90 + panel_bonus

    if (
        features["calibration"]
        and has_token(name, ["calibration", "probability", "reliability", "ece", "brier"])
        and not has_token(name, ["registry", "panel"])
    ):
        scores["hygd_calibration_source"] = 90 + panel_bonus

    if (
        features["anatomy"]
        and has_token(name, ["xai", "heatmap", "anatomical", "gradcam", "integrated"])
        and not has_token(name, ["random_control"])
    ):
        scores["hygd_anatomical_xai_source"] = 100 + panel_bonus

    if path.name == "notebook08_aqpr_heatmap_xai_metrics_consolidated.csv":
        scores["hygd_anatomical_xai_source"] = 300 + panel_bonus

    if (
        features["ablation"]
        or has_token(name, ["ablation", "occlusion", "shortcut"])
    ):
        if not has_token(name, ["case_log", "manifest", "visual"]):
            scores["hygd_ablation_source"] = 100 + panel_bonus

    if path.name == "notebook09_full_aqpr_ablation_delta_summary_vs_original.csv":
        scores["hygd_ablation_source"] = 300 + panel_bonus

    return scores


def get_smdg_source(registry, role, artifact_id):
    row = registry[registry["artifact_id"].eq(artifact_id)]

    if len(row) != 1:
        raise RuntimeError(f"SMDG artifact not found: {artifact_id}")

    row = row.iloc[0]

    if not bool(row["exists"]):
        raise RuntimeError(f"SMDG artifact exists=False: {artifact_id}")

    return {
        "dataset": "SMDG",
        "role": role,
        "artifact_id": artifact_id,
        "selected": True,
        "path": row["path"],
        "file_name": Path(str(row["path"])).name,
        "n_rows": row["n_rows"],
        "n_columns": row["n_columns"],
        "score": np.nan,
        "panel_models_found": "",
        "notes": "Locked Notebook 13 artifact.",
    }


if not SMDG_ARTIFACT_REGISTRY_CSV.exists():
    raise FileNotFoundError("Missing SMDG registry from Cell 1.")

if not HYGD_ARTIFACT_DISCOVERY_CSV.exists():
    raise FileNotFoundError("Missing HYGD discovery table from Cell 1.")


smdg_registry = pd.read_csv(SMDG_ARTIFACT_REGISTRY_CSV)
hygd_discovery = pd.read_csv(HYGD_ARTIFACT_DISCOVERY_CSV)

candidate_paths = set()

for path_str in hygd_discovery["path"].dropna().astype(str):
    path = Path(path_str)
    if path.exists() and path.suffix.lower() == ".csv":
        candidate_paths.add(path)

for pattern in [
    "*hygd*.csv",
    "*HYGD*.csv",
    "*notebook07*.csv",
    "*notebook08*.csv",
    "*notebook09*.csv",
    "*notebook12*.csv",
    "*notebook16*.csv",
]:
    for path in RESULTS_ROOT.rglob(pattern):
        if path.exists() and path.suffix.lower() == ".csv":
            candidate_paths.add(path)


scoring_rows = []

for path in sorted(candidate_paths):
    full = str(path).lower()

    if "papila" in full or "smdg" in full:
        continue

    columns = read_header(path)

    if len(columns) == 0:
        continue

    n_rows, n_cols = csv_shape(path)
    features = detect_features(columns)
    found_models = panel_models_in_file(path)
    scores = role_scores(path, n_rows, features, found_models)

    row = {
        "path": str(path),
        "file_name": path.name,
        "n_rows": n_rows,
        "n_columns": n_cols,
        "file_size_bytes": path.stat().st_size,
        "n_panel_models_found": len(found_models),
        "panel_models_found": ";".join(found_models),
        "columns_preview": " | ".join(columns[:30]),
    }

    for key, value in features.items():
        row[f"has_{key}"] = value

    row.update(scores)
    scoring_rows.append(row)


hygd_scoring = pd.DataFrame(scoring_rows)

if hygd_scoring.empty:
    raise RuntimeError("No HYGD CSV artifacts could be scored.")

score_cols = list(HYGD_ROLE_THRESHOLDS.keys())

hygd_scoring = hygd_scoring.sort_values(
    score_cols + ["file_size_bytes"],
    ascending=[False] * len(score_cols) + [False],
).reset_index(drop=True)

hygd_scoring.to_csv(HYGD_ROLE_SCORING_CSV, index=False)


selected_rows = []

for role, threshold in HYGD_ROLE_THRESHOLDS.items():
    candidates = hygd_scoring[hygd_scoring[role] >= threshold].copy()

    if candidates.empty:
        selected_rows.append(
            {
                "dataset": "HYGD",
                "role": role,
                "artifact_id": "",
                "selected": False,
                "path": "",
                "file_name": "",
                "n_rows": np.nan,
                "n_columns": np.nan,
                "score": np.nan,
                "panel_models_found": "",
                "notes": f"No valid artifact reached threshold={threshold}.",
            }
        )
        continue

    candidates = candidates.sort_values(
        [role, "n_panel_models_found", "file_size_bytes"],
        ascending=[False, False, False],
    )

    best = candidates.iloc[0]

    selected_rows.append(
        {
            "dataset": "HYGD",
            "role": role,
            "artifact_id": "",
            "selected": True,
            "path": best["path"],
            "file_name": best["file_name"],
            "n_rows": best["n_rows"],
            "n_columns": best["n_columns"],
            "score": best[role],
            "panel_models_found": best["panel_models_found"],
            "notes": "Selected by strict role-specific scoring.",
        }
    )


hygd_selected = pd.DataFrame(selected_rows)
hygd_selected.to_csv(HYGD_SELECTED_SOURCES_CSV, index=False)


smdg_selected = pd.DataFrame(
    [
        get_smdg_source(smdg_registry, role, artifact_id)
        for role, artifact_id in SMDG_ROLE_MAP.items()
    ]
)

smdg_selected.to_csv(SMDG_SELECTED_SOURCES_CSV, index=False)


cross_registry = pd.concat(
    [hygd_selected, smdg_selected],
    ignore_index=True,
    sort=False,
)

cross_registry.to_csv(CROSS_EXTERNAL_REGISTRY_CSV, index=False)


hygd_selected_count = int(hygd_selected["selected"].sum())

required_hygd_roles = [
    "hygd_anatomical_xai_source",
    "hygd_ablation_source",
]

required_ready = bool(
    hygd_selected[hygd_selected["role"].isin(required_hygd_roles)]["selected"].all()
)

performance_ready = bool(
    hygd_selected[
        hygd_selected["role"].isin(
            ["hygd_prediction_source", "hygd_global_metric_source"]
        )
    ]["selected"].any()
)

cell2_status = "PASS" if required_ready else "PASS_WITH_HYGD_SELECTION_WARNING"

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 2",
    "status": cell2_status,
    "n_hygd_artifacts_scored": int(len(hygd_scoring)),
    "n_hygd_roles_selected": hygd_selected_count,
    "required_hygd_roles_ready": required_ready,
    "hygd_performance_source_ready": performance_ready,
    "hygd_role_scoring_csv": str(HYGD_ROLE_SCORING_CSV),
    "hygd_selected_sources_csv": str(HYGD_SELECTED_SOURCES_CSV),
    "smdg_selected_sources_csv": str(SMDG_SELECTED_SOURCES_CSV),
    "cross_external_registry_csv": str(CROSS_EXTERNAL_REGISTRY_CSV),
}

with open(CELL2_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

with open(CELL2_SUMMARY_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 14 Cell 2 source-of-truth selection\n")
    f.write("============================================\n")
    f.write(f"Status: {cell2_status}\n")
    f.write(f"HYGD artifacts scored: {len(hygd_scoring)}\n")
    f.write(f"HYGD roles selected: {hygd_selected_count}/{len(HYGD_ROLE_THRESHOLDS)}\n")
    f.write(f"Required HYGD roles ready: {required_ready}\n")
    f.write(f"HYGD performance source ready: {performance_ready}\n")


print("HYGD selected sources:")
print(
    hygd_selected[
        [
            "role",
            "selected",
            "score",
            "n_rows",
            "n_columns",
            "panel_models_found",
            "file_name",
            "path",
        ]
    ].to_string(index=False)
)

print("\nSMDG selected sources:")
print(
    smdg_selected[
        [
            "role",
            "artifact_id",
            "n_rows",
            "n_columns",
            "file_name",
            "path",
        ]
    ].to_string(index=False)
)

print("\nTop HYGD candidates by role:")
for role in HYGD_ROLE_THRESHOLDS:
    candidates = hygd_scoring[hygd_scoring[role] > 0].copy()
    print(f"\n{role}:")
    if candidates.empty:
        print("No valid candidates.")
    else:
        print(
            candidates.sort_values(role, ascending=False)[
                [
                    "file_name",
                    role,
                    "n_rows",
                    "n_columns",
                    "n_panel_models_found",
                    "panel_models_found",
                    "path",
                ]
            ].head(10).to_string(index=False)
        )

print("\nCell 2 summary:")
print(json.dumps(summary, indent=2))

for output_path in [
    HYGD_ROLE_SCORING_CSV,
    HYGD_SELECTED_SOURCES_CSV,
    SMDG_SELECTED_SOURCES_CSV,
    CROSS_EXTERNAL_REGISTRY_CSV,
    CELL2_SUMMARY_JSON,
    CELL2_SUMMARY_TXT,
]:
    if not Path(output_path).exists():
        raise RuntimeError(f"Missing Cell 2 output: {output_path}")

print(f"Status: {cell2_status}")

In [ ]:
# Cell 2B: Clean HYGD source registry after automatic selection

required_vars = [
    "HYGD_SELECTED_SOURCES_CSV",
    "CROSS_EXTERNAL_REGISTRY_CSV",
    "TABLE_DIR",
    "RESULTS_14_DIR",
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run Cell 2 first. Missing variables: {missing_vars}")


HYGD_SELECTED_SOURCES_CLEAN_CSV = TABLE_DIR / "notebook14_hygd_selected_source_of_truth_artifacts_clean.csv"
CROSS_EXTERNAL_REGISTRY_CLEAN_CSV = TABLE_DIR / "notebook14_cross_external_source_of_truth_registry_clean.csv"
CELL2B_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell02b_clean_source_registry_summary.json"

invalid_hygd_files = [
    "notebook07_patient_safe_split.csv",
    "notebook08_aqpr_model_registry_for_xai.csv",
    "notebook09_aqpr_model_panel_for_ablation.csv",
    "notebook07_architecture_screening_results_consolidated.csv",
]

hygd_selected = pd.read_csv(HYGD_SELECTED_SOURCES_CSV)
cross_registry = pd.read_csv(CROSS_EXTERNAL_REGISTRY_CSV)

for df in [hygd_selected, cross_registry]:
    invalid_mask = (
        df["dataset"].astype(str).eq("HYGD")
        & df["file_name"].astype(str).isin(invalid_hygd_files)
    )

    df.loc[invalid_mask, "selected"] = False
    df.loc[invalid_mask, "score"] = np.nan
    df.loc[invalid_mask, "path"] = ""
    df.loc[invalid_mask, "file_name"] = ""
    df.loc[invalid_mask, "n_rows"] = np.nan
    df.loc[invalid_mask, "n_columns"] = np.nan
    df.loc[invalid_mask, "panel_models_found"] = ""
    df.loc[invalid_mask, "notes"] = "Invalidated: registry, split, model-panel, or architecture-screening file is not an analytical source."

hygd_selected.to_csv(HYGD_SELECTED_SOURCES_CLEAN_CSV, index=False)
cross_registry.to_csv(CROSS_EXTERNAL_REGISTRY_CLEAN_CSV, index=False)

valid_hygd_roles = hygd_selected[
    hygd_selected["dataset"].astype(str).eq("HYGD")
    & hygd_selected["selected"].astype(bool)
]["role"].tolist()

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 2B",
    "status": "PASS",
    "valid_hygd_roles_after_cleaning": valid_hygd_roles,
    "hygd_performance_source_ready": bool(
        set(valid_hygd_roles).intersection(
            {"hygd_prediction_source", "hygd_global_metric_source"}
        )
    ),
    "hygd_anatomical_xai_ready": "hygd_anatomical_xai_source" in valid_hygd_roles,
    "hygd_ablation_ready": "hygd_ablation_source" in valid_hygd_roles,
    "hygd_selected_sources_clean_csv": str(HYGD_SELECTED_SOURCES_CLEAN_CSV),
    "cross_external_registry_clean_csv": str(CROSS_EXTERNAL_REGISTRY_CLEAN_CSV),
}

with open(CELL2B_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Clean HYGD selected sources:")
print(
    hygd_selected[
        ["role", "selected", "score", "n_rows", "n_columns", "panel_models_found", "file_name", "path", "notes"]
    ].to_string(index=False)
)

print("\nCell 2B summary:")
print(json.dumps(summary, indent=2))

print(f"\nClean HYGD selected sources CSV: {HYGD_SELECTED_SOURCES_CLEAN_CSV}")
print(f"Clean cross-external registry CSV: {CROSS_EXTERNAL_REGISTRY_CLEAN_CSV}")
print("Status: PASS")

## Reviewer-driven extension

The following cells extend Notebook 14 from a HYGD/SMDG source audit into a reviewer-facing reliability package. The added analyses target the second external review: dataset forensics, HYGD split/patient leakage checks, exact and perceptual duplicate screening, quality-stratified AQPR endpoints, outside-OD shortcut-control forensics, SMDG label/source documentation, compact figure generation, and manuscript handoff artifacts.

No model training, model selection, threshold tuning, or SMDG calibration fitting is performed in this notebook. Any output that depends on unavailable upstream artifacts is recorded as `MISSING_UPSTREAM_ARTIFACT` rather than silently fabricated.


In [ ]:
# Cell 3: Reviewer-required source map for HYGD, AIROGS-light, SMDG, and outside-OD controls

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

required_vars = [
    "PROJECT_ROOT",
    "RESULTS_ROOT",
    "RESULTS_14_DIR",
    "TABLE_DIR",
    "FIGURE_DATA_DIR",
    "HANDOFF_DIR",
    "PRIMARY_MODEL_NAME",
    "AQPR_PANEL_MODEL_NAMES",
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run setup cells first. Missing variables: {missing_vars}")

REVIEWER_SOURCE_MAP_CSV = TABLE_DIR / "notebook14_reviewer_required_source_map.csv"
REVIEWER_SOURCE_AUDIT_JSON = RESULTS_14_DIR / "notebook14_cell03_reviewer_source_map_summary.json"

SEARCH_ROOTS = [PROJECT_ROOT, RESULTS_ROOT]

REVIEWER_SOURCE_SPECS = {
    # HYGD metadata, split, quality, and OD/OC proxy.
    "hygd_metadata_clean": {
        "required": True,
        "exact_names": ["notebook01_metadata_clean.csv", "metadata_clean.csv"],
        "include_tokens": ["metadata", "clean"],
        "exclude_tokens": ["smdg", "papila", "path_registry"],
    },
    "hygd_image_split": {
        "required": True,
        "exact_names": ["notebook02_train_val_test_image_split.csv", "train_val_test_image_split.csv"],
        "include_tokens": ["train", "val", "test", "image", "split"],
        "exclude_tokens": ["smdg", "papila", "groupkfold", "patient"],
    },
    "hygd_quality_manifest": {
        "required": True,
        "exact_names": ["preprocessing_quality_manifest.csv", "notebook03_preprocessing_quality_manifest.csv"],
        "include_tokens": ["quality", "manifest"],
        "exclude_tokens": ["smdg", "papila", "summary", "tertile_summary", "split_quality_summary"],
    },
    "hygd_proxy_analysis_manifest": {
        "required": True,
        "exact_names": [
            "notebook06_hygd_final_od_oc_proxy_analysis_manifest.csv",
            "hygd_final_od_oc_proxy_analysis_manifest.csv",
        ],
        "include_tokens": ["hygd", "proxy", "analysis", "manifest"],
        "exclude_tokens": ["smdg", "papila", "patient_summary", "figure", "visual"],
    },
    "hygd_proxy_qc_manifest": {
        "required": False,
        "exact_names": [
            "notebook06_hygd_final_od_oc_proxy_manifest_with_qc.csv",
            "hygd_final_od_oc_proxy_manifest_with_qc.csv",
        ],
        "include_tokens": ["proxy", "qc"],
        "exclude_tokens": ["smdg", "papila", "summary", "patient"],
    },
    "hygd_proxy_patient_summary": {
        "required": False,
        "exact_names": [
            "notebook06_hygd_final_od_oc_proxy_patient_summary.csv",
            "hygd_final_od_oc_proxy_patient_summary.csv",
        ],
        "include_tokens": ["proxy", "patient", "summary"],
        "exclude_tokens": ["smdg", "papila"],
    },
    # AIROGS-light documentation.
    "airogs_light_split_label_summary": {
        "required": True,
        "exact_names": ["notebook07_airogs_light_split_label_summary.csv"],
        "include_tokens": ["airogs", "split", "label", "summary"],
        "exclude_tokens": ["smdg", "hygd", "papila"],
    },
    "airogs_light_source_summary": {
        "required": False,
        "exact_names": ["notebook07_airogs_light_source_summary.csv"],
        "include_tokens": ["airogs", "source", "summary"],
        "exclude_tokens": ["smdg", "hygd", "papila"],
    },
    "airogs_light_model_panel_manifest": {
        "required": False,
        "exact_names": ["aqpr_model_panel_manifest.csv", "notebook07_frozen_aqpr_model_panel_manifest.csv"],
        "include_tokens": ["aqpr", "model", "panel", "manifest"],
        "exclude_tokens": ["smdg", "papila"],
    },
    # HYGD AQPR endpoints from Notebook 08/09/10.
    "hygd_heatmap_xai_metrics": {
        "required": True,
        "exact_names": ["notebook08_aqpr_heatmap_xai_metrics_consolidated.csv"],
        "include_tokens": ["heatmap", "xai", "metrics", "consolidated"],
        "exclude_tokens": ["smdg", "papila", "random_control_summary"],
    },
    "hygd_heatmap_patient_metrics": {
        "required": False,
        "exact_names": ["notebook08_aqpr_heatmap_xai_patient_level_metrics.csv"],
        "include_tokens": ["heatmap", "patient", "metrics"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_anatomical_occlusion_metrics": {
        "required": False,
        "exact_names": ["notebook08_aqpr_anatomical_occlusion_metrics_consolidated.csv"],
        "include_tokens": ["anatomical", "occlusion", "metrics", "consolidated"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_ablation_predictions": {
        "required": True,
        "exact_names": ["notebook09_full_aqpr_ablation_analysis_predictions_all_models.csv"],
        "include_tokens": ["ablation", "analysis", "predictions", "all", "models"],
        "exclude_tokens": ["smdg", "papila", "row_predictions_", "patient_predictions"],
    },
    "hygd_ablation_image_deltas": {
        "required": True,
        "exact_names": ["notebook09_full_aqpr_ablation_image_level_deltas_vs_original.csv"],
        "include_tokens": ["ablation", "image", "level", "deltas", "original"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_ablation_patient_deltas": {
        "required": False,
        "exact_names": ["notebook09_full_aqpr_ablation_patient_level_deltas_vs_original.csv"],
        "include_tokens": ["ablation", "patient", "level", "deltas", "original"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_ablation_primary_endpoint": {
        "required": False,
        "exact_names": ["notebook09_aqpr_ablation_primary_endpoint_table.csv"],
        "include_tokens": ["ablation", "primary", "endpoint"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_anatomy_constrained_model_roles": {
        "required": False,
        "exact_names": ["notebook10_final_model_roles.csv"],
        "include_tokens": ["notebook10", "final", "model", "roles"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_anatomy_constrained_primary_deltas": {
        "required": False,
        "exact_names": ["notebook10_final_primary_deltas_vs_full_image_efficientnetb0.csv"],
        "include_tokens": ["notebook10", "primary", "deltas", "full", "image"],
        "exclude_tokens": ["smdg", "papila"],
    },
    "hygd_outside_od_manifest": {
        "required": False,
        "exact_names": ["notebook10_outside_od_only_trained_control_input_manifest.csv"],
        "include_tokens": ["outside", "od", "input", "manifest"],
        "exclude_tokens": ["smdg", "papila", "partial"],
    },
    "hygd_outside_od_summary": {
        "required": False,
        "exact_names": ["notebook10_outside_od_only_trained_control_input_summary.csv"],
        "include_tokens": ["outside", "od", "summary"],
        "exclude_tokens": ["smdg", "papila", "partial"],
    },
    # SMDG source and label policy documentation.
    "smdg_source_label_distribution": {
        "required": True,
        "exact_names": ["notebook13_smdg_source_label_distribution.csv"],
        "include_tokens": ["smdg", "source", "label", "distribution"],
        "exclude_tokens": ["papila"],
    },
    "smdg_label_policy": {
        "required": True,
        "exact_names": ["notebook13_smdg_locked_label_policy.csv"],
        "include_tokens": ["smdg", "locked", "label", "policy"],
        "exclude_tokens": ["by_source", "papila"],
    },
    "smdg_label_policy_by_source": {
        "required": False,
        "exact_names": ["notebook13_smdg_locked_label_policy_by_source.csv"],
        "include_tokens": ["smdg", "label", "policy", "source"],
        "exclude_tokens": ["papila"],
    },
    "smdg_source_metrics_ci": {
        "required": True,
        "exact_names": ["notebook13_smdg_bootstrap_source_metrics_ci_5000.csv"],
        "include_tokens": ["smdg", "bootstrap", "source", "metrics", "ci"],
        "exclude_tokens": ["papila"],
    },
    "smdg_source_domain_shift": {
        "required": True,
        "exact_names": ["notebook13_smdg_source_domain_shift_audit_5000.csv"],
        "include_tokens": ["smdg", "source", "domain", "shift"],
        "exclude_tokens": ["papila"],
    },
}


def norm_name(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def csv_nrows_ncols(path):
    path = Path(path)
    if not path.exists() or path.suffix.lower() != ".csv":
        return np.nan, np.nan
    try:
        preview = pd.read_csv(path, nrows=2, low_memory=False)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            nrows = max(sum(1 for _ in f) - 1, 0)
        return int(nrows), int(len(preview.columns))
    except Exception:
        return np.nan, np.nan


def candidate_files():
    seen = set()
    rows = []
    for root in SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        for path in root.rglob("*.csv"):
            path = Path(path)
            key = str(path.resolve())
            if key in seen:
                continue
            seen.add(key)
            rows.append(path)
    return rows

ALL_CSV_CANDIDATES = candidate_files()


def find_source_for_spec(spec):
    exact_names = [str(x).lower() for x in spec.get("exact_names", [])]
    include_tokens = [norm_name(x) for x in spec.get("include_tokens", [])]
    exclude_tokens = [norm_name(x) for x in spec.get("exclude_tokens", [])]

    scored = []
    for path in ALL_CSV_CANDIDATES:
        path_norm = norm_name(path.name)
        full_norm = norm_name(str(path))
        if any(tok in full_norm for tok in exclude_tokens):
            continue
        score = 0
        if path.name.lower() in exact_names:
            score += 1000
        score += 50 * sum(tok in path_norm for tok in include_tokens)
        score += 10 * sum(tok in full_norm for tok in include_tokens)
        if score <= 0:
            continue
        nrows, ncols = csv_nrows_ncols(path)
        if pd.notna(nrows):
            score += min(int(nrows), 10000) / 10000.0
        scored.append((score, path, nrows, ncols))

    if not scored:
        return None

    scored = sorted(scored, key=lambda x: (x[0], str(x[1])), reverse=True)
    return scored[0]

rows = []
for role, spec in REVIEWER_SOURCE_SPECS.items():
    result = find_source_for_spec(spec)
    if result is None:
        rows.append({
            "role": role,
            "required": bool(spec.get("required", False)),
            "selected": False,
            "score": np.nan,
            "path": "",
            "file_name": "",
            "n_rows": np.nan,
            "n_columns": np.nan,
            "status": "MISSING_UPSTREAM_ARTIFACT" if spec.get("required", False) else "OPTIONAL_NOT_FOUND",
        })
    else:
        score, path, nrows, ncols = result
        rows.append({
            "role": role,
            "required": bool(spec.get("required", False)),
            "selected": True,
            "score": float(score),
            "path": str(path),
            "file_name": path.name,
            "n_rows": nrows,
            "n_columns": ncols,
            "status": "PASS",
        })

reviewer_source_map = pd.DataFrame(rows).sort_values(["required", "selected", "role"], ascending=[False, False, True])
reviewer_source_map.to_csv(REVIEWER_SOURCE_MAP_CSV, index=False)

missing_required = reviewer_source_map[
    reviewer_source_map["required"].astype(bool) & ~reviewer_source_map["selected"].astype(bool)
].copy()

cell3_status = "PASS" if missing_required.empty else "PASS_WITH_MISSING_REQUIRED_ARTIFACTS"
summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 3",
    "status": cell3_status,
    "n_csv_candidates_scanned": int(len(ALL_CSV_CANDIDATES)),
    "n_roles": int(len(reviewer_source_map)),
    "n_selected": int(reviewer_source_map["selected"].sum()),
    "n_required_missing": int(len(missing_required)),
    "missing_required_roles": missing_required["role"].astype(str).tolist(),
    "reviewer_source_map_csv": str(REVIEWER_SOURCE_MAP_CSV),
}
with open(REVIEWER_SOURCE_AUDIT_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Reviewer-required source map:")
print(reviewer_source_map[["role", "required", "selected", "n_rows", "n_columns", "file_name", "status", "path"]].to_string(index=False))
print("\nCell 3 summary:")
print(json.dumps(summary, indent=2))
print(f"Status: {cell3_status}")


In [ ]:
# Cell 4: Build unified HYGD AQPR base manifest for forensic and quality-stratified analyses

import numpy as np
import pandas as pd
from pathlib import Path
import json
import re

required_vars = ["REVIEWER_SOURCE_MAP_CSV", "TABLE_DIR", "RESULTS_14_DIR", "PRIMARY_MODEL_NAME"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run Cell 3 first. Missing variables: {missing_vars}")

HYGD_UNIFIED_BASE_CSV = TABLE_DIR / "notebook14_hygd_unified_aqpr_base_manifest.csv"
HYGD_UNIFIED_BASE_AUDIT_CSV = TABLE_DIR / "notebook14_hygd_unified_aqpr_base_manifest_column_audit.csv"
CELL4_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell04_hygd_unified_base_summary.json"

source_map = pd.read_csv(REVIEWER_SOURCE_MAP_CSV)


def source_path(role, required=False):
    block = source_map[source_map["role"].astype(str).eq(role)].copy()
    if block.empty or not bool(block.iloc[0].get("selected", False)):
        if required:
            raise FileNotFoundError(f"Required source role not selected: {role}")
        return None
    path = Path(str(block.iloc[0]["path"]))
    if not path.exists():
        if required:
            raise FileNotFoundError(f"Selected source path does not exist for {role}: {path}")
        return None
    return path


def read_source(role, required=False):
    path = source_path(role, required=required)
    if path is None:
        return pd.DataFrame(), None
    return pd.read_csv(path, low_memory=False), path


def norm_col(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def coalesce_duplicate_columns(df):
    """Collapse duplicated column names left-to-right.

    This prevents pandas from returning a DataFrame when using df["column"].
    It keeps the first non-null value across duplicated columns.
    """
    if df is None or df.empty or pd.Index(df.columns).is_unique:
        return df

    blocks = []
    for col in pd.Index(df.columns).unique():
        same = df.loc[:, df.columns == col]
        if same.shape[1] == 1:
            blocks.append(same.iloc[:, 0].rename(col))
        else:
            tmp = same.copy()
            tmp = tmp.replace({
                "": np.nan,
                "nan": np.nan,
                "None": np.nan,
                "NONE": np.nan,
                "null": np.nan,
                "NULL": np.nan,
            })
            blocks.append(tmp.bfill(axis=1).iloc[:, 0].rename(col))

    return pd.concat(blocks, axis=1)


def bool_series_from_column(df, col, default=False):
    """Safely parse a possibly duplicated boolean-like column as a Series."""
    if col not in df.columns:
        return pd.Series(default, index=df.index)

    values = df.loc[:, df.columns == col]
    if isinstance(values, pd.DataFrame):
        if values.shape[1] == 1:
            values = values.iloc[:, 0]
        else:
            values = values.replace({
                "": np.nan,
                "nan": np.nan,
                "None": np.nan,
                "NONE": np.nan,
                "null": np.nan,
                "NULL": np.nan,
            })
            values = values.bfill(axis=1).iloc[:, 0]

    return values.astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])


def find_column(df, candidates=None, contains_all=None, contains_any=None, required=False, role=""):
    if df is None or df.empty:
        if required:
            raise ValueError(f"Cannot find column in empty dataframe: {role}")
        return None

    norm_to_col = {norm_col(c): c for c in df.columns}
    candidates = candidates or []

    for cand in candidates:
        key = norm_col(cand)
        if key in norm_to_col:
            return norm_to_col[key]

    contains_all = [norm_col(x) for x in (contains_all or [])]
    contains_any = [norm_col(x) for x in (contains_any or [])]

    for c in df.columns:
        nc = norm_col(c)
        ok_all = all(tok in nc for tok in contains_all) if contains_all else True
        ok_any = any(tok in nc for tok in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    if required:
        raise ValueError(f"Column not found for {role}; columns={list(df.columns)[:50]}")
    return None


def standardize_id_columns(df, source_name):
    out = df.copy()

    image_col = find_column(
        out,
        ["image_id", "sample_id", "fundus_id", "id"],
        contains_any=["image", "sample"],
    )
    patient_col = find_column(
        out,
        ["patient_id", "subject_id", "patient"],
        contains_any=["patient", "subject"],
    )
    label_col = find_column(
        out,
        ["gon_binary", "y_true", "label", "target", "glaucoma_label", "glaucoma"],
        contains_any=["gon", "true", "label", "target", "glaucoma"],
    )

    if image_col and image_col != "image_id":
        out = out.rename(columns={image_col: "image_id"})
    if patient_col and patient_col != "patient_id":
        out = out.rename(columns={patient_col: "patient_id"})
    if label_col and label_col != "gon_binary":
        out = out.rename(columns={label_col: "gon_binary"})

    out = coalesce_duplicate_columns(out)

    if "image_id" in out.columns:
        out["image_id"] = out["image_id"].astype(str)
    if "patient_id" in out.columns:
        out["patient_id"] = out["patient_id"].astype(str)
    if "gon_binary" in out.columns:
        out["gon_binary"] = pd.to_numeric(out["gon_binary"], errors="coerce")

    return out


def slim_unique(df, keep_cols, key="image_id"):
    if df.empty or key not in df.columns:
        return pd.DataFrame(columns=[key])

    keep = [c for c in keep_cols if c in df.columns]
    keep = list(dict.fromkeys([key] + keep))
    out = df[keep].copy()
    out = coalesce_duplicate_columns(out)
    return out.drop_duplicates(subset=[key]).copy()


# Load upstream tables.
metadata, metadata_path = read_source("hygd_metadata_clean", required=True)
split_df, split_path = read_source("hygd_image_split", required=True)
quality_df, quality_path = read_source("hygd_quality_manifest", required=True)
proxy_df, proxy_path = read_source("hygd_proxy_analysis_manifest", required=True)
proxy_qc_df, proxy_qc_path = read_source("hygd_proxy_qc_manifest", required=False)

metadata = standardize_id_columns(metadata, "metadata")
split_df = standardize_id_columns(split_df, "split")
quality_df = standardize_id_columns(quality_df, "quality")
proxy_df = standardize_id_columns(proxy_df, "proxy")
proxy_qc_df = standardize_id_columns(proxy_qc_df, "proxy_qc") if not proxy_qc_df.empty else proxy_qc_df


# Normalize split column.
split_col = find_column(
    split_df,
    ["split", "glaucoma_split", "train_val_test_split", "holdout_split"],
    contains_any=["split"],
)
if split_col and split_col != "split":
    split_df = split_df.rename(columns={split_col: "split"})
split_df = coalesce_duplicate_columns(split_df)


# Normalize quality columns.
quality_score_col = find_column(
    quality_df,
    ["quality_score", "image_quality_score"],
    contains_all=["quality", "score"],
)
quality_tertile_col = find_column(
    quality_df,
    ["quality_tertile", "quality_bin", "quality_stratum"],
    contains_all=["quality"],
    contains_any=["tertile", "bin", "stratum"],
)
image_path_col = find_column(
    quality_df,
    ["image_path", "resolved_image_path", "path"],
    contains_any=["image_path", "path"],
)
fov_path_col = find_column(
    quality_df,
    ["fov_mask_path"],
    contains_all=["fov", "mask", "path"],
)

rename_map = {}
if quality_score_col and quality_score_col != "quality_score":
    rename_map[quality_score_col] = "quality_score"
if quality_tertile_col and quality_tertile_col != "quality_tertile":
    rename_map[quality_tertile_col] = "quality_tertile"
if image_path_col and image_path_col != "image_path":
    rename_map[image_path_col] = "image_path"
if fov_path_col and fov_path_col != "fov_mask_path":
    rename_map[fov_path_col] = "fov_mask_path"

quality_df = quality_df.rename(columns=rename_map)
quality_df = coalesce_duplicate_columns(quality_df)


# Normalize proxy columns.
proxy_rename = {}

proxy_strict_col = find_column(
    proxy_df,
    ["include_strict_anatomical_analysis", "strict_anatomical_proxy_valid"],
    contains_any=["strict"],
)
proxy_mask_col = find_column(
    proxy_df,
    ["mask_512_path", "od_oc_mask_path", "mask_path"],
    contains_all=["mask", "path"],
)
proxy_image_col = find_column(
    proxy_df,
    ["image_path_for_inference", "image_path", "resolved_image_path"],
    contains_any=["image_path", "resolved_image_path"],
)

for src_col, dst_col in [
    (proxy_strict_col, "strict_anatomical_proxy_valid"),
    (proxy_mask_col, "od_oc_mask_path"),
    (proxy_image_col, "proxy_image_path"),
]:
    if src_col and src_col != dst_col:
        proxy_rename[src_col] = dst_col

proxy_df = proxy_df.rename(columns=proxy_rename)
proxy_df = coalesce_duplicate_columns(proxy_df)


# Build base table with one row per HYGD image.
base_cols = ["image_id", "patient_id", "gon_binary"]
base = slim_unique(metadata, base_cols, key="image_id")

if base.empty:
    raise RuntimeError("HYGD metadata could not be standardized to an image-level table.")

split_keep = ["split", "patient_id", "gon_binary"]
base = base.merge(
    slim_unique(split_df, split_keep, key="image_id"),
    on="image_id",
    how="left",
    suffixes=("", "_split"),
)
base = coalesce_duplicate_columns(base)

quality_keep = ["quality_score", "quality_tertile", "image_path", "fov_mask_path", "path_exists"]
base = base.merge(
    slim_unique(quality_df, quality_keep, key="image_id"),
    on="image_id",
    how="left",
)
base = coalesce_duplicate_columns(base)

proxy_keep_candidates = [
    "strict_anatomical_proxy_valid",
    "include_strict_anatomical_analysis",
    "area_cdr_proxy",
    "cup_to_disc_area_ratio",
    "od_area_512",
    "oc_area_512",
    "rim_area_512",
    "od_confidence_mean",
    "oc_confidence_mean",
    "od_oc_mask_path",
    "mask_512_path",
    "proxy_image_path",
    "image_path_for_inference",
]
base = base.merge(
    slim_unique(proxy_df, proxy_keep_candidates, key="image_id"),
    on="image_id",
    how="left",
)
base = coalesce_duplicate_columns(base)


# Fill image_path from proxy if quality path is absent.
for alt_col in ["proxy_image_path", "image_path_for_inference"]:
    if alt_col in base.columns:
        if "image_path" not in base.columns:
            base["image_path"] = ""
        base["image_path"] = base["image_path"].fillna("").astype(str)
        base.loc[
            base["image_path"].eq("") & base[alt_col].notna(),
            "image_path",
        ] = base.loc[
            base["image_path"].eq("") & base[alt_col].notna(),
            alt_col,
        ].astype(str)


# Canonical strict proxy flag.
base = coalesce_duplicate_columns(base)

if "strict_anatomical_proxy_valid" not in base.columns and "include_strict_anatomical_analysis" in base.columns:
    base["strict_anatomical_proxy_valid"] = bool_series_from_column(
        base,
        "include_strict_anatomical_analysis",
        default=False,
    )
elif "strict_anatomical_proxy_valid" in base.columns:
    base["strict_anatomical_proxy_valid"] = bool_series_from_column(
        base,
        "strict_anatomical_proxy_valid",
        default=False,
    )
else:
    base["strict_anatomical_proxy_valid"] = False

base = coalesce_duplicate_columns(base)


# Canonical label and patient columns.
if "patient_id_split" in base.columns and "patient_id" in base.columns:
    base["patient_id"] = base["patient_id"].fillna(base["patient_id_split"])

if "gon_binary_split" in base.columns and "gon_binary" in base.columns:
    base["gon_binary"] = base["gon_binary"].fillna(base["gon_binary_split"])

base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")


# Quality tertiles, if missing.
if "quality_score" in base.columns:
    base["quality_score"] = pd.to_numeric(base["quality_score"], errors="coerce")
else:
    base["quality_score"] = np.nan

if "quality_tertile" not in base.columns or base["quality_tertile"].isna().all():
    valid_q = base["quality_score"].dropna()
    if len(valid_q) >= 3 and valid_q.nunique() >= 3:
        base["quality_tertile"] = pd.qcut(
            base["quality_score"].rank(method="first"),
            q=3,
            labels=["low", "medium", "high"],
        ).astype(str)
    else:
        base["quality_tertile"] = "unknown"
else:
    base["quality_tertile"] = (
        base["quality_tertile"]
        .astype(str)
        .str.lower()
        .replace({"nan": "unknown", "0": "low", "1": "medium", "2": "high"})
    )


# Path existence audit.
if "image_path" in base.columns:
    base["image_path"] = base["image_path"].fillna("").astype(str)
    base["image_path_exists_for_forensics"] = base["image_path"].apply(
        lambda x: Path(x).exists() if x else False
    )
else:
    base["image_path"] = ""
    base["image_path_exists_for_forensics"] = False


# Numeric proxy/QC columns.
for col in [
    "area_cdr_proxy",
    "cup_to_disc_area_ratio",
    "od_area_512",
    "oc_area_512",
    "rim_area_512",
    "od_confidence_mean",
    "oc_confidence_mean",
]:
    if col in base.columns:
        base[col] = pd.to_numeric(base[col], errors="coerce")


# Final cleanup and save.
base = base.sort_values(["patient_id", "image_id"]).reset_index(drop=True)
base = coalesce_duplicate_columns(base)

if not pd.Index(base.columns).is_unique:
    duplicated = base.columns[base.columns.duplicated()].tolist()
    raise RuntimeError(f"Duplicated columns remain after cleanup: {duplicated}")

base.to_csv(HYGD_UNIFIED_BASE_CSV, index=False)


# Column/missingness audit.
audit_rows = []
for col in base.columns:
    audit_rows.append({
        "column": col,
        "n_missing": int(base[col].isna().sum()),
        "missing_fraction": float(base[col].isna().mean()),
        "n_unique": int(base[col].nunique(dropna=True)),
        "dtype": str(base[col].dtype),
    })

column_audit = pd.DataFrame(audit_rows)
column_audit.to_csv(HYGD_UNIFIED_BASE_AUDIT_CSV, index=False)


summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 4",
    "status": "PASS",
    "n_images": int(len(base)),
    "n_patients": int(base["patient_id"].nunique()) if "patient_id" in base.columns else None,
    "n_gon_positive": int((base["gon_binary"] == 1).sum()) if "gon_binary" in base.columns else None,
    "n_gon_negative": int((base["gon_binary"] == 0).sum()) if "gon_binary" in base.columns else None,
    "n_strict_valid_proxy": int(base["strict_anatomical_proxy_valid"].sum()),
    "n_image_paths_existing": int(base["image_path_exists_for_forensics"].sum()),
    "quality_tertile_counts": base["quality_tertile"].value_counts(dropna=False).to_dict(),
    "hygd_unified_base_csv": str(HYGD_UNIFIED_BASE_CSV),
    "column_audit_csv": str(HYGD_UNIFIED_BASE_AUDIT_CSV),
    "source_paths": {
        "metadata": str(metadata_path),
        "split": str(split_path),
        "quality": str(quality_path),
        "proxy": str(proxy_path),
    },
}

with open(CELL4_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


print("HYGD unified AQPR base manifest summary:")
print(json.dumps(summary, indent=2))

print("\nQuality by label:")
print(pd.crosstab(base["quality_tertile"], base["gon_binary"], margins=True).to_string())

print("\nProxy strict-valid by quality:")
print(pd.crosstab(base["quality_tertile"], base["strict_anatomical_proxy_valid"], margins=True).to_string())

print(f"\nUnified base CSV: {HYGD_UNIFIED_BASE_CSV}")
print("Status: PASS")

In [ ]:
# Cell 5: HYGD forensic audit — optimized version
#
# Reviewer target:
# - Verify patient-safe splits and mixed-label patients.
# - Audit exact duplicates and cross-split near-duplicates.
# - Use dHash, aHash, pHash, cached thumbnails, and consensus criteria.
# - Avoid slow repeated image reads from Google Drive.
# - Preserve output names required by downstream cells.

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from PIL import Image, ImageOps, ImageDraw
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    from scipy.fftpack import dct
    SCIPY_DCT_AVAILABLE = True
except Exception:
    SCIPY_DCT_AVAILABLE = False


required_vars = ["HYGD_UNIFIED_BASE_CSV", "TABLE_DIR", "FIGURE_DATA_DIR", "RESULTS_14_DIR"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run Cell 4 first. Missing variables: {missing_vars}")


# -------------------------------------------------------------------------
# Output paths
# -------------------------------------------------------------------------

HYGD_SPLIT_PATIENT_LEAKAGE_CSV = TABLE_DIR / "notebook14_hygd_split_patient_leakage_audit.csv"
HYGD_SPLIT_LABEL_QUALITY_CSV = TABLE_DIR / "notebook14_hygd_split_label_quality_distribution.csv"
HYGD_IMAGES_PER_PATIENT_CSV = TABLE_DIR / "notebook14_hygd_images_per_patient_audit.csv"

HYGD_EXACT_HASH_CSV = TABLE_DIR / "notebook14_hygd_exact_image_hash_audit.csv"
HYGD_EXACT_DUPLICATE_GROUPS_CSV = TABLE_DIR / "notebook14_hygd_exact_duplicate_groups.csv"

HYGD_PHASH_CSV = TABLE_DIR / "notebook14_hygd_perceptual_hash_audit.csv"
HYGD_NEAR_DUPLICATE_PAIRS_CSV = TABLE_DIR / "notebook14_hygd_cross_split_near_duplicate_pairs.csv"

HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_near_duplicate_threshold_summary.csv"
HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV = TABLE_DIR / "notebook14_hygd_suspicious_cross_split_near_duplicate_pairs.csv"
HYGD_NEAR_DUPLICATE_SPLIT_MATRIX_CSV = TABLE_DIR / "notebook14_hygd_near_duplicate_split_matrix.csv"
HYGD_NEAR_DUPLICATE_LABEL_MATRIX_CSV = TABLE_DIR / "notebook14_hygd_near_duplicate_label_matrix.csv"

HYGD_PROXY_QC_BY_QUALITY_CSV = TABLE_DIR / "notebook14_hygd_proxy_qc_by_quality_and_label.csv"

HYGD_NEAR_DUPLICATE_PANEL_PNG = FIGURE_DATA_DIR / "notebook14_hygd_suspicious_near_duplicate_pairs_panel.png"
HYGD_NEAR_DUPLICATE_PANEL_PDF = FIGURE_DATA_DIR / "notebook14_hygd_suspicious_near_duplicate_pairs_panel.pdf"

CELL5_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell05_hygd_forensic_audit_summary.json"


# -------------------------------------------------------------------------
# Utility functions
# -------------------------------------------------------------------------

def safe_bool_series(s):
    return s.astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])


def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")


def normalize_quality(x):
    x = str(x).strip().lower()
    mapping = {
        "0": "low",
        "1": "medium",
        "2": "high",
        "low_quality": "low",
        "medium_quality": "medium",
        "high_quality": "high",
        "nan": "unknown",
        "none": "unknown",
        "null": "unknown",
        "": "unknown",
    }
    return mapping.get(x, x)


def file_sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def hex_to_int_or_none(x):
    try:
        if isinstance(x, str) and len(x) > 0:
            return int(x, 16)
    except Exception:
        pass
    return None


def hamming_int(a, b):
    if a is None or b is None:
        return np.nan
    return int(a ^ b).bit_count()


def dhash_64_from_gray(gray_arr):
    """gray_arr must be 8x9 or image resized later."""
    diff = gray_arr[:, 1:] > gray_arr[:, :-1]
    bits = 0
    for v in diff.flatten():
        bits = (bits << 1) | int(bool(v))
    return f"{bits:016x}"


def ahash_64_from_gray(gray_arr):
    diff = gray_arr > gray_arr.mean()
    bits = 0
    for v in diff.flatten():
        bits = (bits << 1) | int(bool(v))
    return f"{bits:016x}"


def phash_64_from_gray(gray_arr):
    if not SCIPY_DCT_AVAILABLE:
        return ""
    coeff = dct(dct(gray_arr.astype(np.float32), axis=0, norm="ortho"), axis=1, norm="ortho")
    low = coeff[:8, :8].copy()
    vals = low.flatten()
    med = np.median(vals[1:])
    bits_arr = vals > med
    bits = 0
    for v in bits_arr:
        bits = (bits << 1) | int(bool(v))
    return f"{bits:016x}"


def load_image_features(path, thumb_size=(64, 64)):
    """Load image once and compute dHash, aHash, pHash, and normalized thumbnail.

    Returns lightweight features for fast pairwise comparison.
    """
    result = {
        "dhash64": "",
        "ahash64": "",
        "phash64": "",
        "thumb_flat": None,
        "read_ok": False,
        "read_error": "",
    }

    if not PIL_AVAILABLE:
        result["read_error"] = "PIL unavailable"
        return result

    try:
        img = Image.open(path).convert("L")

        dh_img = img.resize((9, 8), Image.Resampling.LANCZOS)
        dh_arr = np.asarray(dh_img, dtype=np.float32)
        result["dhash64"] = dhash_64_from_gray(dh_arr)

        ah_img = img.resize((8, 8), Image.Resampling.LANCZOS)
        ah_arr = np.asarray(ah_img, dtype=np.float32)
        result["ahash64"] = ahash_64_from_gray(ah_arr)

        if SCIPY_DCT_AVAILABLE:
            ph_img = img.resize((32, 32), Image.Resampling.LANCZOS)
            ph_arr = np.asarray(ph_img, dtype=np.float32)
            result["phash64"] = phash_64_from_gray(ph_arr)

        th_img = img.resize(thumb_size, Image.Resampling.LANCZOS)
        th = np.asarray(th_img, dtype=np.float32)
        th = (th - th.mean()) / (th.std() + 1e-8)
        result["thumb_flat"] = th.reshape(-1).astype(np.float32)

        result["read_ok"] = True
        return result

    except Exception as exc:
        result["read_error"] = f"{type(exc).__name__}: {exc}"
        return result


def thumb_similarity_from_cache(idx_a, idx_b, thumbs):
    a = thumbs.get(idx_a)
    b = thumbs.get(idx_b)
    if a is None or b is None:
        return np.nan, np.nan
    diff = a - b
    rmse = float(np.sqrt(np.mean(diff * diff)))
    corr = float(np.mean(a * b))
    return rmse, corr


def image_open_rgb(path):
    return Image.open(path).convert("RGB")


def make_pair_panel(pairs_df, output_png, output_pdf, max_pairs=12, thumb_size=(180, 180)):
    if not PIL_AVAILABLE or pairs_df.empty:
        return False

    rows = []

    for _, r in pairs_df.head(max_pairs).iterrows():
        pa = Path(str(r.get("path_a", "")))
        pb = Path(str(r.get("path_b", "")))

        if not pa.exists() or not pb.exists():
            continue

        try:
            ia = ImageOps.contain(image_open_rgb(pa), thumb_size)
            ib = ImageOps.contain(image_open_rgb(pb), thumb_size)

            canvas_w = thumb_size[0] * 2 + 40
            canvas_h = thumb_size[1] + 62
            canvas = Image.new("RGB", (canvas_w, canvas_h), "white")
            canvas.paste(ia, (10, 10))
            canvas.paste(ib, (thumb_size[0] + 30, 10))

            draw = ImageDraw.Draw(canvas)

            text1 = (
                f"d={r.get('dhash_distance', np.nan)} "
                f"a={r.get('ahash_distance', np.nan)} "
                f"p={r.get('phash_distance', np.nan)} "
                f"corr={r.get('thumb_corr64', np.nan):.2f}"
            )
            text2 = (
                f"{r.get('split_a','?')}:{r.get('label_a','?')} "
                f"vs {r.get('split_b','?')}:{r.get('label_b','?')}"
            )
            text3 = (
                f"pt {r.get('patient_id_a','?')} vs {r.get('patient_id_b','?')}"
            )

            draw.text((10, thumb_size[1] + 12), text1, fill="black")
            draw.text((10, thumb_size[1] + 30), text2, fill="black")
            draw.text((10, thumb_size[1] + 48), text3, fill="black")

            rows.append(canvas)

        except Exception:
            continue

    if not rows:
        return False

    panel_w = max(im.width for im in rows)
    panel_h = sum(im.height for im in rows)
    panel = Image.new("RGB", (panel_w, panel_h), "white")

    y = 0
    for im in rows:
        panel.paste(im, (0, y))
        y += im.height

    panel.save(output_png)
    panel.save(output_pdf)
    return True


def empty_near_pairs_df():
    return pd.DataFrame(columns=[
        "image_id_a", "patient_id_a", "split_a", "label_a", "quality_tertile_a",
        "image_id_b", "patient_id_b", "split_b", "label_b", "quality_tertile_b",
        "dhash_distance", "ahash_distance", "phash_distance",
        "min_hash_distance", "mean_hash_distance",
        "n_hashes_le_6", "n_hashes_le_10",
        "thumb_rmse64", "thumb_corr64",
        "same_patient", "same_label",
        "path_a", "path_b",
    ])


# -------------------------------------------------------------------------
# Load and normalize base manifest
# -------------------------------------------------------------------------

base = pd.read_csv(HYGD_UNIFIED_BASE_CSV, low_memory=False)
base["image_id"] = base["image_id"].astype(str)
base["patient_id"] = base["patient_id"].astype(str)

if "split" not in base.columns:
    base["split"] = "unknown"
base["split"] = base["split"].fillna("unknown").astype(str)

base["gon_binary"] = safe_numeric(base["gon_binary"])

if "quality_score" in base.columns:
    base["quality_score"] = safe_numeric(base["quality_score"])
else:
    base["quality_score"] = np.nan

if "quality_tertile" not in base.columns:
    base["quality_tertile"] = "unknown"
base["quality_tertile"] = base["quality_tertile"].apply(normalize_quality)

if "strict_anatomical_proxy_valid" not in base.columns:
    base["strict_anatomical_proxy_valid"] = False
base["strict_anatomical_proxy_valid"] = safe_bool_series(base["strict_anatomical_proxy_valid"])

if "image_path" not in base.columns:
    base["image_path"] = ""
base["image_path"] = base["image_path"].fillna("").astype(str)


# -------------------------------------------------------------------------
# 5.1 Patient-safe split audit
# -------------------------------------------------------------------------

patient_split = (
    base.groupby("patient_id")
    .agg(
        n_images=("image_id", "nunique"),
        n_splits=("split", "nunique"),
        splits=("split", lambda x: ";".join(sorted(set(map(str, x))))),
        n_labels=("gon_binary", "nunique"),
        labels=("gon_binary", lambda x: ";".join(sorted(set(map(lambda v: str(int(v)) if pd.notna(v) else "nan", x))))),
        quality_score_mean=("quality_score", "mean"),
        strict_proxy_fraction=("strict_anatomical_proxy_valid", "mean"),
    )
    .reset_index()
)

patient_split["patient_crosses_splits"] = patient_split["n_splits"] > 1
patient_split["patient_has_mixed_labels"] = patient_split["n_labels"] > 1
patient_split.to_csv(HYGD_SPLIT_PATIENT_LEAKAGE_CSV, index=False)


# -------------------------------------------------------------------------
# 5.2 Split / label / quality distribution
# -------------------------------------------------------------------------

split_label_quality = (
    base.groupby(["split", "quality_tertile", "gon_binary"], dropna=False)
    .agg(
        n_images=("image_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        quality_score_mean=("quality_score", "mean"),
        quality_score_std=("quality_score", "std"),
        strict_proxy_fraction=("strict_anatomical_proxy_valid", "mean"),
    )
    .reset_index()
)
split_label_quality.to_csv(HYGD_SPLIT_LABEL_QUALITY_CSV, index=False)


# -------------------------------------------------------------------------
# 5.3 Images per patient audit
# -------------------------------------------------------------------------

images_per_patient = (
    base.groupby("patient_id")
    .agg(
        n_images=("image_id", "nunique"),
        gon_binary=("gon_binary", "max"),
        split=("split", lambda x: ";".join(sorted(set(map(str, x))))),
        quality_score_mean=("quality_score", "mean"),
        quality_score_min=("quality_score", "min"),
        quality_score_max=("quality_score", "max"),
        strict_proxy_fraction=("strict_anatomical_proxy_valid", "mean"),
    )
    .reset_index()
)
images_per_patient.to_csv(HYGD_IMAGES_PER_PATIENT_CSV, index=False)


# -------------------------------------------------------------------------
# 5.4 Exact hash audit
# Optimized:
# - File size is computed for all images.
# - SHA256 is computed only for file-size groups with at least two images.
#   This is sufficient for exact duplicate detection and avoids unnecessary I/O.
# -------------------------------------------------------------------------

exact_rows = []

for _, row in base.iterrows():
    path = Path(str(row.get("image_path", "")))
    exists = path.exists()
    fsize = int(path.stat().st_size) if exists else np.nan

    exact_rows.append({
        "image_id": row["image_id"],
        "patient_id": row["patient_id"],
        "split": row["split"],
        "gon_binary": row["gon_binary"],
        "quality_tertile": row["quality_tertile"],
        "image_path": str(path),
        "image_path_exists": bool(exists),
        "file_size_bytes": fsize,
        "sha256": "",
        "sha256_status": "pending_if_duplicate_size" if exists else "missing_file",
    })

exact_hash = pd.DataFrame(exact_rows)

duplicate_size_values = (
    exact_hash[exact_hash["image_path_exists"]]
    .groupby("file_size_bytes")
    .filter(lambda g: len(g) > 1)["file_size_bytes"]
    .unique()
)

needs_sha = exact_hash["file_size_bytes"].isin(duplicate_size_values) & exact_hash["image_path_exists"]

for idx in exact_hash.index[needs_sha]:
    try:
        exact_hash.loc[idx, "sha256"] = file_sha256(exact_hash.loc[idx, "image_path"])
        exact_hash.loc[idx, "sha256_status"] = "computed_duplicate_size_group"
    except Exception as exc:
        exact_hash.loc[idx, "sha256"] = ""
        exact_hash.loc[idx, "sha256_status"] = f"sha256_error:{type(exc).__name__}"

exact_hash.loc[exact_hash["image_path_exists"] & ~needs_sha, "sha256_status"] = "skipped_unique_file_size"

exact_hash.to_csv(HYGD_EXACT_HASH_CSV, index=False)

exact_dups = []
valid_hashes = exact_hash[exact_hash["sha256"].astype(str).str.len() > 0].copy()

for sha, group in valid_hashes.groupby("sha256"):
    if len(group) <= 1:
        continue

    exact_dups.append({
        "sha256": sha,
        "n_images": int(len(group)),
        "n_patients": int(group["patient_id"].nunique()),
        "n_splits": int(group["split"].nunique()),
        "n_labels": int(group["gon_binary"].nunique(dropna=True)),
        "cross_split": bool(group["split"].nunique() > 1),
        "cross_patient": bool(group["patient_id"].nunique() > 1),
        "mixed_labels": bool(group["gon_binary"].nunique(dropna=True) > 1),
        "splits": ";".join(sorted(group["split"].astype(str).unique())),
        "labels": ";".join(sorted(group["gon_binary"].dropna().astype(int).astype(str).unique())),
        "image_ids": ";".join(group["image_id"].astype(str).tolist()),
        "patient_ids": ";".join(sorted(group["patient_id"].astype(str).unique())),
        "image_paths": ";".join(group["image_path"].astype(str).tolist()),
    })

exact_duplicate_groups = pd.DataFrame(exact_dups)
exact_duplicate_groups.to_csv(HYGD_EXACT_DUPLICATE_GROUPS_CSV, index=False)


# -------------------------------------------------------------------------
# 5.5 Perceptual hash audit
# Optimized:
# - Image is opened once.
# - Hashes and normalized thumbnails are cached.
# - Pairwise loop uses integer hashes and cached thumbnails only.
# -------------------------------------------------------------------------

feature_rows = []
thumb_cache = {}

if PIL_AVAILABLE:
    for idx, row in exact_hash.iterrows():
        path = Path(str(row["image_path"]))
        feats = {
            "dhash64": "",
            "ahash64": "",
            "phash64": "",
            "read_ok": False,
            "read_error": "",
        }

        if bool(row["image_path_exists"]):
            loaded = load_image_features(path)
            feats.update({
                "dhash64": loaded["dhash64"],
                "ahash64": loaded["ahash64"],
                "phash64": loaded["phash64"],
                "read_ok": loaded["read_ok"],
                "read_error": loaded["read_error"],
            })
            if loaded["thumb_flat"] is not None:
                thumb_cache[idx] = loaded["thumb_flat"]

        feature_rows.append({
            **row.to_dict(),
            **feats,
        })
else:
    print("PIL is not available; perceptual hash audit will be skipped.")

phash_df = pd.DataFrame(feature_rows)

for col in ["dhash64", "ahash64", "phash64"]:
    if col not in phash_df.columns:
        phash_df[col] = ""

phash_df["dhash_int"] = phash_df["dhash64"].apply(hex_to_int_or_none)
phash_df["ahash_int"] = phash_df["ahash64"].apply(hex_to_int_or_none)
phash_df["phash_int"] = phash_df["phash64"].apply(hex_to_int_or_none)

phash_df.to_csv(HYGD_PHASH_CSV, index=False)


# Pairwise cross-split screening.
valid = phash_df[
    phash_df["dhash_int"].notna()
    &
    phash_df["image_path_exists"].astype(bool)
].copy()

valid = valid.reset_index().rename(columns={"index": "original_index"})

near_pairs = []

if not valid.empty:
    records = valid.to_dict("records")

    # Precompute split groups to avoid same-split comparisons.
    split_values = sorted(valid["split"].astype(str).unique())

    for split_a_i in range(len(split_values)):
        for split_b_i in range(split_a_i + 1, len(split_values)):
            split_a = split_values[split_a_i]
            split_b = split_values[split_b_i]

            group_a = [r for r in records if str(r["split"]) == split_a]
            group_b = [r for r in records if str(r["split"]) == split_b]

            for r1 in group_a:
                for r2 in group_b:
                    dh_dist = hamming_int(r1.get("dhash_int"), r2.get("dhash_int"))
                    ah_dist = hamming_int(r1.get("ahash_int"), r2.get("ahash_int"))
                    ph_dist = hamming_int(r1.get("phash_int"), r2.get("phash_int"))

                    distances = [x for x in [dh_dist, ah_dist, ph_dist] if pd.notna(x)]
                    if not distances:
                        continue

                    min_dist = int(np.min(distances))
                    mean_dist = float(np.mean(distances))
                    n_hashes_le_6 = int(np.sum([x <= 6 for x in distances]))
                    n_hashes_le_10 = int(np.sum([x <= 10 for x in distances]))

                    # Save only plausible candidate pairs.
                    # This keeps the CSV compact and avoids heavy downstream processing.
                    keep_pair = (
                        (pd.notna(dh_dist) and dh_dist <= 8)
                        or min_dist <= 12
                        or n_hashes_le_10 >= 2
                    )

                    if not keep_pair:
                        continue

                    rmse64, corr64 = thumb_similarity_from_cache(
                        r1["original_index"],
                        r2["original_index"],
                        thumb_cache,
                    )

                    near_pairs.append({
                        "image_id_a": r1["image_id"],
                        "patient_id_a": r1["patient_id"],
                        "split_a": r1["split"],
                        "label_a": r1["gon_binary"],
                        "quality_tertile_a": r1["quality_tertile"],
                        "image_id_b": r2["image_id"],
                        "patient_id_b": r2["patient_id"],
                        "split_b": r2["split"],
                        "label_b": r2["gon_binary"],
                        "quality_tertile_b": r2["quality_tertile"],
                        "dhash_distance": int(dh_dist) if pd.notna(dh_dist) else np.nan,
                        "ahash_distance": int(ah_dist) if pd.notna(ah_dist) else np.nan,
                        "phash_distance": int(ph_dist) if pd.notna(ph_dist) else np.nan,
                        "min_hash_distance": min_dist,
                        "mean_hash_distance": mean_dist,
                        "n_hashes_le_6": n_hashes_le_6,
                        "n_hashes_le_10": n_hashes_le_10,
                        "thumb_rmse64": rmse64,
                        "thumb_corr64": corr64,
                        "same_patient": str(r1["patient_id"]) == str(r2["patient_id"]),
                        "same_label": str(r1["gon_binary"]) == str(r2["gon_binary"]),
                        "path_a": r1["image_path"],
                        "path_b": r2["image_path"],
                    })

near_duplicate_pairs = pd.DataFrame(near_pairs)

if near_duplicate_pairs.empty:
    near_duplicate_pairs = empty_near_pairs_df()
else:
    sort_cols = [c for c in ["min_hash_distance", "dhash_distance", "thumb_rmse64"] if c in near_duplicate_pairs.columns]
    near_duplicate_pairs = near_duplicate_pairs.sort_values(sort_cols).reset_index(drop=True)

near_duplicate_pairs.to_csv(HYGD_NEAR_DUPLICATE_PAIRS_CSV, index=False)


# Conservative suspicious subset.
if not near_duplicate_pairs.empty:
    suspicious = near_duplicate_pairs[
        (
            (pd.to_numeric(near_duplicate_pairs["dhash_distance"], errors="coerce") <= 2)
            |
            (pd.to_numeric(near_duplicate_pairs["min_hash_distance"], errors="coerce") <= 2)
            |
            (
                (pd.to_numeric(near_duplicate_pairs["n_hashes_le_6"], errors="coerce") >= 2)
                &
                (pd.to_numeric(near_duplicate_pairs["thumb_corr64"], errors="coerce") >= 0.90)
            )
        )
    ].copy()

    suspicious = suspicious.sort_values(["min_hash_distance", "dhash_distance", "thumb_rmse64"]).reset_index(drop=True)
else:
    suspicious = empty_near_pairs_df()

suspicious.to_csv(HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV, index=False)


# Threshold summary.
threshold_rows = []
thresholds = [0, 1, 2, 4, 6, 8, 10, 12]

for t in thresholds:
    if near_duplicate_pairs.empty:
        block = pd.DataFrame(columns=near_duplicate_pairs.columns)
        consensus_block = pd.DataFrame(columns=near_duplicate_pairs.columns)
    else:
        block = near_duplicate_pairs[
            pd.to_numeric(near_duplicate_pairs["dhash_distance"], errors="coerce") <= t
        ].copy()

        consensus_block = near_duplicate_pairs[
            (pd.to_numeric(near_duplicate_pairs["min_hash_distance"], errors="coerce") <= t)
            &
            (pd.to_numeric(near_duplicate_pairs["n_hashes_le_10"], errors="coerce") >= 2)
        ].copy()

    for criterion, current in [
        ("dhash_only", block),
        ("consensus_min_hash_and_at_least_two_hashes_le_10", consensus_block),
    ]:
        threshold_rows.append({
            "threshold": int(t),
            "criterion": criterion,
            "n_cross_split_pairs": int(len(current)),
            "n_same_patient_pairs": int(current["same_patient"].sum()) if not current.empty else 0,
            "n_same_label_pairs": int(current["same_label"].sum()) if not current.empty else 0,
            "n_distinct_patient_pairs": int((~current["same_patient"]).sum()) if not current.empty else 0,
            "n_mixed_label_pairs": int((~current["same_label"]).sum()) if not current.empty else 0,
            "n_unique_images_in_pairs": int(pd.unique(pd.concat([current["image_id_a"], current["image_id_b"]])).size) if not current.empty else 0,
            "median_thumb_corr64": float(current["thumb_corr64"].median()) if not current.empty else np.nan,
            "median_thumb_rmse64": float(current["thumb_rmse64"].median()) if not current.empty else np.nan,
        })

threshold_summary = pd.DataFrame(threshold_rows)
threshold_summary.to_csv(HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV, index=False)


# Split and label matrices for suspicious pairs.
if not suspicious.empty:
    split_matrix = (
        suspicious.assign(
            split_pair=suspicious.apply(
                lambda r: " vs ".join(sorted([str(r["split_a"]), str(r["split_b"])])),
                axis=1,
            )
        )
        .groupby("split_pair")
        .agg(
            n_pairs=("image_id_a", "size"),
            n_same_label=("same_label", "sum"),
            n_same_patient=("same_patient", "sum"),
            median_min_hash_distance=("min_hash_distance", "median"),
            median_thumb_corr64=("thumb_corr64", "median"),
        )
        .reset_index()
    )

    label_matrix = (
        suspicious.assign(
            label_pair=suspicious.apply(
                lambda r: (
                    f"{int(r['label_a']) if pd.notna(r['label_a']) else 'nan'} "
                    f"vs {int(r['label_b']) if pd.notna(r['label_b']) else 'nan'}"
                ),
                axis=1,
            )
        )
        .groupby("label_pair")
        .agg(
            n_pairs=("image_id_a", "size"),
            n_same_patient=("same_patient", "sum"),
            median_min_hash_distance=("min_hash_distance", "median"),
            median_thumb_corr64=("thumb_corr64", "median"),
        )
        .reset_index()
    )
else:
    split_matrix = pd.DataFrame(columns=[
        "split_pair", "n_pairs", "n_same_label", "n_same_patient",
        "median_min_hash_distance", "median_thumb_corr64",
    ])
    label_matrix = pd.DataFrame(columns=[
        "label_pair", "n_pairs", "n_same_patient",
        "median_min_hash_distance", "median_thumb_corr64",
    ])

split_matrix.to_csv(HYGD_NEAR_DUPLICATE_SPLIT_MATRIX_CSV, index=False)
label_matrix.to_csv(HYGD_NEAR_DUPLICATE_LABEL_MATRIX_CSV, index=False)


# Visual panel.
panel_generated = make_pair_panel(
    suspicious,
    HYGD_NEAR_DUPLICATE_PANEL_PNG,
    HYGD_NEAR_DUPLICATE_PANEL_PDF,
    max_pairs=12,
)


# -------------------------------------------------------------------------
# 5.6 OD/OC proxy QC by quality and label
# -------------------------------------------------------------------------

proxy_cols = [
    "strict_anatomical_proxy_valid",
    "area_cdr_proxy",
    "cup_to_disc_area_ratio",
    "od_area_512",
    "oc_area_512",
    "od_confidence_mean",
    "oc_confidence_mean",
]

for col in proxy_cols:
    if col not in base.columns:
        base[col] = np.nan
    if col != "strict_anatomical_proxy_valid":
        base[col] = pd.to_numeric(base[col], errors="coerce")

proxy_qc_quality = (
    base.groupby(["quality_tertile", "gon_binary"], dropna=False)
    .agg(
        n_images=("image_id", "nunique"),
        n_patients=("patient_id", "nunique"),
        strict_valid_fraction=("strict_anatomical_proxy_valid", "mean"),
        cdr_proxy_mean=("area_cdr_proxy", "mean"),
        cdr_proxy_std=("area_cdr_proxy", "std"),
        od_area_mean=("od_area_512", "mean"),
        oc_area_mean=("oc_area_512", "mean"),
        od_confidence_mean=("od_confidence_mean", "mean"),
        oc_confidence_mean=("oc_confidence_mean", "mean"),
    )
    .reset_index()
)
proxy_qc_quality.to_csv(HYGD_PROXY_QC_BY_QUALITY_CSV, index=False)


# -------------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------------

n_exact_cross_split_groups = 0
n_exact_cross_patient_groups = 0
n_exact_mixed_label_groups = 0

if not exact_duplicate_groups.empty:
    n_exact_cross_split_groups = int(exact_duplicate_groups["cross_split"].sum())
    n_exact_cross_patient_groups = int(exact_duplicate_groups["cross_patient"].sum())
    n_exact_mixed_label_groups = int(exact_duplicate_groups["mixed_labels"].sum())

dhash_le_6_count = 0
if not near_duplicate_pairs.empty and "dhash_distance" in near_duplicate_pairs.columns:
    dhash_le_6_count = int((pd.to_numeric(near_duplicate_pairs["dhash_distance"], errors="coerce") <= 6).sum())

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "notebook14",
    "cell": "Cell 5",
    "status": "PASS",
    "optimization": (
        "Image features were cached once per image; pairwise analysis used cached hashes and thumbnails. "
        "SHA256 was computed only for duplicate file-size groups, which is sufficient for exact duplicate detection."
    ),
    "interpretation_status": (
        "PATIENT_SAFE_BUT_VISUAL_SIMILARITY_REQUIRES_CONSERVATIVE_INTERPRETATION"
        if (len(suspicious) > 0 or n_exact_cross_split_groups > 0)
        else "NO_MAJOR_FORENSIC_WARNING_DETECTED"
    ),
    "n_images": int(len(base)),
    "n_patients": int(base["patient_id"].nunique()),
    "n_patients_crossing_splits": int(patient_split["patient_crosses_splits"].sum()),
    "n_patients_with_mixed_labels": int(patient_split["patient_has_mixed_labels"].sum()),
    "n_existing_image_paths": int(exact_hash["image_path_exists"].sum()),
    "pil_available_for_perceptual_hash": bool(PIL_AVAILABLE),
    "scipy_dct_available_for_phash": bool(SCIPY_DCT_AVAILABLE),
    "n_sha256_computed": int(exact_hash["sha256"].astype(str).str.len().gt(0).sum()),
    "n_exact_duplicate_hash_groups": int(len(exact_duplicate_groups)),
    "n_exact_cross_split_duplicate_groups": int(n_exact_cross_split_groups),
    "n_exact_cross_patient_duplicate_groups": int(n_exact_cross_patient_groups),
    "n_exact_mixed_label_duplicate_groups": int(n_exact_mixed_label_groups),
    "n_cross_split_candidate_pairs_saved": int(len(near_duplicate_pairs)),
    "n_cross_split_near_duplicate_pairs_dhash_le_6": int(dhash_le_6_count),
    "n_suspicious_cross_split_pairs": int(len(suspicious)),
    "n_suspicious_same_patient_pairs": int(suspicious["same_patient"].sum()) if not suspicious.empty else 0,
    "n_suspicious_same_label_pairs": int(suspicious["same_label"].sum()) if not suspicious.empty else 0,
    "n_suspicious_distinct_patient_pairs": int((~suspicious["same_patient"]).sum()) if not suspicious.empty else 0,
    "n_suspicious_mixed_label_pairs": int((~suspicious["same_label"]).sum()) if not suspicious.empty else 0,
    "near_duplicate_visual_panel_generated": bool(panel_generated),
    "split_patient_leakage_csv": str(HYGD_SPLIT_PATIENT_LEAKAGE_CSV),
    "exact_duplicate_groups_csv": str(HYGD_EXACT_DUPLICATE_GROUPS_CSV),
    "near_duplicate_pairs_csv": str(HYGD_NEAR_DUPLICATE_PAIRS_CSV),
    "suspicious_near_duplicate_pairs_csv": str(HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV),
    "near_duplicate_threshold_summary_csv": str(HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV),
    "near_duplicate_split_matrix_csv": str(HYGD_NEAR_DUPLICATE_SPLIT_MATRIX_CSV),
    "near_duplicate_label_matrix_csv": str(HYGD_NEAR_DUPLICATE_LABEL_MATRIX_CSV),
    "near_duplicate_panel_png": str(HYGD_NEAR_DUPLICATE_PANEL_PNG) if HYGD_NEAR_DUPLICATE_PANEL_PNG.exists() else "",
    "proxy_qc_by_quality_csv": str(HYGD_PROXY_QC_BY_QUALITY_CSV),
}

with open(CELL5_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


print("HYGD forensic audit summary:")
print(json.dumps(summary, indent=2))

print("\nPatient split leakage rows with problems:")
problem_patients = patient_split[patient_split["patient_crosses_splits"] | patient_split["patient_has_mixed_labels"]]
print(problem_patients.head(20).to_string(index=False) if len(problem_patients) else "No patient split or mixed-label violations detected.")

print("\nExact duplicate groups preview:")
print(exact_duplicate_groups.head(20).to_string(index=False) if len(exact_duplicate_groups) else "No exact duplicate groups detected.")

print("\nNear-duplicate threshold summary:")
print(threshold_summary.to_string(index=False))

print("\nSuspicious cross-split near-duplicate preview:")
print(suspicious.head(20).to_string(index=False) if len(suspicious) else "No suspicious cross-split near-duplicate pairs under conservative criteria.")

print("\nProxy QC by quality and label:")
print(proxy_qc_quality.to_string(index=False))

print("\nStatus: PASS")

In [ ]:
# Cell 6: Quality-stratified AQPR endpoints on HYGD
#
# Reviewer target:
# - Convert the "Quality" axis from bookkeeping into quantitative stratified reporting.
# - Generate a non-empty summary table by HYGD quality tertile.
# - Add available performance/calibration, heatmap enrichment, ablation response, and proxy-QC endpoints.
# - Be explicit when an endpoint is not available from upstream artifacts.

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
)

required_vars = [
    "HYGD_UNIFIED_BASE_CSV",
    "REVIEWER_SOURCE_MAP_CSV",
    "TABLE_DIR",
    "FIGURE_DATA_DIR",
    "RESULTS_14_DIR",
    "PRIMARY_MODEL_NAME",
    "AQPR_PANEL_MODEL_NAMES",
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

# -------------------------------------------------------------------------
# Output paths preserved for downstream cells
# -------------------------------------------------------------------------

HYGD_QUALITY_PERFORMANCE_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_performance.csv"
HYGD_QUALITY_XAI_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_heatmap_enrichment.csv"
HYGD_QUALITY_ABLATION_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_ablation_delta.csv"
HYGD_QUALITY_AQPR_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_aqpr_summary.csv"
HYGD_QUALITY_FIGURE_DATA_CSV = FIGURE_DATA_DIR / "figure_data_notebook14_hygd_quality_aqpr_long.csv"
HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV = TABLE_DIR / "notebook14_hygd_quality_endpoint_availability_audit.csv"
CELL6_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell06_quality_stratified_aqpr_summary.json"

QUALITY_ORDER = ["low", "medium", "high"]

# -------------------------------------------------------------------------
# General utilities
# -------------------------------------------------------------------------

def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def normalize_quality_value(x):
    x = str(x).strip().lower()
    mapping = {
        "0": "low",
        "1": "medium",
        "2": "high",
        "low_quality": "low",
        "medium_quality": "medium",
        "high_quality": "high",
        "lowq": "low",
        "mid": "medium",
        "med": "medium",
        "mediumq": "medium",
        "highq": "high",
        "nan": "unknown",
        "none": "unknown",
        "null": "unknown",
        "": "unknown",
    }
    return mapping.get(x, x)


def canonical_model_name(x):
    s = norm_text(x)
    aliases = {
        "efficientnetb0": "EfficientNetB0",
        "efficientnet_b0": "EfficientNetB0",
        "tf_efficientnet_b0": "EfficientNetB0",
        "densenet121": "DenseNet121",
        "dense_net_121": "DenseNet121",
        "resnet18": "ResNet18",
        "resnet_18": "ResNet18",
        "resnet50": "ResNet50",
        "resnet_50": "ResNet50",
        "convnexttiny": "ConvNeXtTiny",
        "convnext_tiny": "ConvNeXtTiny",
        "swin_tiny": "SwinTiny",
        "swintiny": "SwinTiny",
        "swin_tiny_patch4_window7_224": "SwinTiny",
    }
    return aliases.get(s, str(x))


def canonical_method_name(x):
    s = norm_text(x)
    aliases = {
        "integratedgradients": "IntegratedGradients",
        "integrated_gradients": "IntegratedGradients",
        "ig": "IntegratedGradients",
        "gradcam": "GradCAM",
        "grad_cam": "GradCAM",
        "gradcamplusplus": "GradCAM++",
        "grad_cam_plus_plus": "GradCAM++",
        "gradcampp": "GradCAM++",
        "anatomicalocclusion": "AnatomicalOcclusion",
        "anatomical_occlusion": "AnatomicalOcclusion",
    }
    return aliases.get(s, str(x))


def canonical_region_name(x):
    s = norm_text(x)
    if s in {"od_plus_oc", "od_oc", "disc_cup", "optic_disc_plus_cup", "od_and_oc", "odoc"}:
        return "OD_plus_OC"
    if s in {"od", "optic_disc", "disc"}:
        return "OD"
    if s in {"oc", "optic_cup", "cup"}:
        return "OC"
    if s in {"rim", "neuroretinal_rim"}:
        return "rim"
    return str(x)


def canonical_ablation_mode(x):
    s = norm_text(x)
    aliases = {
        "original_full_image": "original_full_image",
        "original": "original_full_image",
        "full_image": "original_full_image",
        "baseline": "original_full_image",
        "none": "original_full_image",
        "unperturbed": "original_full_image",
        "od_plus_oc_blurred": "od_plus_oc_blurred",
        "od_oc_blurred": "od_plus_oc_blurred",
        "od_plus_oc_blur": "od_plus_oc_blurred",
        "odoc_blurred": "od_plus_oc_blurred",
        "matched_random_od_plus_oc_blurred": "matched_random_od_plus_oc_blurred",
        "random_od_plus_oc_blurred": "matched_random_od_plus_oc_blurred",
        "matched_random_blurred": "matched_random_od_plus_oc_blurred",
        "oc_blurred": "oc_blurred",
        "optic_cup_blurred": "oc_blurred",
        "rim_blurred": "rim_blurred",
        "neuroretinal_rim_blurred": "rim_blurred",
        "od_only": "od_only",
        "outside_od_only": "outside_od_only",
    }
    return aliases.get(s, str(x))


def coalesce_duplicate_columns(df):
    if df is None or df.empty or pd.Index(df.columns).is_unique:
        return df
    blocks = []
    for col in pd.Index(df.columns).unique():
        same = df.loc[:, df.columns == col]
        if same.shape[1] == 1:
            blocks.append(same.iloc[:, 0].rename(col))
        else:
            tmp = same.replace({
                "": np.nan,
                "nan": np.nan,
                "None": np.nan,
                "NONE": np.nan,
                "null": np.nan,
                "NULL": np.nan,
            })
            blocks.append(tmp.bfill(axis=1).iloc[:, 0].rename(col))
    return pd.concat(blocks, axis=1)


def find_col(df, candidates=None, contains_all=None, contains_any=None):
    if df is None or df.empty:
        return None

    candidates = candidates or []
    norm_to_col = {norm_text(c): c for c in df.columns}

    for cand in candidates:
        key = norm_text(cand)
        if key in norm_to_col:
            return norm_to_col[key]

    contains_all = [norm_text(x) for x in (contains_all or [])]
    contains_any = [norm_text(x) for x in (contains_any or [])]

    for c in df.columns:
        nc = norm_text(c)
        ok_all = all(tok in nc for tok in contains_all) if contains_all else True
        ok_any = any(tok in nc for tok in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    return None


def safe_numeric_series(s):
    return pd.to_numeric(s, errors="coerce")


def safe_read_csv(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception:
        return pd.DataFrame()


def ece_score(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return np.nan

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi < 1.0:
            mask = (y_prob >= lo) & (y_prob < hi)
        else:
            mask = (y_prob >= lo) & (y_prob <= hi)

        if not np.any(mask):
            continue

        acc = np.mean(y_true[mask])
        conf = np.mean(y_prob[mask])
        ece += (np.sum(mask) / len(y_true)) * abs(acc - conf)

    return float(ece)


def metric_bundle(y_true, y_prob, threshold=0.5):
    y_true = safe_numeric_series(pd.Series(y_true)).values
    y_prob = safe_numeric_series(pd.Series(y_prob)).values

    valid = np.isfinite(y_true) & np.isfinite(y_prob)
    y_true = y_true[valid].astype(int)
    y_prob = y_prob[valid].astype(float)

    out = {
        "n_images": int(len(y_true)),
        "n_pos": int(np.sum(y_true == 1)) if len(y_true) else 0,
        "n_neg": int(np.sum(y_true == 0)) if len(y_true) else 0,
        "auroc": np.nan,
        "auprc": np.nan,
        "balanced_accuracy": np.nan,
        "sensitivity": np.nan,
        "specificity": np.nan,
        "brier": np.nan,
        "ece_10bin": np.nan,
    }

    if len(y_true) == 0:
        return out

    y_pred = (y_prob >= float(threshold)).astype(int)

    if len(np.unique(y_true)) == 2:
        out["auroc"] = float(roc_auc_score(y_true, y_prob))
        out["auprc"] = float(average_precision_score(y_true, y_prob))
        out["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred))

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    out["sensitivity"] = float(tp / max(tp + fn, 1))
    out["specificity"] = float(tn / max(tn + fp, 1))
    out["brier"] = float(brier_score_loss(y_true, y_prob))
    out["ece_10bin"] = ece_score(y_true, y_prob, n_bins=10)

    return out


def bootstrap_ci_mean(values, n_boot=1000, seed=411):
    values = safe_numeric_series(pd.Series(values)).dropna().values
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots.append(np.mean(sample))
    return float(np.mean(values)), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def source_path(role):
    block = source_map[source_map["role"].astype(str).eq(role)].copy()
    if block.empty:
        return None
    if "selected" in block.columns and not bool(block.iloc[0].get("selected", False)):
        return None
    path = Path(str(block.iloc[0].get("path", "")))
    return path if path.exists() else None


def read_optional(role):
    path = source_path(role)
    if path is None:
        return pd.DataFrame(), None
    return safe_read_csv(path), path


def standardize_prediction_table(df):
    if df is None or df.empty:
        return df

    out = df.copy()
    out = coalesce_duplicate_columns(out)

    rename = {}

    image_col = find_col(out, ["image_id", "sample_id", "fundus_id", "id"], contains_any=["image", "sample"])
    patient_col = find_col(out, ["patient_id", "subject_id", "patient"], contains_any=["patient", "subject"])
    label_col = find_col(out, ["gon_binary", "y_true", "label", "target", "glaucoma_label"], contains_any=["true", "label", "target", "gon", "glaucoma"])
    model_col = find_col(out, ["model_name", "architecture", "model", "classifier"], contains_any=["model", "architecture", "classifier"])
    mode_col = find_col(out, ["ablation_mode", "perturbation_mode", "condition", "input_mode", "variant"], contains_any=["mode", "condition", "variant", "ablation", "perturbation"])
    prob_col = find_col(
        out,
        [
            "temperature_scaled_probability",
            "calibrated_probability",
            "probability",
            "raw_probability",
            "y_prob",
            "prob_gon",
            "p_glaucoma",
            "positive_probability",
            "prediction_probability",
        ],
        contains_any=["prob", "p_glaucoma"],
    )
    threshold_col = find_col(
        out,
        ["validation_selected_threshold", "selected_threshold", "threshold", "operating_threshold"],
        contains_any=["threshold"],
    )

    for src, dst in [
        (image_col, "image_id"),
        (patient_col, "patient_id"),
        (label_col, "gon_binary"),
        (model_col, "model_name"),
        (mode_col, "ablation_mode"),
        (prob_col, "probability_for_metrics"),
        (threshold_col, "threshold"),
    ]:
        if src is not None and src != dst:
            rename[src] = dst

    out = out.rename(columns=rename)
    out = coalesce_duplicate_columns(out)

    if "image_id" in out.columns:
        out["image_id"] = out["image_id"].astype(str)
    if "patient_id" in out.columns:
        out["patient_id"] = out["patient_id"].astype(str)
    if "gon_binary" in out.columns:
        out["gon_binary"] = pd.to_numeric(out["gon_binary"], errors="coerce")
    if "model_name" in out.columns:
        out["model_name"] = out["model_name"].apply(canonical_model_name)
    if "ablation_mode" in out.columns:
        out["ablation_mode"] = out["ablation_mode"].apply(canonical_ablation_mode)
    if "probability_for_metrics" in out.columns:
        out["probability_for_metrics"] = pd.to_numeric(out["probability_for_metrics"], errors="coerce")
    if "threshold" in out.columns:
        out["threshold"] = pd.to_numeric(out["threshold"], errors="coerce")

    return out


# -------------------------------------------------------------------------
# Load base and source map
# -------------------------------------------------------------------------

base = pd.read_csv(HYGD_UNIFIED_BASE_CSV, low_memory=False)
base = coalesce_duplicate_columns(base)

base["image_id"] = base["image_id"].astype(str)
base["patient_id"] = base["patient_id"].astype(str)
base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")

if "quality_tertile" not in base.columns:
    raise RuntimeError("HYGD unified base does not contain quality_tertile. Re-run Cell 4.")

base["quality_tertile"] = base["quality_tertile"].apply(normalize_quality_value)

if "quality_score" not in base.columns:
    base["quality_score"] = np.nan
base["quality_score"] = pd.to_numeric(base["quality_score"], errors="coerce")

if "strict_anatomical_proxy_valid" not in base.columns:
    base["strict_anatomical_proxy_valid"] = False
base["strict_anatomical_proxy_valid"] = (
    base["strict_anatomical_proxy_valid"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes", "y", "t"])
)

for col in ["area_cdr_proxy", "od_area_512", "oc_area_512", "od_confidence_mean", "oc_confidence_mean"]:
    if col not in base.columns:
        base[col] = np.nan
    base[col] = pd.to_numeric(base[col], errors="coerce")

source_map = pd.read_csv(REVIEWER_SOURCE_MAP_CSV)

panel_models = [canonical_model_name(x) for x in AQPR_PANEL_MODEL_NAMES]
primary_model = canonical_model_name(PRIMARY_MODEL_NAME)

# -------------------------------------------------------------------------
# 6.1 Base quality and proxy QC summary. This is always generated.
# -------------------------------------------------------------------------

base_quality_rows = []
for q in QUALITY_ORDER:
    block = base[base["quality_tertile"].eq(q)].copy()

    if block.empty:
        base_quality_rows.append({
            "quality_tertile": q,
            "n_images": 0,
            "n_patients": 0,
            "n_gon_positive": 0,
            "n_gon_negative": 0,
            "quality_score_mean": np.nan,
            "quality_score_std": np.nan,
            "strict_proxy_fraction": np.nan,
            "cdr_proxy_mean": np.nan,
            "cdr_proxy_std": np.nan,
            "od_confidence_mean": np.nan,
            "oc_confidence_mean": np.nan,
        })
        continue

    base_quality_rows.append({
        "quality_tertile": q,
        "n_images": int(block["image_id"].nunique()),
        "n_patients": int(block["patient_id"].nunique()),
        "n_gon_positive": int((block["gon_binary"] == 1).sum()),
        "n_gon_negative": int((block["gon_binary"] == 0).sum()),
        "quality_score_mean": float(block["quality_score"].mean()),
        "quality_score_std": float(block["quality_score"].std(ddof=1)) if block["quality_score"].notna().sum() > 1 else np.nan,
        "strict_proxy_fraction": float(block["strict_anatomical_proxy_valid"].mean()),
        "cdr_proxy_mean": float(block["area_cdr_proxy"].mean()),
        "cdr_proxy_std": float(block["area_cdr_proxy"].std(ddof=1)) if block["area_cdr_proxy"].notna().sum() > 1 else np.nan,
        "od_confidence_mean": float(block["od_confidence_mean"].mean()),
        "oc_confidence_mean": float(block["oc_confidence_mean"].mean()),
    })

base_quality_summary = pd.DataFrame(base_quality_rows)

# -------------------------------------------------------------------------
# 6.2 Performance and calibration by quality tertile
# -------------------------------------------------------------------------

prediction_roles = [
    "hygd_ablation_predictions",
    "hygd_prediction_source",
    "hygd_predictions",
    "hygd_model_predictions",
]

pred_df = pd.DataFrame()
pred_path = None

for role in prediction_roles:
    candidate, candidate_path = read_optional(role)
    candidate = standardize_prediction_table(candidate)
    if (
        candidate is not None
        and not candidate.empty
        and "image_id" in candidate.columns
        and "probability_for_metrics" in candidate.columns
    ):
        pred_df = candidate.copy()
        pred_path = candidate_path
        break

quality_performance_rows = []

if not pred_df.empty:
    # Merge quality/label info from base.
    merge_cols = ["image_id", "patient_id", "gon_binary", "quality_tertile", "quality_score"]
    pred = pred_df.merge(
        base[merge_cols],
        on="image_id",
        how="left",
        suffixes=("", "_base"),
    )
    pred = coalesce_duplicate_columns(pred)

    if "gon_binary_base" in pred.columns:
        pred["gon_binary"] = pred["gon_binary"].fillna(pred["gon_binary_base"])
    if "patient_id_base" in pred.columns:
        pred["patient_id"] = pred["patient_id"].fillna(pred["patient_id_base"])

    if "model_name" not in pred.columns:
        pred["model_name"] = primary_model
    pred["model_name"] = pred["model_name"].apply(canonical_model_name)

    if "ablation_mode" not in pred.columns:
        pred["ablation_mode"] = "original_full_image"
    pred["ablation_mode"] = pred["ablation_mode"].apply(canonical_ablation_mode)

    # Prefer original/full-image rows. If no mode matches, use all rows and document this.
    original_mask = pred["ablation_mode"].eq("original_full_image")
    if original_mask.any():
        pred_original = pred[original_mask].copy()
        prediction_original_selection = "ablation_mode_original_full_image"
    else:
        pred_original = pred.copy()
        prediction_original_selection = "no_original_mode_found_used_all_prediction_rows"

    pred_original["quality_tertile"] = pred_original["quality_tertile"].apply(normalize_quality_value)
    pred_original["probability_for_metrics"] = pd.to_numeric(pred_original["probability_for_metrics"], errors="coerce")
    pred_original["threshold"] = pd.to_numeric(pred_original.get("threshold", 0.5), errors="coerce").fillna(0.5)

    for (model_name, q), group in pred_original.groupby(["model_name", "quality_tertile"], dropna=False):
        model_name = canonical_model_name(model_name)
        q = normalize_quality_value(q)

        if q not in QUALITY_ORDER:
            continue
        if panel_models and model_name not in panel_models:
            continue

        threshold_values = pd.to_numeric(group["threshold"], errors="coerce").dropna()
        threshold = float(threshold_values.iloc[0]) if len(threshold_values) else 0.5

        metrics = metric_bundle(group["gon_binary"], group["probability_for_metrics"], threshold=threshold)

        # Patient-level probability average within quality stratum.
        patient_block = (
            group.dropna(subset=["gon_binary", "probability_for_metrics"])
            .groupby("patient_id")
            .agg(
                gon_binary=("gon_binary", "max"),
                probability_for_metrics=("probability_for_metrics", "mean"),
            )
            .reset_index()
        )

        patient_metrics = metric_bundle(
            patient_block["gon_binary"],
            patient_block["probability_for_metrics"],
            threshold=threshold,
        )

        quality_performance_rows.append({
            "analysis_family": "performance_calibration",
            "model_name": model_name,
            "quality_tertile": q,
            "probability_source": "probability_for_metrics",
            "original_row_selection": prediction_original_selection,
            "threshold": threshold,
            **metrics,
            "patient_n": patient_metrics["n_images"],
            "patient_n_pos": patient_metrics["n_pos"],
            "patient_n_neg": patient_metrics["n_neg"],
            "patient_auroc": patient_metrics["auroc"],
            "patient_auprc": patient_metrics["auprc"],
            "patient_balanced_accuracy": patient_metrics["balanced_accuracy"],
            "patient_brier": patient_metrics["brier"],
            "patient_ece_10bin": patient_metrics["ece_10bin"],
        })

quality_performance = pd.DataFrame(quality_performance_rows)
quality_performance.to_csv(HYGD_QUALITY_PERFORMANCE_CSV, index=False)

# -------------------------------------------------------------------------
# 6.3 Heatmap enrichment by quality tertile
# -------------------------------------------------------------------------

xai_roles = [
    "hygd_heatmap_xai_metrics",
    "hygd_xai_metrics",
    "hygd_xai_source",
    "hygd_anatomical_xai_metrics",
]

xai_metrics = pd.DataFrame()
xai_path = None

for role in xai_roles:
    candidate, candidate_path = read_optional(role)
    if candidate is not None and not candidate.empty:
        xai_metrics = candidate.copy()
        xai_path = candidate_path
        break

xai_rows = []

if not xai_metrics.empty:
    xai = xai_metrics.copy()
    xai = coalesce_duplicate_columns(xai)

    rename = {}

    image_col = find_col(xai, ["image_id", "sample_id", "fundus_id"], contains_any=["image", "sample"])
    patient_col = find_col(xai, ["patient_id", "subject_id", "patient"], contains_any=["patient", "subject"])
    model_col = find_col(xai, ["model_name", "architecture", "model"], contains_any=["model", "architecture"])
    method_col = find_col(xai, ["method_name", "xai_method", "method", "saliency_method"], contains_any=["method"])
    region_col = find_col(xai, ["region_name", "anatomical_region", "roi", "mask_region"], contains_any=["region", "roi", "mask"])
    enr_col = find_col(
        xai,
        [
            "observed_minus_random_mean",
            "mean_observed_minus_random",
            "saliency_observed_minus_random",
            "enrichment_mean",
            "od_oc_observed_minus_random_mean",
        ],
        contains_any=["observed_minus_random", "enrichment"],
    )
    ratio_col = find_col(
        xai,
        ["enrichment_ratio", "observed_over_random", "saliency_ratio"],
        contains_any=["ratio"],
    )

    for src, dst in [
        (image_col, "image_id"),
        (patient_col, "patient_id"),
        (model_col, "model_name"),
        (method_col, "method_name"),
        (region_col, "region_name"),
        (enr_col, "observed_minus_random_mean"),
        (ratio_col, "enrichment_ratio"),
    ]:
        if src is not None and src != dst:
            rename[src] = dst

    xai = xai.rename(columns=rename)
    xai = coalesce_duplicate_columns(xai)

    if "image_id" in xai.columns:
        xai["image_id"] = xai["image_id"].astype(str)
    if "model_name" in xai.columns:
        xai["model_name"] = xai["model_name"].apply(canonical_model_name)
    else:
        xai["model_name"] = primary_model
    if "method_name" in xai.columns:
        xai["method_name"] = xai["method_name"].apply(canonical_method_name)
    else:
        xai["method_name"] = "unknown"
    if "region_name" in xai.columns:
        xai["region_name"] = xai["region_name"].apply(canonical_region_name)
    else:
        xai["region_name"] = "unknown"

    if "observed_minus_random_mean" not in xai.columns:
        xai["observed_minus_random_mean"] = np.nan
    if "enrichment_ratio" not in xai.columns:
        xai["enrichment_ratio"] = np.nan

    xai = xai.merge(
        base[["image_id", "patient_id", "quality_tertile", "quality_score"]],
        on="image_id",
        how="left",
        suffixes=("", "_base"),
    )
    xai = coalesce_duplicate_columns(xai)

    if "patient_id_base" in xai.columns:
        xai["patient_id"] = xai["patient_id"].fillna(xai["patient_id_base"])
    xai["quality_tertile"] = xai["quality_tertile"].apply(normalize_quality_value)

    # Primary endpoint: OD+OC if available; otherwise use whatever region exists, but document it.
    odoc_mask = xai["region_name"].eq("OD_plus_OC")
    if odoc_mask.any():
        xai_primary = xai[odoc_mask].copy()
        xai_region_selection = "OD_plus_OC"
    else:
        xai_primary = xai.copy()
        xai_region_selection = "all_regions_no_OD_plus_OC_column_detected"

    for (model_name, method_name, q), group in xai_primary.groupby(["model_name", "method_name", "quality_tertile"], dropna=False):
        model_name = canonical_model_name(model_name)
        method_name = canonical_method_name(method_name)
        q = normalize_quality_value(q)

        if q not in QUALITY_ORDER:
            continue
        if panel_models and model_name not in panel_models:
            continue

        values = pd.to_numeric(group["observed_minus_random_mean"], errors="coerce").dropna().values
        ratios = pd.to_numeric(group["enrichment_ratio"], errors="coerce").dropna().values
        mean_val, lo_val, hi_val = bootstrap_ci_mean(values, n_boot=1000, seed=411)

        xai_rows.append({
            "analysis_family": "heatmap_enrichment",
            "model_name": model_name,
            "method_name": method_name,
            "region_name": xai_region_selection,
            "quality_tertile": q,
            "n_images": int(group["image_id"].nunique()) if "image_id" in group.columns else int(len(group)),
            "n_patients": int(group["patient_id"].nunique()) if "patient_id" in group.columns else np.nan,
            "observed_minus_random_mean": mean_val,
            "observed_minus_random_ci95_low": lo_val,
            "observed_minus_random_ci95_high": hi_val,
            "observed_minus_random_median": float(np.median(values)) if len(values) else np.nan,
            "observed_minus_random_std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
            "enrichment_ratio_median": float(np.median(ratios)) if len(ratios) else np.nan,
        })

quality_xai = pd.DataFrame(xai_rows)
quality_xai.to_csv(HYGD_QUALITY_XAI_CSV, index=False)

# -------------------------------------------------------------------------
# 6.4 Anatomy-guided perturbation / ablation response by quality tertile
# -------------------------------------------------------------------------

ablation_roles = [
    "hygd_ablation_image_deltas",
    "hygd_ablation_delta_source",
    "hygd_ablation_deltas",
    "hygd_anatomical_ablation_deltas",
]

ablation_deltas = pd.DataFrame()
ablation_deltas_path = None

for role in ablation_roles:
    candidate, candidate_path = read_optional(role)
    if candidate is not None and not candidate.empty:
        ablation_deltas = candidate.copy()
        ablation_deltas_path = candidate_path
        break

ablation_rows = []

if not ablation_deltas.empty:
    ab = ablation_deltas.copy()
    ab = coalesce_duplicate_columns(ab)

    rename = {}

    image_col = find_col(ab, ["image_id", "sample_id", "fundus_id"], contains_any=["image", "sample"])
    patient_col = find_col(ab, ["patient_id", "subject_id", "patient"], contains_any=["patient", "subject"])
    model_col = find_col(ab, ["model_name", "architecture", "model"], contains_any=["model", "architecture"])
    mode_col = find_col(ab, ["ablation_mode", "perturbation_mode", "condition", "input_mode"], contains_any=["mode", "condition", "ablation", "perturbation"])

    raw_drop_col = find_col(
        ab,
        [
            "probability_drop_vs_original",
            "raw_probability_drop_vs_original",
            "drop_vs_original",
            "probability_delta",
            "delta_probability",
            "mean_probability_drop_vs_original",
        ],
        contains_any=["drop", "delta"],
    )
    temp_drop_col = find_col(
        ab,
        [
            "temperature_scaled_probability_drop_vs_original",
            "calibrated_probability_drop_vs_original",
            "temp_probability_drop_vs_original",
        ],
        contains_any=["temperature", "calibrated", "temp"],
    )

    for src, dst in [
        (image_col, "image_id"),
        (patient_col, "patient_id"),
        (model_col, "model_name"),
        (mode_col, "ablation_mode"),
        (raw_drop_col, "probability_drop_vs_original"),
        (temp_drop_col, "temperature_scaled_probability_drop_vs_original"),
    ]:
        if src is not None and src != dst:
            rename[src] = dst

    ab = ab.rename(columns=rename)
    ab = coalesce_duplicate_columns(ab)

    if "image_id" in ab.columns:
        ab["image_id"] = ab["image_id"].astype(str)
    if "model_name" in ab.columns:
        ab["model_name"] = ab["model_name"].apply(canonical_model_name)
    else:
        ab["model_name"] = primary_model
    if "ablation_mode" in ab.columns:
        ab["ablation_mode"] = ab["ablation_mode"].apply(canonical_ablation_mode)
    else:
        ab["ablation_mode"] = "unknown"

    if "probability_drop_vs_original" not in ab.columns:
        ab["probability_drop_vs_original"] = np.nan
    if "temperature_scaled_probability_drop_vs_original" not in ab.columns:
        ab["temperature_scaled_probability_drop_vs_original"] = np.nan

    ab = ab.merge(
        base[["image_id", "patient_id", "quality_tertile", "quality_score", "gon_binary"]],
        on="image_id",
        how="left",
        suffixes=("", "_base"),
    )
    ab = coalesce_duplicate_columns(ab)

    if "patient_id_base" in ab.columns:
        ab["patient_id"] = ab["patient_id"].fillna(ab["patient_id_base"])

    ab["quality_tertile"] = ab["quality_tertile"].apply(normalize_quality_value)

    target_modes = [
        "od_plus_oc_blurred",
        "matched_random_od_plus_oc_blurred",
        "oc_blurred",
        "rim_blurred",
        "outside_od_only",
    ]

    if ab["ablation_mode"].isin(target_modes).any():
        ab_primary = ab[ab["ablation_mode"].isin(target_modes)].copy()
    else:
        ab_primary = ab.copy()

    for (model_name, mode, q), group in ab_primary.groupby(["model_name", "ablation_mode", "quality_tertile"], dropna=False):
        model_name = canonical_model_name(model_name)
        mode = canonical_ablation_mode(mode)
        q = normalize_quality_value(q)

        if q not in QUALITY_ORDER:
            continue
        if panel_models and model_name not in panel_models:
            continue

        raw_values = pd.to_numeric(group["probability_drop_vs_original"], errors="coerce").dropna().values
        temp_values = pd.to_numeric(group["temperature_scaled_probability_drop_vs_original"], errors="coerce").dropna().values

        raw_mean, raw_lo, raw_hi = bootstrap_ci_mean(raw_values, n_boot=1000, seed=411)
        temp_mean, temp_lo, temp_hi = bootstrap_ci_mean(temp_values, n_boot=1000, seed=411)

        ablation_rows.append({
            "analysis_family": "anatomy_guided_ablation",
            "model_name": model_name,
            "ablation_mode": mode,
            "quality_tertile": q,
            "n_images": int(group["image_id"].nunique()) if "image_id" in group.columns else int(len(group)),
            "n_patients": int(group["patient_id"].nunique()) if "patient_id" in group.columns else np.nan,
            "mean_probability_drop_vs_original": raw_mean,
            "mean_probability_drop_ci95_low": raw_lo,
            "mean_probability_drop_ci95_high": raw_hi,
            "median_probability_drop_vs_original": float(np.median(raw_values)) if len(raw_values) else np.nan,
            "mean_temperature_scaled_probability_drop_vs_original": temp_mean,
            "mean_temperature_scaled_probability_drop_ci95_low": temp_lo,
            "mean_temperature_scaled_probability_drop_ci95_high": temp_hi,
            "median_temperature_scaled_probability_drop_vs_original": float(np.median(temp_values)) if len(temp_values) else np.nan,
        })

quality_ablation = pd.DataFrame(ablation_rows)
quality_ablation.to_csv(HYGD_QUALITY_ABLATION_CSV, index=False)

# -------------------------------------------------------------------------
# 6.5 Compact AQPR summary for manuscript/supplement
# -------------------------------------------------------------------------

quality_summary = base_quality_summary.copy()

# Add primary-model performance/calibration columns.
if not quality_performance.empty:
    perf = quality_performance[quality_performance["model_name"].astype(str).eq(primary_model)].copy()

    for metric in [
        "auroc",
        "auprc",
        "balanced_accuracy",
        "brier",
        "ece_10bin",
        "patient_auroc",
        "patient_balanced_accuracy",
        "patient_brier",
        "patient_ece_10bin",
    ]:
        if metric in perf.columns:
            metric_map = perf.set_index("quality_tertile")[metric].to_dict()
            quality_summary[f"primary_{metric}"] = quality_summary["quality_tertile"].map(metric_map)

# Add primary-model Integrated Gradients OD+OC enrichment.
if not quality_xai.empty:
    xai_block = quality_xai[
        quality_xai["model_name"].astype(str).eq(primary_model)
        &
        quality_xai["method_name"].astype(str).eq("IntegratedGradients")
    ].copy()

    if xai_block.empty:
        # Fallback: first available method for the primary model.
        xai_block = quality_xai[quality_xai["model_name"].astype(str).eq(primary_model)].copy()

    if not xai_block.empty:
        for col in [
            "observed_minus_random_mean",
            "observed_minus_random_ci95_low",
            "observed_minus_random_ci95_high",
            "observed_minus_random_median",
        ]:
            if col in xai_block.columns:
                col_map = xai_block.drop_duplicates("quality_tertile").set_index("quality_tertile")[col].to_dict()
                quality_summary[f"primary_ig_od_oc_{col}"] = quality_summary["quality_tertile"].map(col_map)

# Add primary-model OD+OC blur response.
if not quality_ablation.empty:
    ab_block = quality_ablation[
        quality_ablation["model_name"].astype(str).eq(primary_model)
        &
        quality_ablation["ablation_mode"].astype(str).eq("od_plus_oc_blurred")
    ].copy()

    if not ab_block.empty:
        for col in [
            "mean_probability_drop_vs_original",
            "mean_probability_drop_ci95_low",
            "mean_probability_drop_ci95_high",
            "median_probability_drop_vs_original",
        ]:
            if col in ab_block.columns:
                col_map = ab_block.drop_duplicates("quality_tertile").set_index("quality_tertile")[col].to_dict()
                quality_summary[f"primary_od_oc_blur_{col}"] = quality_summary["quality_tertile"].map(col_map)

# Add matched-random response if available.
if not quality_ablation.empty:
    rand_block = quality_ablation[
        quality_ablation["model_name"].astype(str).eq(primary_model)
        &
        quality_ablation["ablation_mode"].astype(str).eq("matched_random_od_plus_oc_blurred")
    ].copy()

    if not rand_block.empty and "mean_probability_drop_vs_original" in rand_block.columns:
        rand_map = rand_block.drop_duplicates("quality_tertile").set_index("quality_tertile")["mean_probability_drop_vs_original"].to_dict()
        quality_summary["primary_matched_random_blur_mean_probability_drop"] = quality_summary["quality_tertile"].map(rand_map)

if (
    "primary_od_oc_blur_mean_probability_drop_vs_original" in quality_summary.columns
    and "primary_matched_random_blur_mean_probability_drop" in quality_summary.columns
):
    quality_summary["primary_od_oc_minus_random_blur_mean_probability_drop"] = (
        quality_summary["primary_od_oc_blur_mean_probability_drop_vs_original"]
        - quality_summary["primary_matched_random_blur_mean_probability_drop"]
    )

# Ensure non-empty, valid CSV.
quality_summary = quality_summary.sort_values(
    by="quality_tertile",
    key=lambda s: s.map({q: i for i, q in enumerate(QUALITY_ORDER)}).fillna(99),
).reset_index(drop=True)

quality_summary.to_csv(HYGD_QUALITY_AQPR_SUMMARY_CSV, index=False)

# -------------------------------------------------------------------------
# 6.6 Long figure-data table for Notebook 16
# -------------------------------------------------------------------------

figure_parts = []

if not quality_performance.empty:
    for metric in ["auroc", "balanced_accuracy", "brier", "ece_10bin", "patient_auroc", "patient_ece_10bin"]:
        if metric in quality_performance.columns:
            tmp = quality_performance[["model_name", "quality_tertile", metric]].copy()
            tmp = tmp.rename(columns={metric: "value"})
            tmp["analysis_family"] = "performance_calibration"
            tmp["endpoint"] = metric
            figure_parts.append(tmp)

if not quality_xai.empty and "observed_minus_random_mean" in quality_xai.columns:
    tmp = quality_xai[["model_name", "method_name", "quality_tertile", "observed_minus_random_mean"]].copy()
    tmp = tmp.rename(columns={"observed_minus_random_mean": "value"})
    tmp["analysis_family"] = "heatmap_enrichment"
    tmp["endpoint"] = "od_oc_observed_minus_random_mean"
    figure_parts.append(tmp)

if not quality_ablation.empty and "mean_probability_drop_vs_original" in quality_ablation.columns:
    tmp = quality_ablation[["model_name", "ablation_mode", "quality_tertile", "mean_probability_drop_vs_original"]].copy()
    tmp = tmp.rename(columns={"mean_probability_drop_vs_original": "value"})
    tmp["analysis_family"] = "anatomy_guided_ablation"
    tmp["endpoint"] = "mean_probability_drop_vs_original"
    figure_parts.append(tmp)

figure_data = pd.concat(figure_parts, ignore_index=True, sort=False) if figure_parts else pd.DataFrame(
    columns=["model_name", "quality_tertile", "value", "analysis_family", "endpoint"]
)
figure_data.to_csv(HYGD_QUALITY_FIGURE_DATA_CSV, index=False)

# -------------------------------------------------------------------------
# 6.7 Endpoint availability audit
# -------------------------------------------------------------------------

endpoint_availability = pd.DataFrame([
    {
        "endpoint_family": "base_quality_proxy_qc",
        "source_path": str(HYGD_UNIFIED_BASE_CSV),
        "available": True,
        "n_rows": int(len(base_quality_summary)),
        "manuscript_interpretation": "Quality bookkeeping and OD/OC proxy QC by tertile are available.",
    },
    {
        "endpoint_family": "performance_calibration",
        "source_path": str(pred_path) if pred_path else "",
        "available": bool(not quality_performance.empty),
        "n_rows": int(len(quality_performance)),
        "manuscript_interpretation": (
            "Performance/calibration by quality can be reported."
            if not quality_performance.empty
            else "No compatible HYGD prediction probability table was detected; do not claim quality-stratified performance/calibration."
        ),
    },
    {
        "endpoint_family": "heatmap_enrichment",
        "source_path": str(xai_path) if xai_path else "",
        "available": bool(not quality_xai.empty),
        "n_rows": int(len(quality_xai)),
        "manuscript_interpretation": (
            "Quality-stratified saliency enrichment can be reported."
            if not quality_xai.empty
            else "No compatible XAI table was detected; do not claim quality-stratified saliency endpoints."
        ),
    },
    {
        "endpoint_family": "anatomy_guided_ablation",
        "source_path": str(ablation_deltas_path) if ablation_deltas_path else "",
        "available": bool(not quality_ablation.empty),
        "n_rows": int(len(quality_ablation)),
        "manuscript_interpretation": (
            "Quality-stratified perturbation response can be reported."
            if not quality_ablation.empty
            else "No compatible ablation-delta table was detected; do not claim quality-stratified perturbation endpoints."
        ),
    },
])
endpoint_availability.to_csv(HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV, index=False)

cell6_status = "PASS"
if quality_performance.empty:
    cell6_status = "PASS_WITHOUT_PERFORMANCE_CALIBRATION_ENDPOINTS"
if quality_xai.empty and quality_ablation.empty:
    cell6_status = "PASS_WITH_BASE_QUALITY_PROXY_ONLY"

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "notebook14",
    "cell": "Cell 6",
    "status": cell6_status,
    "primary_model": primary_model,
    "n_quality_summary_rows": int(len(quality_summary)),
    "n_quality_performance_rows": int(len(quality_performance)),
    "n_quality_xai_rows": int(len(quality_xai)),
    "n_quality_ablation_rows": int(len(quality_ablation)),
    "performance_source": str(pred_path) if pred_path else "",
    "xai_source": str(xai_path) if xai_path else "",
    "ablation_delta_source": str(ablation_deltas_path) if ablation_deltas_path else "",
    "quality_summary_csv": str(HYGD_QUALITY_AQPR_SUMMARY_CSV),
    "quality_performance_csv": str(HYGD_QUALITY_PERFORMANCE_CSV),
    "quality_xai_csv": str(HYGD_QUALITY_XAI_CSV),
    "quality_ablation_csv": str(HYGD_QUALITY_ABLATION_CSV),
    "endpoint_availability_csv": str(HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV),
    "figure_data_csv": str(HYGD_QUALITY_FIGURE_DATA_CSV),
    "manuscript_guardrail": (
        "Report only endpoint families marked available in the endpoint availability audit. "
        "Do not claim quality-stratified performance/calibration if the performance table is empty."
    ),
}

with open(CELL6_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Quality-stratified AQPR summary:")
print(quality_summary.to_string(index=False))

print("\nEndpoint availability audit:")
print(endpoint_availability.to_string(index=False))

print("\nCell 6 summary:")
print(json.dumps(summary, indent=2))

print(f"\nStatus: {cell6_status}")

In [ ]:
# Cell 7: Outside-OD shortcut-control forensic audit and conservative mask-removal input construction

from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

required_vars = ["HYGD_UNIFIED_BASE_CSV", "REVIEWER_SOURCE_MAP_CSV", "TABLE_DIR", "FIGURE_DATA_DIR", "RESULTS_14_DIR"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

OUTSIDE_OD_FORENSIC_AUDIT_CSV = TABLE_DIR / "notebook14_outside_od_shortcut_control_forensic_audit.csv"
OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV = TABLE_DIR / "notebook14_outside_od_conservative_mask_sensitivity_manifest.csv"
OUTSIDE_OD_CONSERVATIVE_SUMMARY_CSV = TABLE_DIR / "notebook14_outside_od_conservative_mask_sensitivity_summary.csv"
OUTSIDE_OD_VISUAL_PANEL_PNG = FIGURE_DATA_DIR / "notebook14_outside_od_conservative_mask_visual_panel.png"
OUTSIDE_OD_VISUAL_PANEL_PDF = FIGURE_DATA_DIR / "notebook14_outside_od_conservative_mask_visual_panel.pdf"
CELL7_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell07_outside_od_forensic_summary.json"

base = pd.read_csv(HYGD_UNIFIED_BASE_CSV, low_memory=False)
base["image_id"] = base["image_id"].astype(str)
base["patient_id"] = base["patient_id"].astype(str)
base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")

source_map = pd.read_csv(REVIEWER_SOURCE_MAP_CSV)

def get_path(role):
    block = source_map[source_map["role"].astype(str).eq(role)].copy()
    if block.empty or not bool(block.iloc[0].get("selected", False)):
        return None
    path = Path(str(block.iloc[0]["path"]))
    return path if path.exists() else None

outside_manifest_path = get_path("hygd_outside_od_manifest")
outside_summary_path = get_path("hygd_outside_od_summary")
model_roles_path = get_path("hygd_anatomy_constrained_model_roles")
primary_deltas_path = get_path("hygd_anatomy_constrained_primary_deltas")

outside_manifest = pd.read_csv(outside_manifest_path, low_memory=False) if outside_manifest_path else pd.DataFrame()
outside_summary = pd.read_csv(outside_summary_path, low_memory=False) if outside_summary_path else pd.DataFrame()
model_roles = pd.read_csv(model_roles_path, low_memory=False) if model_roles_path else pd.DataFrame()
primary_deltas = pd.read_csv(primary_deltas_path, low_memory=False) if primary_deltas_path else pd.DataFrame()

# 7.1 Existing outside-OD control audit from Notebook 10.
forensic_rows = []
if not outside_manifest.empty:
    om = outside_manifest.copy()
    if "sample_id" in om.columns and "image_id" not in om.columns:
        om = om.rename(columns={"sample_id": "image_id"})
    if "glaucoma_split" not in om.columns and "split" in om.columns:
        om = om.rename(columns={"split": "glaucoma_split"})
    if "gon_binary" not in om.columns:
        om = om.merge(base[["image_id", "gon_binary"]], on="image_id", how="left")
    geometry_cols = ["outside_od_dilation_pixels", "removed_od_fraction_of_fov", "removed_od_pixels", "preserved_outside_od_fov_pixels", "fov_pixels"]
    for col in geometry_cols:
        if col not in om.columns:
            om[col] = np.nan
        om[col] = pd.to_numeric(om[col], errors="coerce")
    for split_name, group in om.groupby("glaucoma_split", dropna=False):
        forensic_rows.append({
            "audit_scope": "existing_notebook10_outside_od_inputs",
            "split": split_name,
            "n_images": int(group["image_id"].nunique()),
            "n_patients": int(group["patient_id"].nunique()) if "patient_id" in group.columns else np.nan,
            "n_positive": int((pd.to_numeric(group["gon_binary"], errors="coerce") == 1).sum()),
            "n_negative": int((pd.to_numeric(group["gon_binary"], errors="coerce") == 0).sum()),
            "dilation_pixels": float(group["outside_od_dilation_pixels"].dropna().median()) if group["outside_od_dilation_pixels"].notna().any() else np.nan,
            "removed_od_fraction_of_fov_mean": float(group["removed_od_fraction_of_fov"].mean()),
            "removed_od_fraction_of_fov_min": float(group["removed_od_fraction_of_fov"].min()),
            "preserved_outside_od_fov_pixels_mean": float(group["preserved_outside_od_fov_pixels"].mean()),
            "source_manifest": str(outside_manifest_path),
        })
else:
    forensic_rows.append({
        "audit_scope": "existing_notebook10_outside_od_inputs",
        "split": "all",
        "n_images": 0,
        "n_patients": 0,
        "n_positive": 0,
        "n_negative": 0,
        "dilation_pixels": np.nan,
        "removed_od_fraction_of_fov_mean": np.nan,
        "removed_od_fraction_of_fov_min": np.nan,
        "preserved_outside_od_fov_pixels_mean": np.nan,
        "source_manifest": "MISSING",
    })

# 7.2 Existing outside-OD performance from Notebook 10, if available.
if not model_roles.empty:
    mr = model_roles.copy()
    model_id_col = "model_id" if "model_id" in mr.columns else "experiment_id" if "experiment_id" in mr.columns else None
    if model_id_col:
        outside_perf = mr[mr[model_id_col].astype(str).str.contains("outside_od", case=False, na=False)].copy()
        for _, row in outside_perf.iterrows():
            forensic_rows.append({
                "audit_scope": "existing_notebook10_outside_od_performance",
                "split": "test_patient_level",
                "n_images": np.nan,
                "n_patients": np.nan,
                "n_positive": np.nan,
                "n_negative": np.nan,
                "dilation_pixels": np.nan,
                "removed_od_fraction_of_fov_mean": np.nan,
                "removed_od_fraction_of_fov_min": np.nan,
                "preserved_outside_od_fov_pixels_mean": np.nan,
                "source_manifest": str(model_roles_path),
                "model_id": str(row.get(model_id_col, "")),
                "patient_auroc": row.get("patient_auroc", np.nan),
                "patient_auroc_ci95_low": row.get("patient_auroc_ci95_low", np.nan),
                "patient_auroc_ci95_high": row.get("patient_auroc_ci95_high", np.nan),
                "patient_balanced_accuracy": row.get("patient_balanced_accuracy", np.nan),
                "patient_brier_score": row.get("patient_brier_score", np.nan),
            })

forensic_audit = pd.DataFrame(forensic_rows)
forensic_audit.to_csv(OUTSIDE_OD_FORENSIC_AUDIT_CSV, index=False)

# 7.3 Construct conservative OD-removal inputs at larger dilation radii for visual/QC sensitivity.
# This cell does not retrain or select models. It produces inputs and geometry summaries that can be used by Notebook 16 or a Notebook 10 rerun.
CONSERVATIVE_DILATIONS = [7, 14, 21, 32]
OUTPUT_SIZE = 384
CONSERVATIVE_INPUT_ROOT = RESULTS_14_DIR / "outside_od_conservative_inputs_384"
CONSERVATIVE_INPUT_ROOT.mkdir(parents=True, exist_ok=True)


def read_rgb(path):
    path = Path(path)
    if CV2_AVAILABLE:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if PIL_AVAILABLE:
        return np.asarray(Image.open(path).convert("RGB"))
    raise RuntimeError("Neither cv2 nor PIL is available for image reading.")


def read_mask(path):
    path = Path(path)
    if CV2_AVAILABLE:
        mask = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise FileNotFoundError(path)
        if mask.ndim == 3:
            # OD/OC mask convention from earlier notebooks may encode OD/OC in channels or colors.
            # Conservative union: any non-zero pixel.
            mask = np.any(mask > 0, axis=2).astype(np.uint8)
        else:
            mask = (mask > 0).astype(np.uint8)
        return mask.astype(bool)
    if PIL_AVAILABLE:
        arr = np.asarray(Image.open(path).convert("L"))
        return (arr > 0)
    raise RuntimeError("Neither cv2 nor PIL is available for mask reading.")


def compute_fov_mask(rgb):
    arr = rgb.astype(np.uint8)
    gray = arr.mean(axis=2)
    # Conservative non-background estimate robust to black borders.
    return gray > 10


def resize_bool(mask, size):
    if CV2_AVAILABLE:
        out = cv2.resize(mask.astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
        return out.astype(bool)
    img = Image.fromarray(mask.astype(np.uint8) * 255).resize((size, size), Image.Resampling.NEAREST)
    return np.asarray(img) > 0


def resize_rgb(rgb, size):
    if CV2_AVAILABLE:
        return cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    img = Image.fromarray(rgb.astype(np.uint8)).resize((size, size), Image.Resampling.LANCZOS)
    return np.asarray(img)


def dilate_mask(mask, pixels):
    if pixels <= 0:
        return mask.astype(bool)
    if CV2_AVAILABLE:
        k = int(2 * pixels + 1)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        return cv2.dilate(mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    # PIL fallback: approximate by max-filter.
    from PIL import ImageFilter
    img = Image.fromarray(mask.astype(np.uint8) * 255)
    img = img.filter(ImageFilter.MaxFilter(size=int(2 * pixels + 1)))
    return np.asarray(img) > 0


def save_rgb(path, rgb):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if CV2_AVAILABLE:
        cv2.imwrite(str(path), cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2BGR))
    else:
        Image.fromarray(rgb.astype(np.uint8)).save(path)
    return str(path)

candidate = base.copy()
mask_col = None
for c in ["od_oc_mask_path", "mask_512_path", "source_mask_path"]:
    if c in candidate.columns:
        mask_col = c
        break
if mask_col is None and not outside_manifest.empty and "source_mask_path" in outside_manifest.columns:
    outside_tmp = outside_manifest.copy()
    if "sample_id" in outside_tmp.columns and "image_id" not in outside_tmp.columns:
        outside_tmp = outside_tmp.rename(columns={"sample_id": "image_id"})
    candidate = candidate.merge(outside_tmp[["image_id", "source_mask_path"]].drop_duplicates("image_id"), on="image_id", how="left")
    mask_col = "source_mask_path"

if "image_path" not in candidate.columns and not outside_manifest.empty and "source_image_path" in outside_manifest.columns:
    outside_tmp = outside_manifest.copy()
    if "sample_id" in outside_tmp.columns and "image_id" not in outside_tmp.columns:
        outside_tmp = outside_tmp.rename(columns={"sample_id": "image_id"})
    candidate = candidate.merge(outside_tmp[["image_id", "source_image_path"]].drop_duplicates("image_id"), on="image_id", how="left")
    candidate["image_path"] = candidate["source_image_path"]

candidate["image_path"] = candidate.get("image_path", "").fillna("").astype(str)
if mask_col:
    candidate[mask_col] = candidate[mask_col].fillna("").astype(str)
else:
    candidate["missing_mask_path"] = ""

# Use test split when available for conservative visual/QC audit; otherwise use all strict-valid images.
if "split" in candidate.columns:
    build_df = candidate[candidate["split"].astype(str).isin(["test", "holdout_test"])].copy()
    if build_df.empty:
        build_df = candidate.copy()
else:
    build_df = candidate.copy()

if "strict_anatomical_proxy_valid" in build_df.columns:
    strict_flag = build_df["strict_anatomical_proxy_valid"].astype(str).str.lower().isin(["true", "1", "yes", "y"])
    build_df = build_df[strict_flag].copy()

# Generate at most all test images. HYGD is small; this is still manageable.
manifest_rows = []
error_rows = []
if CV2_AVAILABLE or PIL_AVAILABLE:
    for _, row in build_df.iterrows():
        image_path = Path(str(row.get("image_path", "")))
        mask_path = Path(str(row.get(mask_col, ""))) if mask_col else Path("")
        if not image_path.exists() or not mask_path.exists():
            error_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "image_path": str(image_path),
                "mask_path": str(mask_path),
                "error": "missing_image_or_mask",
            })
            continue
        try:
            rgb = read_rgb(image_path)
            mask = read_mask(mask_path)
            rgb_resized = resize_rgb(rgb, OUTPUT_SIZE)
            mask_resized = resize_bool(mask, OUTPUT_SIZE)
            fov_mask = compute_fov_mask(rgb_resized)
            for dilation in CONSERVATIVE_DILATIONS:
                dilated = dilate_mask(mask_resized, dilation) & fov_mask
                outside = fov_mask & (~dilated)
                mean_color = rgb_resized[outside].mean(axis=0) if outside.sum() > 0 else rgb_resized[fov_mask].mean(axis=0)
                out_img = rgb_resized.copy().astype(np.float32)
                out_img[dilated] = mean_color
                out_img = np.clip(out_img, 0, 255).astype(np.uint8)
                out_path = CONSERVATIVE_INPUT_ROOT / f"dilation_{dilation:02d}" / f"{row['image_id']}_outside_od_dilation_{dilation:02d}.png"
                save_rgb(out_path, out_img)
                manifest_rows.append({
                    "image_id": row["image_id"],
                    "patient_id": row.get("patient_id", ""),
                    "split": row.get("split", ""),
                    "gon_binary": row.get("gon_binary", np.nan),
                    "quality_tertile": row.get("quality_tertile", ""),
                    "dilation_pixels": int(dilation),
                    "output_size": int(OUTPUT_SIZE),
                    "source_image_path": str(image_path),
                    "source_mask_path": str(mask_path),
                    "conservative_outside_od_image_path": str(out_path),
                    "fov_pixels": int(fov_mask.sum()),
                    "removed_pixels": int(dilated.sum()),
                    "preserved_outside_od_pixels": int(outside.sum()),
                    "removed_fraction_of_fov": float(dilated.sum() / max(1, fov_mask.sum())),
                    "preserved_fraction_of_fov": float(outside.sum() / max(1, fov_mask.sum())),
                })
        except Exception as exc:
            error_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "image_path": str(image_path),
                "mask_path": str(mask_path),
                "error": repr(exc),
            })

conservative_manifest = pd.DataFrame(manifest_rows)
conservative_manifest.to_csv(OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV, index=False)

if not conservative_manifest.empty:
    conservative_summary = (
        conservative_manifest.groupby("dilation_pixels")
        .agg(
            n_images=("image_id", "nunique"),
            n_patients=("patient_id", "nunique"),
            removed_fraction_of_fov_mean=("removed_fraction_of_fov", "mean"),
            removed_fraction_of_fov_min=("removed_fraction_of_fov", "min"),
            removed_fraction_of_fov_max=("removed_fraction_of_fov", "max"),
            preserved_fraction_of_fov_mean=("preserved_fraction_of_fov", "mean"),
        )
        .reset_index()
    )
else:
    conservative_summary = pd.DataFrame(columns=["dilation_pixels", "n_images", "n_patients", "removed_fraction_of_fov_mean"])
conservative_summary.to_csv(OUTSIDE_OD_CONSERVATIVE_SUMMARY_CSV, index=False)

# 7.4 Compact visual panel for manuscript supplement / internal QC.
if not conservative_manifest.empty:
    import matplotlib.pyplot as plt
    selected_ids = conservative_manifest["image_id"].drop_duplicates().head(4).astype(str).tolist()
    panel_rows = []
    for image_id in selected_ids:
        block = conservative_manifest[conservative_manifest["image_id"].astype(str).eq(image_id)].copy()
        block = block.sort_values("dilation_pixels")
        panel_rows.append(block)
    panel_df = pd.concat(panel_rows, ignore_index=True) if panel_rows else pd.DataFrame()
    if not panel_df.empty:
        n_images = len(selected_ids)
        n_cols = len(CONSERVATIVE_DILATIONS)
        fig, axes = plt.subplots(n_images, n_cols, figsize=(2.4 * n_cols, 2.4 * n_images))
        if n_images == 1:
            axes = np.expand_dims(axes, axis=0)
        for r, image_id in enumerate(selected_ids):
            block = panel_df[panel_df["image_id"].astype(str).eq(image_id)].sort_values("dilation_pixels")
            for c, (_, item) in enumerate(block.iterrows()):
                ax = axes[r, c]
                img = read_rgb(item["conservative_outside_od_image_path"])
                ax.imshow(img)
                ax.set_title(f"d={int(item['dilation_pixels'])}\nremoved={item['removed_fraction_of_fov']:.2f}", fontsize=8)
                ax.axis("off")
        fig.suptitle("Outside-OD conservative mask-removal sensitivity inputs", fontsize=11)
        fig.tight_layout()
        fig.savefig(OUTSIDE_OD_VISUAL_PANEL_PNG, dpi=220, bbox_inches="tight")
        fig.savefig(OUTSIDE_OD_VISUAL_PANEL_PDF, bbox_inches="tight")
        plt.close(fig)

cell7_status = "PASS" if not conservative_manifest.empty else "PASS_WITH_NO_CONSERVATIVE_INPUTS_BUILT"
summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 7",
    "status": cell7_status,
    "existing_outside_manifest_found": bool(outside_manifest_path is not None),
    "existing_outside_model_roles_found": bool(model_roles_path is not None),
    "n_conservative_input_rows": int(len(conservative_manifest)),
    "n_conservative_unique_images": int(conservative_manifest["image_id"].nunique()) if not conservative_manifest.empty else 0,
    "conservative_dilations": CONSERVATIVE_DILATIONS,
    "n_errors": int(len(error_rows)),
    "forensic_audit_csv": str(OUTSIDE_OD_FORENSIC_AUDIT_CSV),
    "conservative_manifest_csv": str(OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV),
    "conservative_summary_csv": str(OUTSIDE_OD_CONSERVATIVE_SUMMARY_CSV),
    "visual_panel_png": str(OUTSIDE_OD_VISUAL_PANEL_PNG) if OUTSIDE_OD_VISUAL_PANEL_PNG.exists() else "",
    "important_note": "This cell constructs conservative outside-OD inputs and geometry QC. It does not retrain or tune models.",
}
with open(CELL7_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Outside-OD forensic summary:")
print(json.dumps(summary, indent=2))
print("\nExisting outside-OD forensic audit:")
print(forensic_audit.to_string(index=False))
print("\nConservative dilation geometry summary:")
print(conservative_summary.to_string(index=False))
if error_rows:
    print("\nFirst conservative-input construction errors:")
    print(pd.DataFrame(error_rows).head(20).to_string(index=False))
print(f"Status: {cell7_status}")


In [ ]:
# Cell 8: SMDG label harmonization and source-level external audit
#
# Reviewer target:
# - Consolidate SMDG source-level external validation into one row per source.
# - Use the actual Notebook 13 artifacts detected by Cell 3:
#     smdg_label_policy
#     smdg_source_label_distribution
#     smdg_source_metrics_ci
#     smdg_source_domain_shift
#     smdg_label_policy_by_source
# - Correctly parse:
#     label distribution columns:
#       smdg_source_dataset, smdg_type_numeric, smdg_label_policy_status_preliminary,
#       n_rows, n_fundus_existing
#     metrics columns:
#       analysis_scope, external_source_dataset, model_name, probability_type,
#       metric, point_estimate, ci_low, ci_high, n_samples, n_positive,
#       n_negative, metric_eligible
# - Do not fit thresholds, temperature, or calibration on SMDG.

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

required_vars = [
    "REVIEWER_SOURCE_MAP_CSV",
    "TABLE_DIR",
    "RESULTS_14_DIR",
    "PRIMARY_MODEL_NAME",
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

# -------------------------------------------------------------------------
# Output paths expected by downstream cells
# -------------------------------------------------------------------------

SMDG_LABEL_POLICY_CSV = TABLE_DIR / "notebook14_smdg_label_harmonization_policy.csv"
SMDG_SOURCE_PREVALENCE_CSV = TABLE_DIR / "notebook14_smdg_source_prevalence.csv"
SMDG_SOURCE_METRICS_COMPACT_CSV = TABLE_DIR / "notebook14_smdg_source_metrics_compact.csv"
SMDG_SOURCE_PREVALENCE_METRIC_CSV = TABLE_DIR / "notebook14_smdg_source_prevalence_with_metrics.csv"
SMDG_SOURCE_METRICS_LONG_CSV = TABLE_DIR / "notebook14_smdg_source_metrics_long.csv"
SMDG_SOURCE_AUDIT_INPUTS_CSV = TABLE_DIR / "notebook14_smdg_source_audit_input_files.csv"
CELL8_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell08_smdg_source_audit_summary.json"


# -------------------------------------------------------------------------
# Utilities
# -------------------------------------------------------------------------

def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def canonical_model_name(x):
    s = norm_text(x)
    aliases = {
        "efficientnetb0": "EfficientNetB0",
        "efficientnet_b0": "EfficientNetB0",
        "tf_efficientnet_b0": "EfficientNetB0",
        "densenet121": "DenseNet121",
        "dense_net_121": "DenseNet121",
        "convnexttiny": "ConvNeXtTiny",
        "convnext_tiny": "ConvNeXtTiny",
        "swintiny": "SwinTiny",
        "swin_tiny": "SwinTiny",
        "swin_tiny_patch4_window7_224": "SwinTiny",
        "resnet18": "ResNet18",
        "resnet_18": "ResNet18",
        "resnet50": "ResNet50",
        "resnet_50": "ResNet50",
    }
    return aliases.get(s, str(x))


def safe_read_csv(path_like):
    if path_like is None:
        return pd.DataFrame()
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        print(f"Could not read {path}: {type(exc).__name__}: {exc}")
        return pd.DataFrame()


def get_role_path(source_map, role):
    block = source_map[source_map["role"].astype(str).eq(role)].copy()
    if block.empty:
        return None

    if "selected" in block.columns:
        selected = block["selected"].astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])
        if selected.any():
            block = block[selected].copy()

    path = Path(str(block.iloc[0].get("path", "")))
    return path if path.exists() else None


def discover_smdg_roles(source_map):
    block = source_map[source_map["role"].astype(str).str.lower().str.contains("smdg", na=False)].copy()
    rows = []

    for _, r in block.iterrows():
        path = Path(str(r.get("path", "")))
        rows.append({
            "role": r.get("role", ""),
            "selected": r.get("selected", ""),
            "path": str(path),
            "exists": bool(path.exists()),
            "file_name": r.get("file_name", path.name),
        })

    return pd.DataFrame(rows)


def first_available_count(row, candidates):
    for c in candidates:
        if c in row.index and pd.notna(row[c]):
            try:
                return int(row[c])
            except Exception:
                pass
    return 0


def probability_type_priority(x):
    s = norm_text(x)

    # Prefer calibrated/temperature-scaled probabilities when Notebook 13 already generated them.
    # This does not fit anything on SMDG; it only selects among precomputed outputs.
    if any(tok in s for tok in ["temperature", "temp", "calibrated", "scaled"]):
        return 0
    if any(tok in s for tok in ["raw", "uncalibrated", "probability"]):
        return 1
    return 2


def metric_priority(x):
    order = {
        "auroc": 0,
        "auc": 0,
        "roc_auc": 0,
        "auprc": 1,
        "average_precision": 1,
        "ap": 1,
        "balanced_accuracy": 2,
        "bal_acc": 2,
        "bacc": 2,
        "sensitivity": 3,
        "specificity": 4,
        "brier": 5,
        "brier_score": 5,
        "ece": 6,
        "ece_10bin": 6,
        "expected_calibration_error": 6,
    }
    return order.get(norm_text(x), 99)


def canonical_metric_name(x):
    s = norm_text(x)
    aliases = {
        "auc": "auroc",
        "roc_auc": "auroc",
        "auc_roc": "auroc",
        "auroc": "auroc",
        "average_precision": "auprc",
        "ap": "auprc",
        "pr_auc": "auprc",
        "auc_pr": "auprc",
        "auprc": "auprc",
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "bacc": "balanced_accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "sensitivity": "sensitivity",
        "specificity": "specificity",
        "brier": "brier",
        "brier_score": "brier",
        "ece": "ece_10bin",
        "ece_10": "ece_10bin",
        "ece_10bin": "ece_10bin",
        "expected_calibration_error": "ece_10bin",
    }
    return aliases.get(s, s)


# -------------------------------------------------------------------------
# Load source map and actual SMDG artifacts
# -------------------------------------------------------------------------

source_map = pd.read_csv(REVIEWER_SOURCE_MAP_CSV)
smdg_roles = discover_smdg_roles(source_map)

paths = {
    "label_policy": get_role_path(source_map, "smdg_label_policy"),
    "label_policy_by_source": get_role_path(source_map, "smdg_label_policy_by_source"),
    "label_distribution": get_role_path(source_map, "smdg_source_label_distribution"),
    "metrics_ci": get_role_path(source_map, "smdg_source_metrics_ci"),
    "domain_shift": get_role_path(source_map, "smdg_source_domain_shift"),
}

label_policy_raw = safe_read_csv(paths["label_policy"])
label_policy_by_source_raw = safe_read_csv(paths["label_policy_by_source"])
label_dist_raw = safe_read_csv(paths["label_distribution"])
metrics_raw = safe_read_csv(paths["metrics_ci"])
domain_shift_raw = safe_read_csv(paths["domain_shift"])

input_audit_rows = []
for role_type, path in paths.items():
    df = safe_read_csv(path)
    input_audit_rows.append({
        "role_type": role_type,
        "path": str(path) if path is not None else "",
        "exists": bool(path is not None and Path(path).exists()),
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]) if not df.empty else 0,
        "columns": ";".join(map(str, df.columns[:100])) if not df.empty else "",
    })

for _, r in smdg_roles.iterrows():
    input_audit_rows.append({
        "role_type": "discovered_smdg_role",
        "path": r.get("path", ""),
        "exists": r.get("exists", False),
        "n_rows": np.nan,
        "n_columns": np.nan,
        "columns": r.get("role", ""),
    })

input_audit = pd.DataFrame(input_audit_rows)
input_audit.to_csv(SMDG_SOURCE_AUDIT_INPUTS_CSV, index=False)

primary_model = canonical_model_name(PRIMARY_MODEL_NAME)


# -------------------------------------------------------------------------
# 8.1 Label policy table
# -------------------------------------------------------------------------

if not label_policy_raw.empty:
    label_policy = label_policy_raw.copy()

    if "policy" not in label_policy.columns:
        label_policy.insert(0, "policy", [f"source_policy_{i+1}" for i in range(len(label_policy))])

    if "manuscript_use" not in label_policy.columns:
        label_policy["manuscript_use"] = "Methods: SMDG label harmonization and source-level external audit."

else:
    label_policy = pd.DataFrame([
        {
            "policy": "binary_target",
            "description": "SMDG labels are harmonized to binary glaucoma/referable-glaucoma versus non-glaucoma/control when eligible.",
            "manuscript_use": "Methods: SMDG label harmonization.",
        },
        {
            "policy": "no_smdg_fitting",
            "description": "No model selection, temperature fitting, threshold fitting, or recalibration is performed on SMDG.",
            "manuscript_use": "Methods: external validation guardrail.",
        },
    ])

label_policy.to_csv(SMDG_LABEL_POLICY_CSV, index=False)


# -------------------------------------------------------------------------
# 8.2 Source prevalence from label distribution
# -------------------------------------------------------------------------

prevalence_rows = []

if not label_dist_raw.empty:
    required_cols = {
        "smdg_source_dataset",
        "smdg_type_numeric",
        "smdg_label_policy_status_preliminary",
    }

    if required_cols.issubset(set(label_dist_raw.columns)):
        df = label_dist_raw.copy()

        # Use existing fundus count when available; otherwise n_rows.
        if "n_fundus_existing" in df.columns:
            df["count_for_prevalence"] = pd.to_numeric(df["n_fundus_existing"], errors="coerce").fillna(0).astype(int)
        elif "n_rows" in df.columns:
            df["count_for_prevalence"] = pd.to_numeric(df["n_rows"], errors="coerce").fillna(0).astype(int)
        else:
            df["count_for_prevalence"] = 0

        df["smdg_type_numeric"] = pd.to_numeric(df["smdg_type_numeric"], errors="coerce")
        df["policy_status_norm"] = df["smdg_label_policy_status_preliminary"].astype(str).str.lower()

        # Keep only binary included labels. Exclude suspects/ambiguous.
        df["is_positive"] = (
            df["policy_status_norm"].str.contains("included_binary_positive", na=False)
            | (df["smdg_type_numeric"] == 1)
        )
        df["is_negative"] = (
            df["policy_status_norm"].str.contains("included_binary_negative", na=False)
            | (df["smdg_type_numeric"] == 0)
        )

        # Do not allow ambiguous -1 rows to enter prevalence.
        df.loc[df["smdg_type_numeric"] < 0, ["is_positive", "is_negative"]] = False

        for source, g in df.groupby("smdg_source_dataset"):
            n_pos = int(g.loc[g["is_positive"], "count_for_prevalence"].sum())
            n_neg = int(g.loc[g["is_negative"], "count_for_prevalence"].sum())
            n_images = n_pos + n_neg

            prevalence_rows.append({
                "source": str(source),
                "n_images": n_images,
                "n_positive": n_pos,
                "n_negative": n_neg,
                "positive_prevalence": float(n_pos / n_images) if n_images > 0 else np.nan,
                "source_has_both_classes": bool(n_pos > 0 and n_neg > 0),
                "prevalence_source": "smdg_source_label_distribution",
            })

    else:
        print("Label distribution table exists but required columns were not found.")
        print("Columns:", list(label_dist_raw.columns))

prevalence = pd.DataFrame(prevalence_rows)

if prevalence.empty:
    prevalence = pd.DataFrame(columns=[
        "source",
        "n_images",
        "n_positive",
        "n_negative",
        "positive_prevalence",
        "source_has_both_classes",
        "prevalence_source",
    ])
else:
    prevalence = prevalence.sort_values("source").reset_index(drop=True)

prevalence.to_csv(SMDG_SOURCE_PREVALENCE_CSV, index=False)


# -------------------------------------------------------------------------
# 8.3 Source metrics from Notebook 13 bootstrap source metrics CI
# -------------------------------------------------------------------------

metrics_compact_rows = []

if not metrics_raw.empty:
    required_cols = {
        "external_source_dataset",
        "model_name",
        "probability_type",
        "metric",
        "point_estimate",
        "ci_low",
        "ci_high",
    }

    if required_cols.issubset(set(metrics_raw.columns)):
        m = metrics_raw.copy()
        m["model_name"] = m["model_name"].apply(canonical_model_name)
        m["metric_canonical"] = m["metric"].apply(canonical_metric_name)
        m["probability_priority"] = m["probability_type"].apply(probability_type_priority)
        m["metric_priority"] = m["metric_canonical"].apply(metric_priority)

        for col in [
            "point_estimate",
            "ci_low",
            "ci_high",
            "bootstrap_mean",
            "bootstrap_std",
            "n_bootstrap",
            "n_bootstrap_valid",
            "n_samples",
            "n_positive",
            "n_negative",
        ]:
            if col in m.columns:
                m[col] = pd.to_numeric(m[col], errors="coerce")

        if "metric_eligible" in m.columns:
            eligible = m["metric_eligible"].astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])
            if eligible.any():
                m = m[eligible].copy()

        # Prefer primary model if present.
        if m["model_name"].eq(primary_model).any():
            m = m[m["model_name"].eq(primary_model)].copy()

        # If multiple probability types exist per source/metric, choose calibrated/temperature-scaled first.
        m = m.sort_values([
            "external_source_dataset",
            "metric_canonical",
            "probability_priority",
            "metric_priority",
        ]).copy()

        chosen_rows = []
        for (source, metric), g in m.groupby(["external_source_dataset", "metric_canonical"]):
            g = g.sort_values("probability_priority").copy()
            chosen_rows.append(g.iloc[0].to_dict())

        chosen = pd.DataFrame(chosen_rows)

        for source, g in chosen.groupby("external_source_dataset"):
            row = {
                "source": str(source),
                "model_name": g["model_name"].dropna().astype(str).iloc[0] if g["model_name"].notna().any() else primary_model,
                "metric_source": "notebook13_smdg_bootstrap_source_metrics_ci_5000",
            }

            # Counts: use max across metrics because they should be identical for a source.
            for count_col in ["n_samples", "n_positive", "n_negative"]:
                if count_col in g.columns:
                    vals = pd.to_numeric(g[count_col], errors="coerce").dropna()
                    row[count_col] = int(vals.max()) if len(vals) else np.nan

            # Store metrics and CIs.
            for _, r in g.iterrows():
                metric = r["metric_canonical"]
                row[metric] = r.get("point_estimate", np.nan)
                row[f"{metric}_ci95_low"] = r.get("ci_low", np.nan)
                row[f"{metric}_ci95_high"] = r.get("ci_high", np.nan)

                if "bootstrap_mean" in r:
                    row[f"{metric}_bootstrap_mean"] = r.get("bootstrap_mean", np.nan)
                if "bootstrap_std" in r:
                    row[f"{metric}_bootstrap_std"] = r.get("bootstrap_std", np.nan)

            metrics_compact_rows.append(row)

    else:
        print("Metrics CI table exists but required columns were not found.")
        print("Columns:", list(metrics_raw.columns))

metrics_compact = pd.DataFrame(metrics_compact_rows)

if metrics_compact.empty:
    metrics_compact = pd.DataFrame(columns=[
        "source",
        "model_name",
        "metric_source",
        "n_samples",
        "n_positive",
        "n_negative",
        "auroc",
        "auroc_ci95_low",
        "auroc_ci95_high",
        "balanced_accuracy",
        "balanced_accuracy_ci95_low",
        "balanced_accuracy_ci95_high",
        "brier",
        "brier_ci95_low",
        "brier_ci95_high",
        "ece_10bin",
        "ece_10bin_ci95_low",
        "ece_10bin_ci95_high",
    ])
else:
    metrics_compact = metrics_compact.sort_values("source").reset_index(drop=True)

metrics_compact.to_csv(SMDG_SOURCE_METRICS_COMPACT_CSV, index=False)


# -------------------------------------------------------------------------
# 8.4 Merge prevalence and metrics
# -------------------------------------------------------------------------

if not prevalence.empty:
    final = prevalence.copy()
else:
    final = pd.DataFrame(columns=["source"])

if not metrics_compact.empty and "source" in metrics_compact.columns:
    final = final.merge(
        metrics_compact,
        on="source",
        how="outer",
        suffixes=("", "_metric"),
    )

# Fill counts from metric table where prevalence is unavailable or zero.
if "n_samples" in final.columns:
    if "n_images" not in final.columns:
        final["n_images"] = final["n_samples"]
    else:
        final["n_images"] = pd.to_numeric(final["n_images"], errors="coerce")
        final["n_samples"] = pd.to_numeric(final["n_samples"], errors="coerce")
        final["n_images"] = final["n_images"].where(final["n_images"] > 0, final["n_samples"])

for col in ["n_positive", "n_negative"]:
    metric_col = f"{col}_metric"
    if metric_col in final.columns:
        if col not in final.columns:
            final[col] = final[metric_col]
        else:
            final[col] = pd.to_numeric(final[col], errors="coerce")
            final[metric_col] = pd.to_numeric(final[metric_col], errors="coerce")
            final[col] = final[col].where(final[col] > 0, final[metric_col])
        final = final.drop(columns=[metric_col])

for col in ["n_images", "n_positive", "n_negative"]:
    if col in final.columns:
        final[col] = pd.to_numeric(final[col], errors="coerce")

if "positive_prevalence" not in final.columns:
    final["positive_prevalence"] = np.nan

if {"n_positive", "n_images"}.issubset(final.columns):
    final["positive_prevalence"] = pd.to_numeric(final["positive_prevalence"], errors="coerce")
    computed_prev = final["n_positive"] / final["n_images"].replace(0, np.nan)
    final["positive_prevalence"] = final["positive_prevalence"].fillna(computed_prev)

if "source_has_both_classes" not in final.columns:
    final["source_has_both_classes"] = False

if {"n_positive", "n_negative"}.issubset(final.columns):
    final["source_has_both_classes"] = (
        final["source_has_both_classes"].astype(str).str.lower().isin(["true", "1", "yes", "y", "t"])
        | ((final["n_positive"] > 0) & (final["n_negative"] > 0))
    )

if "model_name" not in final.columns:
    final["model_name"] = primary_model
else:
    final["model_name"] = final["model_name"].fillna(primary_model).apply(canonical_model_name)

if "metric_source" not in final.columns:
    final["metric_source"] = "notebook13_smdg_bootstrap_source_metrics_ci_5000"

# Numeric cleanup.
for col in final.columns:
    if col not in ["source", "model_name", "metric_source", "prevalence_source"]:
        try:
            final[col] = pd.to_numeric(final[col], errors="coerce")
        except Exception:
            pass

# Flags.
if "auroc" in final.columns:
    final["below_chance_auroc_flag"] = final["auroc"] < 0.5
else:
    final["below_chance_auroc_flag"] = False

if "ece_10bin" in final.columns:
    final["high_ece_flag"] = final["ece_10bin"] >= 0.15
else:
    final["high_ece_flag"] = False

if "n_images" in final.columns:
    final["small_source_flag"] = final["n_images"] < 50
else:
    final["small_source_flag"] = False

# Column order.
preferred_cols = [
    "source",
    "model_name",
    "metric_source",
    "prevalence_source",
    "n_images",
    "n_positive",
    "n_negative",
    "positive_prevalence",
    "source_has_both_classes",
    "auroc",
    "auroc_ci95_low",
    "auroc_ci95_high",
    "auprc",
    "auprc_ci95_low",
    "auprc_ci95_high",
    "balanced_accuracy",
    "balanced_accuracy_ci95_low",
    "balanced_accuracy_ci95_high",
    "sensitivity",
    "specificity",
    "brier",
    "brier_ci95_low",
    "brier_ci95_high",
    "ece_10bin",
    "ece_10bin_ci95_low",
    "ece_10bin_ci95_high",
    "below_chance_auroc_flag",
    "high_ece_flag",
    "small_source_flag",
]

final_cols = [c for c in preferred_cols if c in final.columns] + [c for c in final.columns if c not in preferred_cols]
final = final[final_cols].copy()

if "source" in final.columns:
    final = final.sort_values("source").reset_index(drop=True)

final.to_csv(SMDG_SOURCE_PREVALENCE_METRIC_CSV, index=False)


# -------------------------------------------------------------------------
# 8.5 Long metrics table for supplement
# -------------------------------------------------------------------------

metric_names = ["auroc", "auprc", "balanced_accuracy", "sensitivity", "specificity", "brier", "ece_10bin"]
long_rows = []

for _, r in final.iterrows():
    for metric in metric_names:
        if metric not in final.columns:
            continue

        long_rows.append({
            "source": r.get("source", ""),
            "model_name": r.get("model_name", primary_model),
            "metric": metric,
            "value": r.get(metric, np.nan),
            "ci95_low": r.get(f"{metric}_ci95_low", np.nan),
            "ci95_high": r.get(f"{metric}_ci95_high", np.nan),
            "n_images": r.get("n_images", np.nan),
            "n_positive": r.get("n_positive", np.nan),
            "n_negative": r.get("n_negative", np.nan),
            "positive_prevalence": r.get("positive_prevalence", np.nan),
            "metric_source": r.get("metric_source", ""),
        })

metrics_long = pd.DataFrame(long_rows)
metrics_long.to_csv(SMDG_SOURCE_METRICS_LONG_CSV, index=False)


# -------------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------------

n_sources = int(final["source"].nunique()) if "source" in final.columns else 0
n_sources_with_both_classes = int(final["source_has_both_classes"].sum()) if "source_has_both_classes" in final.columns else 0
n_sources_with_metrics = int(final["auroc"].notna().sum()) if "auroc" in final.columns else 0
n_below_chance = int(final["below_chance_auroc_flag"].sum()) if "below_chance_auroc_flag" in final.columns else 0
n_high_ece = int(final["high_ece_flag"].sum()) if "high_ece_flag" in final.columns else 0

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "notebook14",
    "cell": "Cell 8",
    "status": "PASS" if n_sources_with_metrics > 0 else "PASS_WITHOUT_SMDG_METRICS",
    "primary_model": primary_model,
    "label_policy_source_used": str(paths["label_policy"]) if paths["label_policy"] else "",
    "label_policy_by_source_used": str(paths["label_policy_by_source"]) if paths["label_policy_by_source"] else "",
    "label_distribution_source_used": str(paths["label_distribution"]) if paths["label_distribution"] else "",
    "metric_source_used": str(paths["metrics_ci"]) if paths["metrics_ci"] else "",
    "domain_shift_source_used": str(paths["domain_shift"]) if paths["domain_shift"] else "",
    "n_smdg_roles_discovered": int(len(smdg_roles)),
    "n_sources": n_sources,
    "n_sources_with_both_classes": n_sources_with_both_classes,
    "n_sources_with_metrics": n_sources_with_metrics,
    "n_below_chance_source_aurocs": n_below_chance,
    "n_high_ece_sources": n_high_ece,
    "label_policy_csv": str(SMDG_LABEL_POLICY_CSV),
    "source_prevalence_csv": str(SMDG_SOURCE_PREVALENCE_CSV),
    "source_metrics_compact_csv": str(SMDG_SOURCE_METRICS_COMPACT_CSV),
    "source_prevalence_metric_csv": str(SMDG_SOURCE_PREVALENCE_METRIC_CSV),
    "source_metrics_long_csv": str(SMDG_SOURCE_METRICS_LONG_CSV),
    "input_audit_csv": str(SMDG_SOURCE_AUDIT_INPUTS_CSV),
    "manuscript_guardrail": (
        "Report SMDG as source-level external validation only. This notebook consolidates existing Notebook 13 outputs; "
        "it does not fit temperature, thresholds, recalibration, or model parameters on SMDG."
    ),
}

with open(CELL8_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)


# -------------------------------------------------------------------------
# Console output
# -------------------------------------------------------------------------

print("SMDG source-level external audit summary:")
print(json.dumps(summary, indent=2))

print("\nCell 8 input audit:")
print(input_audit.to_string(index=False))

print("\nSMDG label harmonization policy preview:")
print(label_policy.head(20).to_string(index=False))

print("\nSMDG source prevalence + metrics, one row per source:")
display_cols = [
    "source",
    "n_images",
    "n_positive",
    "n_negative",
    "positive_prevalence",
    "auroc",
    "auroc_ci95_low",
    "auroc_ci95_high",
    "balanced_accuracy",
    "balanced_accuracy_ci95_low",
    "balanced_accuracy_ci95_high",
    "brier",
    "ece_10bin",
    "below_chance_auroc_flag",
    "high_ece_flag",
]
display_cols = [c for c in display_cols if c in final.columns]
print(final[display_cols].to_string(index=False) if len(final) else "No SMDG source rows were generated.")

print("\nStatus:", summary["status"])

In [ ]:
# Cell 9: Compact reviewer-facing figures for ICT Express and supplement
#
# Reviewer target:
# - Produce compact figures supporting:
#   1) HYGD quality-stratified AQPR endpoints.
#   2) SMDG source-level domain shift and prevalence.
#   3) HYGD forensic duplicate/near-duplicate audit.
# - Avoid crashing when optional endpoint families are unavailable.
# - Generate main-text candidates and supplementary candidates separately.

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.errors import EmptyDataError

required_vars = [
    "TABLE_DIR",
    "FIGURE_DATA_DIR",
    "RESULTS_14_DIR",
    "PRIMARY_MODEL_NAME",
]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

TABLE_DIR = Path(TABLE_DIR)
FIGURE_DATA_DIR = Path(FIGURE_DATA_DIR)
RESULTS_14_DIR = Path(RESULTS_14_DIR)
FIGURE_DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_14_DIR.mkdir(parents=True, exist_ok=True)

QUALITY_ORDER = ["low", "medium", "high"]

# -------------------------------------------------------------------------
# Input paths
# -------------------------------------------------------------------------

HYGD_QUALITY_AQPR_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_aqpr_summary.csv"
HYGD_QUALITY_XAI_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_heatmap_enrichment.csv"
HYGD_QUALITY_ABLATION_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_ablation_delta.csv"
HYGD_QUALITY_PERFORMANCE_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_performance.csv"
HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV = TABLE_DIR / "notebook14_hygd_quality_endpoint_availability_audit.csv"

SMDG_SOURCE_PREVALENCE_METRIC_CSV = TABLE_DIR / "notebook14_smdg_source_prevalence_with_metrics.csv"

HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_near_duplicate_threshold_summary.csv"
HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV = TABLE_DIR / "notebook14_hygd_suspicious_cross_split_near_duplicate_pairs.csv"
HYGD_EXACT_DUPLICATE_GROUPS_CSV = TABLE_DIR / "notebook14_hygd_exact_duplicate_groups.csv"

# -------------------------------------------------------------------------
# Output paths
# -------------------------------------------------------------------------

FIGURE_HYGD_QUALITY_AQPR_PNG = FIGURE_DATA_DIR / "notebook14_figure_hygd_quality_stratified_aqpr.png"
FIGURE_HYGD_QUALITY_AQPR_PDF = FIGURE_DATA_DIR / "notebook14_figure_hygd_quality_stratified_aqpr.pdf"

FIGURE_SMDG_SOURCE_SHIFT_PNG = FIGURE_DATA_DIR / "notebook14_figure_smdg_source_shift_prevalence.png"
FIGURE_SMDG_SOURCE_SHIFT_PDF = FIGURE_DATA_DIR / "notebook14_figure_smdg_source_shift_prevalence.pdf"

FIGURE_HYGD_FORENSIC_AUDIT_PNG = FIGURE_DATA_DIR / "notebook14_figure_hygd_forensic_duplicate_audit.png"
FIGURE_HYGD_FORENSIC_AUDIT_PDF = FIGURE_DATA_DIR / "notebook14_figure_hygd_forensic_duplicate_audit.pdf"

FIGURE_MANUSCRIPT_CANDIDATES_CSV = FIGURE_DATA_DIR / "notebook14_figure_candidates_for_manuscript.csv"
CELL9_INPUT_AUDIT_CSV = FIGURE_DATA_DIR / "notebook14_cell09_input_table_audit.csv"
CELL9_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell09_compact_figures_summary.json"


# -------------------------------------------------------------------------
# Utilities
# -------------------------------------------------------------------------

def safe_read_csv(path_like, label):
    path = Path(path_like)

    audit = {
        "label": label,
        "path": str(path),
        "exists": bool(path.exists()),
        "size_bytes": int(path.stat().st_size) if path.exists() else 0,
        "status": "not_loaded",
        "n_rows": 0,
        "n_columns": 0,
        "columns": "",
    }

    if not path.exists():
        audit["status"] = "file_not_found"
        return pd.DataFrame(), audit

    if path.stat().st_size == 0:
        audit["status"] = "empty_file_zero_bytes"
        return pd.DataFrame(), audit

    try:
        df = pd.read_csv(path, low_memory=False)
    except EmptyDataError:
        audit["status"] = "empty_data_error"
        return pd.DataFrame(), audit
    except Exception as exc:
        audit["status"] = f"read_error:{type(exc).__name__}:{exc}"
        return pd.DataFrame(), audit

    audit["n_rows"] = int(len(df))
    audit["n_columns"] = int(df.shape[1])
    audit["columns"] = ";".join(map(str, df.columns[:100]))
    audit["status"] = "loaded" if not df.empty else "empty_dataframe"

    return df, audit


def norm_text(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).lower()).strip("_")


def normalize_quality(x):
    x = str(x).strip().lower()
    mapping = {
        "0": "low",
        "1": "medium",
        "2": "high",
        "low_quality": "low",
        "medium_quality": "medium",
        "high_quality": "high",
        "nan": "unknown",
        "none": "unknown",
        "null": "unknown",
        "": "unknown",
    }
    return mapping.get(x, x)


def first_existing_col(df, candidates=None, contains_all=None, contains_any=None):
    if df is None or df.empty:
        return None

    candidates = candidates or []
    norm_to_col = {norm_text(c): c for c in df.columns}

    for cand in candidates:
        key = norm_text(cand)
        if key in norm_to_col:
            return norm_to_col[key]

    contains_all = [norm_text(x) for x in (contains_all or [])]
    contains_any = [norm_text(x) for x in (contains_any or [])]

    for c in df.columns:
        nc = norm_text(c)
        ok_all = all(tok in nc for tok in contains_all) if contains_all else True
        ok_any = any(tok in nc for tok in contains_any) if contains_any else True
        if ok_all and ok_any:
            return c

    return None


def numeric_col(df, col):
    if col is None or col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(df[col], errors="coerce")


def quality_sort_df(df):
    if df.empty or "quality_tertile" not in df.columns:
        return df
    out = df.copy()
    out["quality_tertile"] = out["quality_tertile"].apply(normalize_quality)
    out["quality_tertile"] = pd.Categorical(
        out["quality_tertile"],
        categories=QUALITY_ORDER,
        ordered=True,
    )
    return out.sort_values("quality_tertile").reset_index(drop=True)


def get_quality_series(df, candidates):
    col = first_existing_col(df, candidates)
    if col is None:
        return None, None
    return col, numeric_col(df, col)


def fmt_metric_with_ci(row, metric):
    val = row.get(metric, np.nan)
    lo = row.get(f"{metric}_ci95_low", np.nan)
    hi = row.get(f"{metric}_ci95_high", np.nan)

    if pd.isna(val):
        return "NA"
    if pd.notna(lo) and pd.notna(hi):
        return f"{val:.3f} [{lo:.3f}, {hi:.3f}]"
    return f"{val:.3f}"


# -------------------------------------------------------------------------
# Load input tables
# -------------------------------------------------------------------------

input_audits = []

quality_summary, audit = safe_read_csv(HYGD_QUALITY_AQPR_SUMMARY_CSV, "hygd_quality_aqpr_summary")
input_audits.append(audit)

quality_xai, audit = safe_read_csv(HYGD_QUALITY_XAI_CSV, "hygd_quality_xai")
input_audits.append(audit)

quality_ablation, audit = safe_read_csv(HYGD_QUALITY_ABLATION_CSV, "hygd_quality_ablation")
input_audits.append(audit)

quality_performance, audit = safe_read_csv(HYGD_QUALITY_PERFORMANCE_CSV, "hygd_quality_performance")
input_audits.append(audit)

quality_endpoint_availability, audit = safe_read_csv(HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV, "hygd_quality_endpoint_availability")
input_audits.append(audit)

smdg_source, audit = safe_read_csv(SMDG_SOURCE_PREVALENCE_METRIC_CSV, "smdg_source_prevalence_metric")
input_audits.append(audit)

near_threshold, audit = safe_read_csv(HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV, "hygd_near_duplicate_threshold_summary")
input_audits.append(audit)

suspicious_pairs, audit = safe_read_csv(HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV, "hygd_suspicious_near_duplicate_pairs")
input_audits.append(audit)

exact_duplicate_groups, audit = safe_read_csv(HYGD_EXACT_DUPLICATE_GROUPS_CSV, "hygd_exact_duplicate_groups")
input_audits.append(audit)

pd.DataFrame(input_audits).to_csv(CELL9_INPUT_AUDIT_CSV, index=False)


# -------------------------------------------------------------------------
# Figure 1: HYGD quality-stratified AQPR endpoints
# -------------------------------------------------------------------------

quality_figure_generated = False
quality_figure_reason = ""

if not quality_summary.empty and "quality_tertile" in quality_summary.columns:
    qdf = quality_sort_df(quality_summary)

    # Ensure all tertiles appear.
    template = pd.DataFrame({
        "quality_tertile": pd.Categorical(QUALITY_ORDER, categories=QUALITY_ORDER, ordered=True)
    })
    qdf = template.merge(qdf, on="quality_tertile", how="left")

    # Candidate columns from Cell 6.
    auroc_col, auroc_values = get_quality_series(qdf, [
        "primary_patient_auroc",
        "primary_auroc",
        "patient_auroc",
        "auroc",
    ])

    ece_col, ece_values = get_quality_series(qdf, [
        "primary_patient_ece_10bin",
        "primary_ece_10bin",
        "patient_ece_10bin",
        "ece_10bin",
    ])

    xai_col, xai_values = get_quality_series(qdf, [
        "primary_ig_od_oc_observed_minus_random_mean",
        "primary_ig_od_oc_observed_minus_random_mean_mean",
        "primary_ig_od_oc_observed_minus_random",
        "observed_minus_random_mean",
        "ig_od_oc_enrichment",
    ])

    blur_col, blur_values = get_quality_series(qdf, [
        "primary_od_oc_blur_mean_probability_drop_vs_original",
        "primary_od_oc_blur_mean_probability_drop",
        "od_oc_blur_probability_drop",
        "mean_probability_drop_vs_original",
    ])

    random_blur_col, random_blur_values = get_quality_series(qdf, [
        "primary_matched_random_blur_mean_probability_drop",
    ])

    strict_proxy_col, strict_proxy_values = get_quality_series(qdf, [
        "strict_proxy_fraction",
        "strict_valid_fraction",
    ])

    cdr_col, cdr_values = get_quality_series(qdf, [
        "cdr_proxy_mean",
        "area_cdr_proxy_mean",
    ])

    n_col, n_values = get_quality_series(qdf, [
        "n_images",
    ])

    x = qdf["quality_tertile"].astype(str).tolist()

    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.2))

    # Panel A: available performance/calibration or proxy validity fallback.
    ax = axes[0]
    if auroc_col is not None and auroc_values.notna().any():
        ax.plot(x, auroc_values, marker="o")
        ax.set_ylabel("AUROC")
        ax.set_ylim(0.0, 1.05)
        ax.set_title("Performance")
    elif strict_proxy_col is not None and strict_proxy_values.notna().any():
        ax.plot(x, strict_proxy_values, marker="o")
        ax.set_ylabel("Strict-valid proxy fraction")
        ax.set_ylim(0.0, 1.05)
        ax.set_title("Proxy QC")
    else:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Performance / QC")
    ax.set_xlabel("Quality tertile")

    # Panel B: calibration if available; otherwise CDR proxy.
    ax = axes[1]
    if ece_col is not None and ece_values.notna().any():
        ax.plot(x, ece_values, marker="o")
        ax.set_ylabel("ECE")
        ax.set_title("Calibration")
    elif cdr_col is not None and cdr_values.notna().any():
        ax.plot(x, cdr_values, marker="o")
        ax.set_ylabel("Mean CDR proxy")
        ax.set_title("Proxy CDR")
    else:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Calibration / CDR")
    ax.set_xlabel("Quality tertile")

    # Panel C: XAI enrichment.
    ax = axes[2]
    if xai_col is not None and xai_values.notna().any():
        ax.plot(x, xai_values, marker="o")
        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_ylabel("Observed - random")
        ax.set_title("IG OD+OC enrichment")
    else:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("XAI enrichment")
    ax.set_xlabel("Quality tertile")

    # Panel D: perturbation response.
    ax = axes[3]
    if blur_col is not None and blur_values.notna().any():
        ax.plot(x, blur_values, marker="o", label="OD+OC")
        if random_blur_col is not None and random_blur_values.notna().any():
            ax.plot(x, random_blur_values, marker="o", linestyle="--", label="matched random")
            ax.legend(fontsize=7)
        ax.axhline(0, linestyle="--", linewidth=1)
        ax.set_ylabel("Probability drop")
        ax.set_title("Perturbation response")
    else:
        ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Perturbation")
    ax.set_xlabel("Quality tertile")

    # Add small n annotation if available.
    if n_col is not None and n_values.notna().any():
        for ax in axes:
            ylim = ax.get_ylim()
            y0 = ylim[0] + 0.03 * (ylim[1] - ylim[0])
            for xi, nv in zip(range(len(x)), n_values):
                if pd.notna(nv):
                    ax.text(xi, y0, f"n={int(nv)}", ha="center", va="bottom", fontsize=7)

    fig.suptitle("HYGD quality-stratified AQPR audit", fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURE_HYGD_QUALITY_AQPR_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_HYGD_QUALITY_AQPR_PDF, bbox_inches="tight")
    plt.close(fig)

    quality_figure_generated = True
    quality_figure_reason = (
        "Generated from Cell 6 quality summary. Missing endpoints are shown using available fallback QC/proxy endpoints."
    )
else:
    quality_figure_reason = (
        "Skipped because the quality summary table was unavailable or lacked quality_tertile."
    )


# -------------------------------------------------------------------------
# Figure 2: SMDG source-level external transportability and prevalence
# -------------------------------------------------------------------------

smdg_figure_generated = False
smdg_figure_reason = ""

if not smdg_source.empty and "source" in smdg_source.columns:
    sdf = smdg_source.copy()

    for col in [
        "n_images", "n_positive", "n_negative", "positive_prevalence",
        "auroc", "auroc_ci95_low", "auroc_ci95_high",
        "balanced_accuracy", "brier", "ece_10bin",
    ]:
        if col in sdf.columns:
            sdf[col] = pd.to_numeric(sdf[col], errors="coerce")

    if "auroc" in sdf.columns and sdf["auroc"].notna().any():
        sdf = sdf.sort_values("auroc", ascending=True).reset_index(drop=True)

        fig_h = max(3.6, 0.28 * len(sdf))
        fig, ax = plt.subplots(figsize=(9.5, fig_h))

        y = np.arange(len(sdf))
        auroc = sdf["auroc"].astype(float).values

        # Horizontal AUROC bars.
        ax.barh(y, auroc)

        # CI lines if available.
        if {"auroc_ci95_low", "auroc_ci95_high"}.issubset(sdf.columns):
            lo = pd.to_numeric(sdf["auroc_ci95_low"], errors="coerce").values
            hi = pd.to_numeric(sdf["auroc_ci95_high"], errors="coerce").values

            for yi, v, l, h in zip(y, auroc, lo, hi):
                if pd.notna(l) and pd.notna(h):
                    ax.plot([l, h], [yi, yi], linewidth=1.2)
                    ax.plot([l, l], [yi - 0.08, yi + 0.08], linewidth=1.0)
                    ax.plot([h, h], [yi - 0.08, yi + 0.08], linewidth=1.0)

        labels = []
        for _, row in sdf.iterrows():
            src = str(row.get("source", "unknown"))
            n = row.get("n_images", np.nan)
            prev = row.get("positive_prevalence", np.nan)

            if pd.notna(n) and pd.notna(prev):
                labels.append(f"{src} (n={int(n)}, pos={prev:.2f})")
            elif pd.notna(n):
                labels.append(f"{src} (n={int(n)})")
            else:
                labels.append(src)

        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=7)
        ax.set_xlim(0.0, 1.05)
        ax.axvline(0.5, linestyle="--", linewidth=1)
        ax.set_xlabel("AUROC")
        ax.set_title("SMDG source-level transportability")

        # Optional prevalence as top x-axis markers.
        if "positive_prevalence" in sdf.columns and sdf["positive_prevalence"].notna().any():
            ax2 = ax.twiny()
            prev = sdf["positive_prevalence"].values
            ax2.plot(prev, y, marker="o", linestyle="", markersize=4)
            ax2.set_xlim(0.0, 1.0)
            ax2.set_xlabel("Positive prevalence")

        fig.tight_layout()
        fig.savefig(FIGURE_SMDG_SOURCE_SHIFT_PNG, dpi=300, bbox_inches="tight")
        fig.savefig(FIGURE_SMDG_SOURCE_SHIFT_PDF, bbox_inches="tight")
        plt.close(fig)

        smdg_figure_generated = True
        smdg_figure_reason = "Generated from one-row-per-source SMDG table with prevalence and AUROC."
    else:
        smdg_figure_reason = "Skipped because no numeric AUROC column was available in SMDG source table."
else:
    smdg_figure_reason = "Skipped because SMDG source table was unavailable or lacked source column."


# -------------------------------------------------------------------------
# Figure 3: HYGD forensic duplicate / near-duplicate audit
# -------------------------------------------------------------------------

forensic_figure_generated = False
forensic_figure_reason = ""

if not near_threshold.empty and {"threshold", "criterion", "n_cross_split_pairs"}.issubset(near_threshold.columns):
    ntdf = near_threshold.copy()
    ntdf["threshold"] = pd.to_numeric(ntdf["threshold"], errors="coerce")
    ntdf["n_cross_split_pairs"] = pd.to_numeric(ntdf["n_cross_split_pairs"], errors="coerce")

    # Prefer dHash-only and consensus curves.
    dhash = ntdf[ntdf["criterion"].astype(str).eq("dhash_only")].copy()
    consensus = ntdf[ntdf["criterion"].astype(str).str.contains("consensus", case=False, na=False)].copy()

    fig, axes = plt.subplots(1, 3, figsize=(12.8, 3.2))

    # Panel A: threshold curve.
    ax = axes[0]
    if not dhash.empty:
        ax.plot(dhash["threshold"], dhash["n_cross_split_pairs"], marker="o", label="dHash")
    if not consensus.empty:
        ax.plot(consensus["threshold"], consensus["n_cross_split_pairs"], marker="o", linestyle="--", label="consensus")
    ax.set_xlabel("Hash-distance threshold")
    ax.set_ylabel("Cross-split candidate pairs")
    ax.set_title("Near-duplicate sensitivity")
    ax.legend(fontsize=7)

    # Panel B: suspicious pair composition.
    ax = axes[1]
    if not suspicious_pairs.empty:
        total = len(suspicious_pairs)
        same_label = int(suspicious_pairs["same_label"].sum()) if "same_label" in suspicious_pairs.columns else 0
        same_patient = int(suspicious_pairs["same_patient"].sum()) if "same_patient" in suspicious_pairs.columns else 0
        mixed_label = total - same_label if "same_label" in suspicious_pairs.columns else np.nan
        distinct_patient = total - same_patient if "same_patient" in suspicious_pairs.columns else np.nan

        names = ["all", "same label", "mixed label", "same patient", "distinct patient"]
        vals = [total, same_label, mixed_label, same_patient, distinct_patient]
        ax.bar(names, vals)
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("Pairs")
        ax.set_title("Suspicious pair composition")
    else:
        ax.text(0.5, 0.5, "No suspicious pairs", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Suspicious pair composition")

    # Panel C: exact duplicate groups.
    ax = axes[2]
    if not exact_duplicate_groups.empty:
        total_groups = len(exact_duplicate_groups)
        cross_split = int(exact_duplicate_groups["cross_split"].sum()) if "cross_split" in exact_duplicate_groups.columns else 0
        cross_patient = int(exact_duplicate_groups["cross_patient"].sum()) if "cross_patient" in exact_duplicate_groups.columns else 0
        mixed_labels = int(exact_duplicate_groups["mixed_labels"].sum()) if "mixed_labels" in exact_duplicate_groups.columns else 0

        names = ["all groups", "cross split", "cross patient", "mixed label"]
        vals = [total_groups, cross_split, cross_patient, mixed_labels]
        ax.bar(names, vals)
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("Exact duplicate groups")
        ax.set_title("Exact SHA256 duplicates")
    else:
        ax.text(0.5, 0.5, "No exact duplicate groups", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Exact SHA256 duplicates")

    fig.suptitle("HYGD forensic split-similarity audit", fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURE_HYGD_FORENSIC_AUDIT_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_HYGD_FORENSIC_AUDIT_PDF, bbox_inches="tight")
    plt.close(fig)

    forensic_figure_generated = True
    forensic_figure_reason = "Generated from Cell 5 duplicate and near-duplicate forensic outputs."
else:
    forensic_figure_reason = "Skipped because near-duplicate threshold summary was unavailable or incomplete."


# -------------------------------------------------------------------------
# Figure candidates table
# -------------------------------------------------------------------------

figure_candidates = pd.DataFrame([
    {
        "figure_id": "Fig_main_candidate_quality_aqpr",
        "path_png": str(FIGURE_HYGD_QUALITY_AQPR_PNG) if FIGURE_HYGD_QUALITY_AQPR_PNG.exists() else "",
        "path_pdf": str(FIGURE_HYGD_QUALITY_AQPR_PDF) if FIGURE_HYGD_QUALITY_AQPR_PDF.exists() else "",
        "generated": bool(quality_figure_generated),
        "recommended_location": "Main or Supplement",
        "purpose": "Responds to reviewer request for Quality-stratified AQPR endpoints.",
        "space_cost": "Medium; four compact panels.",
        "notes": quality_figure_reason,
    },
    {
        "figure_id": "Fig_main_candidate_smdg_source_shift",
        "path_png": str(FIGURE_SMDG_SOURCE_SHIFT_PNG) if FIGURE_SMDG_SOURCE_SHIFT_PNG.exists() else "",
        "path_pdf": str(FIGURE_SMDG_SOURCE_SHIFT_PDF) if FIGURE_SMDG_SOURCE_SHIFT_PDF.exists() else "",
        "generated": bool(smdg_figure_generated),
        "recommended_location": "Main if space allows; otherwise Supplement",
        "purpose": "Shows source-level external transportability and prevalence.",
        "space_cost": "Medium to high depending on number of sources.",
        "notes": smdg_figure_reason,
    },
    {
        "figure_id": "Fig_supp_candidate_hygd_forensic_audit",
        "path_png": str(FIGURE_HYGD_FORENSIC_AUDIT_PNG) if FIGURE_HYGD_FORENSIC_AUDIT_PNG.exists() else "",
        "path_pdf": str(FIGURE_HYGD_FORENSIC_AUDIT_PDF) if FIGURE_HYGD_FORENSIC_AUDIT_PDF.exists() else "",
        "generated": bool(forensic_figure_generated),
        "recommended_location": "Supplement",
        "purpose": "Documents duplicate/near-duplicate forensic audit requested by reviewer.",
        "space_cost": "Medium; likely too detailed for ICT Express main text.",
        "notes": forensic_figure_reason,
    },
])

figure_candidates.to_csv(FIGURE_MANUSCRIPT_CANDIDATES_CSV, index=False)


# -------------------------------------------------------------------------
# Final summary
# -------------------------------------------------------------------------

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "notebook14",
    "cell": "Cell 9",
    "status": "PASS",
    "quality_figure_generated": bool(quality_figure_generated),
    "quality_figure_reason": quality_figure_reason,
    "hygd_quality_figure_png": str(FIGURE_HYGD_QUALITY_AQPR_PNG) if FIGURE_HYGD_QUALITY_AQPR_PNG.exists() else "",
    "hygd_quality_figure_pdf": str(FIGURE_HYGD_QUALITY_AQPR_PDF) if FIGURE_HYGD_QUALITY_AQPR_PDF.exists() else "",
    "smdg_figure_generated": bool(smdg_figure_generated),
    "smdg_figure_reason": smdg_figure_reason,
    "smdg_source_shift_figure_png": str(FIGURE_SMDG_SOURCE_SHIFT_PNG) if FIGURE_SMDG_SOURCE_SHIFT_PNG.exists() else "",
    "smdg_source_shift_figure_pdf": str(FIGURE_SMDG_SOURCE_SHIFT_PDF) if FIGURE_SMDG_SOURCE_SHIFT_PDF.exists() else "",
    "forensic_figure_generated": bool(forensic_figure_generated),
    "forensic_figure_reason": forensic_figure_reason,
    "hygd_forensic_figure_png": str(FIGURE_HYGD_FORENSIC_AUDIT_PNG) if FIGURE_HYGD_FORENSIC_AUDIT_PNG.exists() else "",
    "hygd_forensic_figure_pdf": str(FIGURE_HYGD_FORENSIC_AUDIT_PDF) if FIGURE_HYGD_FORENSIC_AUDIT_PDF.exists() else "",
    "figure_candidates_csv": str(FIGURE_MANUSCRIPT_CANDIDATES_CSV),
    "input_audit_csv": str(CELL9_INPUT_AUDIT_CSV),
    "cell9_input_audits": input_audits,
}

with open(CELL9_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Compact figure generation summary:")
print(json.dumps(summary, indent=2))

print("\nFigure candidates:")
print(figure_candidates.to_string(index=False))

print("\nCell 9 input audit:")
print(pd.DataFrame(input_audits).to_string(index=False))

print("\nStatus: PASS")

In [ ]:
# Cell 10: Expanded quality, proxy-QC, patient-aggregation robustness, and final Notebook 16 handoff
#
# Purpose:
# - Consolidate reviewer-facing evidence from Notebook 14.
# - Add formal HYGD quality-score definition and tertile thresholds.
# - Add proxy strict-valid / proxy-quality evidence summary.
# - Add patient-level aggregation robustness:
#     mean, median, max, quality-weighted mean.
# - Produce a clean handoff for Notebook 16 / manuscript v3.2.
#
# This cell does not train models and does not run inference.
# It reads existing Notebook 14 outputs plus the selected HYGD prediction artifact when available.

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception as e:
    print(f"Drive mount warning: {e}")

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    confusion_matrix,
)

# -------------------------------------------------------------------------
# Required setup
# -------------------------------------------------------------------------

required_vars = [
    "TABLE_DIR",
    "FIGURE_DATA_DIR",
    "RESULTS_14_DIR",
    "PRIMARY_MODEL_NAME",
]

missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run Cell 1 first. Missing variables: {missing_vars}")

TABLE_DIR = Path(TABLE_DIR)
FIGURE_DATA_DIR = Path(FIGURE_DATA_DIR)
RESULTS_14_DIR = Path(RESULTS_14_DIR)
HANDOFF_DIR = RESULTS_14_DIR / "handoff"

for path in [TABLE_DIR, FIGURE_DATA_DIR, HANDOFF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

QUALITY_ORDER = ["low", "medium", "high"]
BOOTSTRAP_REPEATS = 5000
BOOTSTRAP_SEED = 411
PRIMARY_MODEL_NAME = str(PRIMARY_MODEL_NAME)

# -------------------------------------------------------------------------
# Input paths from previous Notebook 14 cells
# -------------------------------------------------------------------------

REVIEWER_SOURCE_MAP_CSV = TABLE_DIR / "notebook14_reviewer_required_source_map.csv"
HYGD_UNIFIED_BASE_CSV = TABLE_DIR / "notebook14_hygd_unified_aqpr_base_manifest.csv"

HYGD_QUALITY_AQPR_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_quality_stratified_aqpr_summary.csv"
HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV = TABLE_DIR / "notebook14_hygd_quality_endpoint_availability_audit.csv"
HYGD_PROXY_QC_BY_QUALITY_CSV = TABLE_DIR / "notebook14_hygd_proxy_qc_by_quality_and_label.csv"

HYGD_SPLIT_PATIENT_LEAKAGE_CSV = TABLE_DIR / "notebook14_hygd_split_patient_leakage_audit.csv"
HYGD_EXACT_DUPLICATE_GROUPS_CSV = TABLE_DIR / "notebook14_hygd_exact_duplicate_groups.csv"
HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV = TABLE_DIR / "notebook14_hygd_near_duplicate_threshold_summary.csv"
HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV = TABLE_DIR / "notebook14_hygd_suspicious_cross_split_near_duplicate_pairs.csv"

OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV = TABLE_DIR / "notebook14_outside_od_conservative_mask_sensitivity_manifest.csv"
OUTSIDE_OD_CONSERVATIVE_SUMMARY_CSV = TABLE_DIR / "notebook14_outside_od_conservative_mask_sensitivity_summary.csv"

SMDG_SOURCE_METRICS_COMPACT_CSV = TABLE_DIR / "notebook14_smdg_source_metrics_compact.csv"
SMDG_SOURCE_PREVALENCE_METRIC_CSV = TABLE_DIR / "notebook14_smdg_source_prevalence_with_metrics.csv"
SMDG_LABEL_POLICY_CSV = TABLE_DIR / "notebook14_smdg_label_harmonization_policy.csv"

FIGURE_MANUSCRIPT_CANDIDATES_CSV = FIGURE_DATA_DIR / "notebook14_figure_candidates_for_manuscript.csv"

# Optional updated outputs from Notebook 13 and Notebook 15.
NOTEBOOK13_CALIBRATION_HANDOFF_JSON = (
    RESULTS_DIR
    / "13_external_validation_dataset_selection_and_preparation"
    / "smdg_journal_ready_package"
    / "notebook13_smdg_calibration_handoff_for_notebook16.json"
)

NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV = (
    RESULTS_DIR
    / "15_aqpr_expanded_core_audit_all_hygd_v31"
    / "tables"
    / "notebook15_expanded_model_claim_boundary_full.csv"
)

NOTEBOOK15_CORE_ENDPOINT_SUMMARY_CSV = (
    RESULTS_DIR
    / "15_aqpr_expanded_core_audit_all_hygd_v31"
    / "tables"
    / "notebook15_expanded_stratified_core_endpoint_summary.csv"
)

# -------------------------------------------------------------------------
# New expanded outputs
# -------------------------------------------------------------------------

HYGD_QUALITY_SCORE_DEFINITION_CSV = TABLE_DIR / "notebook14_hygd_quality_score_definition.csv"
HYGD_QUALITY_SCORE_DEFINITION_MD = TABLE_DIR / "notebook14_hygd_quality_score_definition.md"

HYGD_QUALITY_TERTILE_THRESHOLDS_CSV = TABLE_DIR / "notebook14_hygd_quality_tertile_thresholds.csv"
HYGD_QUALITY_TERTILE_THRESHOLDS_MD = TABLE_DIR / "notebook14_hygd_quality_tertile_thresholds.md"

HYGD_PROXY_QUALITY_EVIDENCE_CSV = TABLE_DIR / "notebook14_hygd_proxy_quality_evidence_summary.csv"
HYGD_PROXY_QUALITY_EVIDENCE_MD = TABLE_DIR / "notebook14_hygd_proxy_quality_evidence_summary.md"

HYGD_PROXY_CDR_PATIENT_DIFF_CSV = TABLE_DIR / "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.csv"
HYGD_PROXY_CDR_PATIENT_DIFF_MD = TABLE_DIR / "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.md"

HYGD_PROXY_FAILURE_AUDIT_CSV = TABLE_DIR / "notebook14_hygd_proxy_failure_audit.csv"
HYGD_PROXY_FAILURE_AUDIT_MD = TABLE_DIR / "notebook14_hygd_proxy_failure_audit.md"

HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_CSV = TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_long.csv"
HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_MD = TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_long.md"

HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV = TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.csv"
HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_MD = TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.md"

NOTEBOOK14_REVIEWER_RESPONSE_MATRIX_CSV = TABLE_DIR / "notebook14_reviewer_response_evidence_matrix.csv"
NOTEBOOK14_MANUSCRIPT_EVIDENCE_SUMMARY_CSV = TABLE_DIR / "notebook14_manuscript_evidence_summary.csv"
NOTEBOOK14_MAIN_TEXT_CANDIDATE_TABLE_CSV = TABLE_DIR / "notebook14_main_text_candidate_table.csv"
NOTEBOOK14_SUPPLEMENTARY_CANDIDATE_TABLE_CSV = TABLE_DIR / "notebook14_supplementary_candidate_table.csv"

NOTEBOOK14_TO_NOTEBOOK16_ACTION_ITEMS_CSV = TABLE_DIR / "notebook14_to_notebook16_action_items.csv"
NOTEBOOK14_FINAL_HANDOFF_JSON = HANDOFF_DIR / "notebook14_quality_proxy_patient_aggregation_handoff_for_notebook16.json"
NOTEBOOK14_FINAL_HANDOFF_TXT = RESULTS_14_DIR / "notebook14_FINAL_HANDOFF_FOR_NOTEBOOK16.txt"
CELL10_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_cell10_expanded_quality_proxy_patient_handoff_summary.json"

# -------------------------------------------------------------------------
# Utility functions
# -------------------------------------------------------------------------

def safe_read_csv(path_like):
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as e:
        print(f"Could not read CSV {path}: {e}")
        return pd.DataFrame()

def safe_read_json(path_like):
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Could not read JSON {path}: {e}")
        return {}

def save_csv_md(df, csv_path, md_path=None, max_rows=120):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    if md_path is not None:
        md_path = Path(md_path)
        md_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            df.head(max_rows).to_markdown(md_path, index=False)
        except Exception:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(df.head(max_rows).to_string(index=False))
    return str(csv_path)

def normalize_quality_value(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    mapping = {
        "0": "low",
        "1": "medium",
        "2": "high",
        "q1": "low",
        "q2": "medium",
        "q3": "high",
        "low_quality": "low",
        "medium_quality": "medium",
        "high_quality": "high",
    }
    return mapping.get(s, s if s not in ["", "nan", "none"] else "unknown")

def normalize_bool(x):
    if pd.isna(x):
        return False
    return str(x).strip().lower() in ["1", "true", "t", "yes", "y"]

def canonical_model_name(x):
    s = str(x)
    sl = s.lower().replace("-", "").replace("_", "")
    if "efficientnet" in sl and "b0" in sl:
        return "EfficientNetB0"
    if "convnext" in sl and "tiny" in sl:
        return "ConvNeXtTiny"
    if "densenet" in sl and "121" in sl:
        return "DenseNet121"
    if "swintiny" in sl or ("swin" in sl and "tiny" in sl):
        return "SwinTiny"
    return s

def canonical_ablation_mode(x):
    s = str(x).strip().lower()
    if s in ["", "nan", "none"]:
        return "unknown"
    if any(t in s for t in ["original", "full_image", "full-image", "baseline", "unperturbed"]):
        return "original_full_image"
    return s

def find_col(df, exact_names=None, contains_any=None, contains_all=None):
    if df is None or df.empty:
        return None

    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    if exact_names:
        for name in exact_names:
            if name.lower() in lower_map:
                return lower_map[name.lower()]

    for c in cols:
        lc = c.lower()
        if contains_all and not all(tok.lower() in lc for tok in contains_all):
            continue
        if contains_any and not any(tok.lower() in lc for tok in contains_any):
            continue
        return c

    return None

def coalesce_duplicate_columns(df):
    if df is None or df.empty:
        return df

    out = df.copy()
    if pd.Index(out.columns).is_unique:
        return out

    new_cols = []
    seen = {}
    for col in out.columns:
        if col not in seen:
            seen[col] = 0
            new_cols.append(col)
        else:
            seen[col] += 1
            new_cols.append(f"{col}__dup{seen[col]}")
    out.columns = new_cols

    base_names = {}
    for col in out.columns:
        base = re.sub(r"__dup\d+$", "", col)
        base_names.setdefault(base, []).append(col)

    rebuilt = pd.DataFrame(index=out.index)
    for base, group_cols in base_names.items():
        if len(group_cols) == 1:
            rebuilt[base] = out[group_cols[0]]
        else:
            series = out[group_cols[0]].copy()
            for c in group_cols[1:]:
                series = series.combine_first(out[c])
            rebuilt[base] = series

    return rebuilt

def compute_ece(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    if len(y_true) == 0:
        return np.nan

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for lo, hi in zip(bins[:-1], bins[1:]):
        if hi == 1.0:
            mask = (y_prob >= lo) & (y_prob <= hi)
        else:
            mask = (y_prob >= lo) & (y_prob < hi)

        if not np.any(mask):
            continue

        ece += float(np.mean(mask)) * abs(float(np.mean(y_prob[mask])) - float(np.mean(y_true[mask])))

    return float(ece)

def specificity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return float(tn / (tn + fp)) if (tn + fp) > 0 else np.nan

def metric_bundle(y_true, y_prob, threshold=0.5):
    y_true = pd.to_numeric(pd.Series(y_true), errors="coerce")
    y_prob = pd.to_numeric(pd.Series(y_prob), errors="coerce")
    valid = y_true.notna() & y_prob.notna()
    y_true = y_true[valid].astype(int).to_numpy()
    y_prob = y_prob[valid].astype(float).to_numpy()

    if len(y_true) == 0:
        return {
            "n_units": 0,
            "n_positive": 0,
            "n_negative": 0,
            "prevalence": np.nan,
            "auroc": np.nan,
            "auprc": np.nan,
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "sensitivity": np.nan,
            "specificity": np.nan,
            "precision": np.nan,
            "f1": np.nan,
            "brier": np.nan,
            "ece_10bin": np.nan,
        }

    y_pred = (y_prob >= float(threshold)).astype(int)

    n_pos = int((y_true == 1).sum())
    n_neg = int((y_true == 0).sum())

    return {
        "n_units": int(len(y_true)),
        "n_positive": n_pos,
        "n_negative": n_neg,
        "prevalence": float(n_pos / len(y_true)),
        "auroc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "auprc": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)) if len(np.unique(y_true)) == 2 else np.nan,
        "sensitivity": float(recall_score(y_true, y_pred, zero_division=0)),
        "specificity": specificity_score(y_true, y_pred),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, y_prob)),
        "ece_10bin": compute_ece(y_true, y_prob, n_bins=10),
    }

def bootstrap_two_group_mean_difference(pos_values, neg_values, n_boot=5000, seed=411):
    pos = pd.to_numeric(pd.Series(pos_values), errors="coerce").dropna().to_numpy(dtype=float)
    neg = pd.to_numeric(pd.Series(neg_values), errors="coerce").dropna().to_numpy(dtype=float)

    if len(pos) == 0 or len(neg) == 0:
        return {
            "positive_mean": np.nan,
            "negative_mean": np.nan,
            "mean_difference_positive_minus_negative": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
            "n_positive_units": int(len(pos)),
            "n_negative_units": int(len(neg)),
        }

    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(int(n_boot)):
        pos_sample = rng.choice(pos, size=len(pos), replace=True)
        neg_sample = rng.choice(neg, size=len(neg), replace=True)
        diffs.append(float(np.mean(pos_sample) - np.mean(neg_sample)))

    lo, hi = np.percentile(diffs, [2.5, 97.5])

    return {
        "positive_mean": float(np.mean(pos)),
        "negative_mean": float(np.mean(neg)),
        "mean_difference_positive_minus_negative": float(np.mean(pos) - np.mean(neg)),
        "ci95_low": float(lo),
        "ci95_high": float(hi),
        "n_positive_units": int(len(pos)),
        "n_negative_units": int(len(neg)),
    }

def source_path_from_map(role, source_map):
    if source_map is None or source_map.empty or "role" not in source_map.columns:
        return None

    block = source_map[source_map["role"].astype(str).eq(str(role))].copy()
    if block.empty:
        return None

    if "selected" in block.columns:
        selected_block = block[block["selected"].astype(str).str.lower().isin(["true", "1", "yes", "y"])]
        if not selected_block.empty:
            block = selected_block

    path_col = "path" if "path" in block.columns else None
    if path_col is None:
        return None

    path = Path(str(block.iloc[0][path_col]))
    return path if path.exists() else None

def infer_quality_source_kind(path_like):
    path = str(path_like).lower()
    if not path or path == "none":
        return "unknown"
    if "hygd" in path or "physionet" in path:
        return "upstream_hygd_quality_manifest_or_precomputed_hygd_artifact"
    return "pipeline_detected_quality_artifact"

# -------------------------------------------------------------------------
# Load primary Notebook 14 artifacts
# -------------------------------------------------------------------------

if not HYGD_UNIFIED_BASE_CSV.exists():
    raise FileNotFoundError(
        f"Missing {HYGD_UNIFIED_BASE_CSV}. Run Notebook 14 through Cell 4 first."
    )

base = safe_read_csv(HYGD_UNIFIED_BASE_CSV)
base = coalesce_duplicate_columns(base)

if base.empty:
    raise RuntimeError("HYGD unified base manifest is empty.")

for col in ["image_id", "patient_id"]:
    if col in base.columns:
        base[col] = base[col].astype(str)

if "gon_binary" in base.columns:
    base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")
else:
    raise RuntimeError("HYGD unified base does not contain gon_binary.")

if "quality_score" not in base.columns:
    base["quality_score"] = np.nan
base["quality_score"] = pd.to_numeric(base["quality_score"], errors="coerce")

if "quality_tertile" not in base.columns:
    base["quality_tertile"] = "unknown"
base["quality_tertile"] = base["quality_tertile"].apply(normalize_quality_value)

if "strict_anatomical_proxy_valid" not in base.columns:
    base["strict_anatomical_proxy_valid"] = False
base["strict_anatomical_proxy_valid"] = base["strict_anatomical_proxy_valid"].apply(normalize_bool)

for col in [
    "area_cdr_proxy",
    "cup_to_disc_area_ratio",
    "od_area_512",
    "oc_area_512",
    "rim_area_512",
    "od_confidence_mean",
    "oc_confidence_mean",
]:
    if col not in base.columns:
        base[col] = np.nan
    base[col] = pd.to_numeric(base[col], errors="coerce")

cdr_col = "area_cdr_proxy" if base["area_cdr_proxy"].notna().any() else "cup_to_disc_area_ratio"

source_map = safe_read_csv(REVIEWER_SOURCE_MAP_CSV)

quality_summary = safe_read_csv(HYGD_QUALITY_AQPR_SUMMARY_CSV)
endpoint_availability = safe_read_csv(HYGD_QUALITY_ENDPOINT_AVAILABILITY_CSV)
proxy_qc_by_quality = safe_read_csv(HYGD_PROXY_QC_BY_QUALITY_CSV)

split_leakage = safe_read_csv(HYGD_SPLIT_PATIENT_LEAKAGE_CSV)
exact_duplicates = safe_read_csv(HYGD_EXACT_DUPLICATE_GROUPS_CSV)
near_duplicate_thresholds = safe_read_csv(HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV)
near_duplicate_pairs = safe_read_csv(HYGD_SUSPICIOUS_NEAR_DUPLICATE_PAIRS_CSV)
outside_od_manifest = safe_read_csv(OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV)
outside_od_summary = safe_read_csv(OUTSIDE_OD_CONSERVATIVE_SUMMARY_CSV)
smdg_source_metrics = safe_read_csv(SMDG_SOURCE_METRICS_COMPACT_CSV)
figure_candidates = safe_read_csv(FIGURE_MANUSCRIPT_CANDIDATES_CSV)

notebook13_calibration_handoff = safe_read_json(NOTEBOOK13_CALIBRATION_HANDOFF_JSON)
notebook15_claim_boundary = safe_read_csv(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV)
notebook15_core_endpoint_summary = safe_read_csv(NOTEBOOK15_CORE_ENDPOINT_SUMMARY_CSV)

# -------------------------------------------------------------------------
# 10.1 HYGD quality-score definition and tertile thresholds
# -------------------------------------------------------------------------

quality_source_path = source_path_from_map("hygd_quality_manifest", source_map)
quality_source_kind = infer_quality_source_kind(quality_source_path)

quality_source_df = safe_read_csv(quality_source_path) if quality_source_path else pd.DataFrame()

quality_manifest_has_score = False
quality_manifest_has_tertile = False

if not quality_source_df.empty:
    q_score_col = find_col(
        quality_source_df,
        exact_names=["quality_score", "image_quality_score"],
        contains_all=["quality", "score"],
    )
    q_tertile_col = find_col(
        quality_source_df,
        exact_names=["quality_tertile", "quality_bin", "quality_stratum"],
        contains_any=["quality", "tertile", "stratum", "bin"],
    )
    quality_manifest_has_score = q_score_col is not None
    quality_manifest_has_tertile = q_tertile_col is not None

valid_quality = base["quality_score"].dropna()

quality_definition_rows = [{
    "item": "quality_score_source_file",
    "value": str(quality_source_path) if quality_source_path is not None else "not_detected",
    "interpretation": "Source selected by Notebook 14 reviewer source map.",
}, {
    "item": "quality_score_source_kind",
    "value": quality_source_kind,
    "interpretation": (
        "The manuscript should describe the score as coming from the selected HYGD quality artifact. "
        "If this artifact was produced upstream by the local pipeline rather than released directly with HYGD, "
        "state that it is a precomputed quality score used only for stratification."
    ),
}, {
    "item": "quality_manifest_has_score_column",
    "value": str(bool(quality_manifest_has_score)),
    "interpretation": "Whether the selected upstream quality artifact contains a score-like column.",
}, {
    "item": "quality_manifest_has_tertile_column",
    "value": str(bool(quality_manifest_has_tertile)),
    "interpretation": "Whether the selected upstream quality artifact already contained a quality stratum.",
}, {
    "item": "quality_tertile_origin",
    "value": (
        "upstream_quality_manifest"
        if quality_manifest_has_tertile
        else "computed_in_notebook14_from_quality_score_rank_qcut"
        if valid_quality.shape[0] > 0
        else "unknown_no_quality_score_available"
    ),
    "interpretation": (
        "Tertiles are used only for reliability stratification, not for model training or threshold tuning."
    ),
}, {
    "item": "n_images_total",
    "value": str(int(base["image_id"].nunique())) if "image_id" in base.columns else str(int(len(base))),
    "interpretation": "HYGD image count in the unified AQPR manifest.",
}, {
    "item": "n_patients_total",
    "value": str(int(base["patient_id"].nunique())) if "patient_id" in base.columns else "NA",
    "interpretation": "HYGD patient count in the unified AQPR manifest.",
}, {
    "item": "quality_score_missing_images",
    "value": str(int(base["quality_score"].isna().sum())),
    "interpretation": "Images without usable quality score.",
}, {
    "item": "quality_score_min",
    "value": f"{float(valid_quality.min()):.6f}" if len(valid_quality) else "NA",
    "interpretation": "Minimum available quality score.",
}, {
    "item": "quality_score_max",
    "value": f"{float(valid_quality.max()):.6f}" if len(valid_quality) else "NA",
    "interpretation": "Maximum available quality score.",
}, {
    "item": "quality_score_mean",
    "value": f"{float(valid_quality.mean()):.6f}" if len(valid_quality) else "NA",
    "interpretation": "Mean available quality score.",
}, {
    "item": "quality_score_std",
    "value": f"{float(valid_quality.std(ddof=1)):.6f}" if len(valid_quality) > 1 else "NA",
    "interpretation": "Standard deviation of available quality score.",
}]

quality_definition = pd.DataFrame(quality_definition_rows)
save_csv_md(quality_definition, HYGD_QUALITY_SCORE_DEFINITION_CSV, HYGD_QUALITY_SCORE_DEFINITION_MD)

threshold_rows = []

for q in QUALITY_ORDER:
    block = base[base["quality_tertile"].eq(q)].copy()
    scores = block["quality_score"].dropna()

    threshold_rows.append({
        "quality_tertile": q,
        "n_images": int(block["image_id"].nunique()) if "image_id" in block.columns else int(len(block)),
        "n_patients": int(block["patient_id"].nunique()) if "patient_id" in block.columns else np.nan,
        "n_gon_positive_images": int((block["gon_binary"] == 1).sum()),
        "n_gon_negative_images": int((block["gon_binary"] == 0).sum()),
        "score_min": float(scores.min()) if len(scores) else np.nan,
        "score_q25": float(scores.quantile(0.25)) if len(scores) else np.nan,
        "score_median": float(scores.median()) if len(scores) else np.nan,
        "score_q75": float(scores.quantile(0.75)) if len(scores) else np.nan,
        "score_max": float(scores.max()) if len(scores) else np.nan,
        "score_mean": float(scores.mean()) if len(scores) else np.nan,
        "score_std": float(scores.std(ddof=1)) if len(scores) > 1 else np.nan,
        "strict_proxy_valid_images": int(block["strict_anatomical_proxy_valid"].sum()),
        "strict_proxy_valid_fraction": float(block["strict_anatomical_proxy_valid"].mean()) if len(block) else np.nan,
    })

quality_thresholds = pd.DataFrame(threshold_rows)
save_csv_md(quality_thresholds, HYGD_QUALITY_TERTILE_THRESHOLDS_CSV, HYGD_QUALITY_TERTILE_THRESHOLDS_MD)

# -------------------------------------------------------------------------
# 10.2 Proxy strict-valid and proxy-quality evidence
# -------------------------------------------------------------------------

strict_block = base[base["strict_anatomical_proxy_valid"]].copy()

proxy_evidence_rows = []

proxy_evidence_rows.extend([
    {
        "evidence_axis": "strict_validity",
        "metric": "n_total_images",
        "value": int(base["image_id"].nunique()) if "image_id" in base.columns else int(len(base)),
        "interpretation": "Total HYGD images in unified AQPR manifest.",
    },
    {
        "evidence_axis": "strict_validity",
        "metric": "n_strict_valid_proxy_images",
        "value": int(base["strict_anatomical_proxy_valid"].sum()),
        "interpretation": "Images eligible for strict OD/OC proxy anatomical endpoints.",
    },
    {
        "evidence_axis": "strict_validity",
        "metric": "n_proxy_failures",
        "value": int((~base["strict_anatomical_proxy_valid"]).sum()),
        "interpretation": "Images excluded from strict anatomy-mask endpoints because proxy validity criteria were not met.",
    },
    {
        "evidence_axis": "strict_validity",
        "metric": "strict_valid_fraction",
        "value": float(base["strict_anatomical_proxy_valid"].mean()),
        "interpretation": "Fraction of HYGD images passing strict OD/OC proxy validity.",
    },
    {
        "evidence_axis": "claim_boundary",
        "metric": "proxy_claim_boundary",
        "value": "proxy_audit_region_not_manual_hygd_ground_truth",
        "interpretation": (
            "OD/OC masks are externally developed segmentation proxies and support anatomical audit-region definition; "
            "they are not manual HYGD OD/OC ground truth."
        ),
    },
])

for col, label in [
    (cdr_col, "cdr_proxy"),
    ("od_area_512", "od_area_proxy"),
    ("oc_area_512", "oc_area_proxy"),
    ("rim_area_512", "rim_area_proxy"),
    ("od_confidence_mean", "od_confidence"),
    ("oc_confidence_mean", "oc_confidence"),
]:
    values = pd.to_numeric(strict_block[col], errors="coerce").dropna() if col in strict_block.columns else pd.Series(dtype=float)
    proxy_evidence_rows.extend([
        {
            "evidence_axis": "proxy_distribution",
            "metric": f"{label}_n",
            "value": int(len(values)),
            "interpretation": f"Number of strict-valid images with {label}.",
        },
        {
            "evidence_axis": "proxy_distribution",
            "metric": f"{label}_mean",
            "value": float(values.mean()) if len(values) else np.nan,
            "interpretation": f"Mean {label} among strict-valid images.",
        },
        {
            "evidence_axis": "proxy_distribution",
            "metric": f"{label}_std",
            "value": float(values.std(ddof=1)) if len(values) > 1 else np.nan,
            "interpretation": f"Standard deviation of {label} among strict-valid images.",
        },
        {
            "evidence_axis": "proxy_distribution",
            "metric": f"{label}_q05",
            "value": float(values.quantile(0.05)) if len(values) else np.nan,
            "interpretation": f"5th percentile of {label} among strict-valid images.",
        },
        {
            "evidence_axis": "proxy_distribution",
            "metric": f"{label}_q95",
            "value": float(values.quantile(0.95)) if len(values) else np.nan,
            "interpretation": f"95th percentile of {label} among strict-valid images.",
        },
    ])

# Patient-level CDR proxy difference: GON+ minus GON-.
patient_cdr = pd.DataFrame()

if "patient_id" in strict_block.columns and cdr_col in strict_block.columns:
    patient_cdr = (
        strict_block
        .dropna(subset=["patient_id", "gon_binary", cdr_col])
        .groupby("patient_id", dropna=False)
        .agg(
            gon_binary=("gon_binary", "max"),
            cdr_proxy=(cdr_col, "mean"),
            n_images=("image_id", "nunique") if "image_id" in strict_block.columns else (cdr_col, "size"),
        )
        .reset_index()
    )

cdr_pos = patient_cdr.loc[patient_cdr["gon_binary"].eq(1), "cdr_proxy"] if not patient_cdr.empty else []
cdr_neg = patient_cdr.loc[patient_cdr["gon_binary"].eq(0), "cdr_proxy"] if not patient_cdr.empty else []

cdr_diff = bootstrap_two_group_mean_difference(
    pos_values=cdr_pos,
    neg_values=cdr_neg,
    n_boot=BOOTSTRAP_REPEATS,
    seed=BOOTSTRAP_SEED,
)

cdr_diff_row = {
    "comparison": "patient_level_cdr_proxy_gon_positive_minus_gon_negative",
    "proxy_metric": cdr_col,
    **cdr_diff,
    "bootstrap_repeats": int(BOOTSTRAP_REPEATS),
    "bootstrap_seed": int(BOOTSTRAP_SEED),
    "claim_boundary": "face_validity_for_proxy_region_not_manual_hygd_segmentation_validation",
}

cdr_diff_table = pd.DataFrame([cdr_diff_row])
save_csv_md(cdr_diff_table, HYGD_PROXY_CDR_PATIENT_DIFF_CSV, HYGD_PROXY_CDR_PATIENT_DIFF_MD)

proxy_evidence_rows.append({
    "evidence_axis": "disease_association_face_validity",
    "metric": "patient_level_cdr_proxy_gon_positive_minus_gon_negative",
    "value": cdr_diff["mean_difference_positive_minus_negative"],
    "interpretation": (
        f"GON+ patients showed higher patient-level {cdr_col} than GON- patients "
        f"with bootstrap 95% CI [{cdr_diff['ci95_low']}, {cdr_diff['ci95_high']}]."
    ),
})

proxy_quality_evidence = pd.DataFrame(proxy_evidence_rows)
save_csv_md(proxy_quality_evidence, HYGD_PROXY_QUALITY_EVIDENCE_CSV, HYGD_PROXY_QUALITY_EVIDENCE_MD)

# Proxy failure audit: capture possible failure/status columns if present.
failure_cols = [
    c for c in base.columns
    if any(tok in c.lower() for tok in ["fail", "failure", "reason", "status", "valid", "include_strict"])
]

failure_audit = base.loc[
    ~base["strict_anatomical_proxy_valid"],
    [c for c in ["image_id", "patient_id", "gon_binary", "quality_tertile", "quality_score"] + failure_cols if c in base.columns]
].copy()

if failure_audit.empty:
    failure_audit = pd.DataFrame([{
        "status": "no_proxy_failures_or_failure_rows_not_available",
        "n_failures": int((~base["strict_anatomical_proxy_valid"]).sum()),
    }])

save_csv_md(failure_audit, HYGD_PROXY_FAILURE_AUDIT_CSV, HYGD_PROXY_FAILURE_AUDIT_MD)

# -------------------------------------------------------------------------
# 10.3 Patient aggregation robustness
# -------------------------------------------------------------------------

def standardize_prediction_table(df):
    if df is None or df.empty:
        return pd.DataFrame()

    out = coalesce_duplicate_columns(df.copy())

    image_col = find_col(out, ["image_id", "sample_id", "fundus_id", "id"], contains_any=["image", "sample"])
    patient_col = find_col(out, ["patient_id", "subject_id", "patient"], contains_any=["patient", "subject"])
    label_col = find_col(out, ["gon_binary", "y_true", "label", "target", "glaucoma_label"], contains_any=["label", "target", "gon", "glaucoma", "true"])
    model_col = find_col(out, ["model_name", "architecture", "model", "classifier"], contains_any=["model", "architecture", "classifier"])
    mode_col = find_col(out, ["ablation_mode", "perturbation_mode", "condition", "input_mode", "variant"], contains_any=["mode", "condition", "variant", "ablation", "perturbation"])

    # Prefer raw/original model probability if available; fall back to calibrated probability.
    prob_col = find_col(
        out,
        [
            "probability",
            "raw_probability",
            "original_probability",
            "y_prob",
            "prob_gon",
            "p_glaucoma",
            "positive_probability",
            "prediction_probability",
            "temperature_scaled_probability",
            "calibrated_probability",
        ],
        contains_any=["prob", "p_glaucoma"],
    )

    threshold_col = find_col(
        out,
        ["validation_selected_threshold", "selected_threshold", "threshold", "operating_threshold"],
        contains_any=["threshold"],
    )

    rename = {}
    for src, dst in [
        (image_col, "image_id"),
        (patient_col, "patient_id"),
        (label_col, "gon_binary"),
        (model_col, "model_name"),
        (mode_col, "ablation_mode"),
        (prob_col, "probability_for_metrics"),
        (threshold_col, "threshold"),
    ]:
        if src is not None and src != dst:
            rename[src] = dst

    out = out.rename(columns=rename)
    out = coalesce_duplicate_columns(out)

    if "image_id" in out.columns:
        out["image_id"] = out["image_id"].astype(str)
    if "patient_id" in out.columns:
        out["patient_id"] = out["patient_id"].astype(str)
    if "gon_binary" in out.columns:
        out["gon_binary"] = pd.to_numeric(out["gon_binary"], errors="coerce")
    if "model_name" in out.columns:
        out["model_name"] = out["model_name"].apply(canonical_model_name)
    else:
        out["model_name"] = PRIMARY_MODEL_NAME
    if "ablation_mode" in out.columns:
        out["ablation_mode"] = out["ablation_mode"].apply(canonical_ablation_mode)
    else:
        out["ablation_mode"] = "original_full_image"
    if "probability_for_metrics" in out.columns:
        out["probability_for_metrics"] = pd.to_numeric(out["probability_for_metrics"], errors="coerce")
    if "threshold" in out.columns:
        out["threshold"] = pd.to_numeric(out["threshold"], errors="coerce")
    else:
        out["threshold"] = 0.5

    return out

prediction_roles = [
    "hygd_ablation_predictions",
    "hygd_prediction_source",
    "hygd_predictions",
    "hygd_model_predictions",
]

pred_df = pd.DataFrame()
pred_path = None
pred_role = None

for role in prediction_roles:
    path = source_path_from_map(role, source_map)
    if path is None:
        continue
    candidate = standardize_prediction_table(safe_read_csv(path))
    if (
        not candidate.empty
        and "image_id" in candidate.columns
        and "probability_for_metrics" in candidate.columns
    ):
        pred_df = candidate.copy()
        pred_path = path
        pred_role = role
        break

aggregation_rows = []

if pred_df.empty:
    aggregation_rows.append({
        "analysis_status": "MISSING_UPSTREAM_PREDICTION_ARTIFACT",
        "message": "No compatible HYGD prediction table was detected from Notebook 14 source map.",
    })
    aggregation_robustness_long = pd.DataFrame(aggregation_rows)
    aggregation_robustness_compact = aggregation_robustness_long.copy()

else:
    merge_cols = [
        c for c in [
            "image_id",
            "patient_id",
            "gon_binary",
            "quality_score",
            "quality_tertile",
        ] if c in base.columns
    ]

    pred = pred_df.merge(
        base[merge_cols],
        on="image_id",
        how="left",
        suffixes=("", "_base"),
    )
    pred = coalesce_duplicate_columns(pred)

    for col in ["patient_id", "gon_binary", "quality_score", "quality_tertile"]:
        base_col = f"{col}_base"
        if base_col in pred.columns:
            if col not in pred.columns:
                pred[col] = pred[base_col]
            else:
                pred[col] = pred[col].combine_first(pred[base_col])

    pred["quality_tertile"] = pred["quality_tertile"].apply(normalize_quality_value)
    pred["probability_for_metrics"] = pd.to_numeric(pred["probability_for_metrics"], errors="coerce")
    pred["gon_binary"] = pd.to_numeric(pred["gon_binary"], errors="coerce")
    pred["quality_score"] = pd.to_numeric(pred["quality_score"], errors="coerce")
    pred["threshold"] = pd.to_numeric(pred["threshold"], errors="coerce").fillna(0.5)

    # Prefer original/full image rows.
    original_mask = pred["ablation_mode"].astype(str).eq("original_full_image")
    if original_mask.any():
        pred_eval = pred[original_mask].copy()
        original_selection_policy = "ablation_mode_original_full_image"
    else:
        pred_eval = pred.copy()
        original_selection_policy = "no_original_full_image_mode_found_used_all_prediction_rows"

    pred_eval = pred_eval.dropna(subset=["patient_id", "gon_binary", "probability_for_metrics"]).copy()

    # If multiple rows per image/model remain, average them to prevent duplicate modes from dominating.
    image_level = (
        pred_eval
        .groupby(["model_name", "image_id"], dropna=False)
        .agg(
            patient_id=("patient_id", "first"),
            gon_binary=("gon_binary", "max"),
            probability_for_metrics=("probability_for_metrics", "mean"),
            threshold=("threshold", "median"),
            quality_score=("quality_score", "mean"),
            quality_tertile=("quality_tertile", "first"),
        )
        .reset_index()
    )

    def patient_quality_weighted_mean(group):
        probs = pd.to_numeric(group["probability_for_metrics"], errors="coerce").to_numpy(dtype=float)
        qs = pd.to_numeric(group["quality_score"], errors="coerce").to_numpy(dtype=float)

        valid = np.isfinite(probs)
        probs = probs[valid]
        qs = qs[valid]

        if len(probs) == 0:
            return np.nan
        if len(probs) == 1:
            return float(probs[0])

        if np.isfinite(qs).sum() == 0:
            return float(np.mean(probs))

        # Higher score is treated as higher image quality. Shift to positive weights.
        q_valid = np.where(np.isfinite(qs), qs, np.nanmedian(qs[np.isfinite(qs)]))
        q_min = np.nanmin(q_valid)
        weights = q_valid - q_min + 1e-6

        if np.sum(weights) <= 0 or not np.isfinite(np.sum(weights)):
            return float(np.mean(probs))

        return float(np.average(probs, weights=weights))

    aggregation_methods = {
        "mean": lambda g: float(g["probability_for_metrics"].mean()),
        "median": lambda g: float(g["probability_for_metrics"].median()),
        "max": lambda g: float(g["probability_for_metrics"].max()),
        "quality_weighted_mean": patient_quality_weighted_mean,
    }

    for model_name, model_block in image_level.groupby("model_name", dropna=False):
        model_name = canonical_model_name(model_name)

        if model_name != PRIMARY_MODEL_NAME and model_name not in ["EfficientNetB0", "ConvNeXtTiny", "DenseNet121", "SwinTiny"]:
            continue

        threshold_values = pd.to_numeric(model_block["threshold"], errors="coerce").dropna()
        threshold = float(threshold_values.median()) if len(threshold_values) else 0.5

        strata = [("overall", "overall", model_block)]
        for q in QUALITY_ORDER:
            strata.append((f"quality_{q}", q, model_block[model_block["quality_tertile"].eq(q)].copy()))

        for stratum_type, stratum_label, stratum_block in strata:
            if stratum_block.empty:
                continue

            for method_name, method_fn in aggregation_methods.items():
                patient_rows = []
                for patient_id, group in stratum_block.groupby("patient_id", dropna=False):
                    if group.empty:
                        continue

                    patient_rows.append({
                        "model_name": model_name,
                        "patient_id": patient_id,
                        "gon_binary": int(pd.to_numeric(group["gon_binary"], errors="coerce").max()),
                        "patient_probability": method_fn(group),
                        "n_images": int(group["image_id"].nunique()),
                        "mean_quality_score": float(pd.to_numeric(group["quality_score"], errors="coerce").mean()),
                    })

                patient_df = pd.DataFrame(patient_rows)
                if patient_df.empty:
                    continue

                metrics = metric_bundle(
                    patient_df["gon_binary"],
                    patient_df["patient_probability"],
                    threshold=threshold,
                )

                aggregation_rows.append({
                    "analysis_status": "PASS",
                    "prediction_source_role": pred_role,
                    "prediction_source_path": str(pred_path),
                    "original_row_selection_policy": original_selection_policy,
                    "model_name": model_name,
                    "stratum_type": stratum_type,
                    "stratum_label": stratum_label,
                    "aggregation_method": method_name,
                    "threshold": threshold,
                    "n_patients": int(patient_df["patient_id"].nunique()),
                    "n_images_contributing": int(patient_df["n_images"].sum()),
                    "median_images_per_patient": float(patient_df["n_images"].median()),
                    **metrics,
                })

    aggregation_robustness_long = pd.DataFrame(aggregation_rows)

    if not aggregation_robustness_long.empty and "analysis_status" in aggregation_robustness_long.columns:
        compact_block = aggregation_robustness_long[
            aggregation_robustness_long["analysis_status"].eq("PASS")
            & aggregation_robustness_long["stratum_type"].eq("overall")
            & aggregation_robustness_long["model_name"].eq(PRIMARY_MODEL_NAME)
        ].copy()

        aggregation_robustness_compact = compact_block[
            [
                "model_name",
                "aggregation_method",
                "n_patients",
                "n_images_contributing",
                "median_images_per_patient",
                "auroc",
                "balanced_accuracy",
                "brier",
                "ece_10bin",
            ]
        ].sort_values("aggregation_method").reset_index(drop=True)
    else:
        aggregation_robustness_compact = aggregation_robustness_long.copy()

save_csv_md(
    aggregation_robustness_long,
    HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_CSV,
    HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_MD,
)

save_csv_md(
    aggregation_robustness_compact,
    HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV,
    HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_MD,
)

# -------------------------------------------------------------------------
# 10.4 Reviewer-response evidence matrix and manuscript evidence summary
# -------------------------------------------------------------------------

def exists_nonempty(path_like):
    path = Path(path_like)
    return path.exists() and path.stat().st_size > 0

reviewer_rows = [
    {
        "reviewer_concern": "HYGD OD/OC proxy masks are not manual target-domain ground truth.",
        "evidence_generated": "Cross-dataset segmenter evidence is reported upstream; Notebook 14 consolidates HYGD strict-valid QC, proxy failure count, CDR distribution, and GON+/GON- CDR face validity.",
        "primary_outputs": f"{HYGD_PROXY_QUALITY_EVIDENCE_CSV}; {HYGD_PROXY_CDR_PATIENT_DIFF_CSV}; {HYGD_PROXY_FAILURE_AUDIT_CSV}",
        "status": "addressed_with_claim_boundary",
        "manuscript_action": "State that HYGD OD/OC masks are proxy audit regions, not manual HYGD ground truth; report 745/747 strict-valid and patient-level CDR separation.",
    },
    {
        "reviewer_concern": "Image-quality stratification lacks detail.",
        "evidence_generated": "Quality score source, origin, missingness, tertile thresholds, image/patient counts, and label counts were consolidated.",
        "primary_outputs": f"{HYGD_QUALITY_SCORE_DEFINITION_CSV}; {HYGD_QUALITY_TERTILE_THRESHOLDS_CSV}",
        "status": "addressed",
        "manuscript_action": "Add one Methods sentence defining quality score origin and tertile construction; include thresholds in supplement.",
    },
    {
        "reviewer_concern": "Patient-level aggregation may depend on unweighted mean pooling.",
        "evidence_generated": "Patient-level aggregation robustness was computed for mean, median, max, and quality-weighted mean.",
        "primary_outputs": f"{HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_CSV}; {HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV}",
        "status": "addressed",
        "manuscript_action": "Report mean aggregation as primary; cite robustness under median/max/quality-weighted pooling in supplement.",
    },
    {
        "reviewer_concern": "Core saliency/deletion/perturbation endpoints were previously based on a small subset.",
        "evidence_generated": "Handled by expanded Notebook 15 all-HYGD audit with patient-level CIs, insertion/deletion AUC, and perturbation method-stability.",
        "primary_outputs": str(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV),
        "status": "addressed_by_notebook15" if exists_nonempty(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV) else "pending_notebook15",
        "manuscript_action": "Update Results claim-boundary table using Notebook 15 expanded all-HYGD outputs.",
    },
    {
        "reviewer_concern": "Calibration was pooled; per-source calibration was requested.",
        "evidence_generated": "Handled by updated Notebook 13 SMDG per-source calibration and reliability diagrams.",
        "primary_outputs": str(NOTEBOOK13_CALIBRATION_HANDOFF_JSON),
        "status": "addressed_by_notebook13" if exists_nonempty(NOTEBOOK13_CALIBRATION_HANDOFF_JSON) else "pending_notebook13",
        "manuscript_action": "Add SMDG per-source ECE/Brier raw vs temperature-scaled table or supplement figure from Notebook 13.",
    },
    {
        "reviewer_concern": "Dataset forensics and outside-OD shortcuts require conservative interpretation.",
        "evidence_generated": "Notebook 14 generated patient split leakage, exact/near duplicate, and outside-OD conservative control artifacts.",
        "primary_outputs": f"{HYGD_SPLIT_PATIENT_LEAKAGE_CSV}; {HYGD_EXACT_DUPLICATE_GROUPS_CSV}; {HYGD_NEAR_DUPLICATE_THRESHOLD_SUMMARY_CSV}; {OUTSIDE_OD_CONSERVATIVE_MANIFEST_CSV}",
        "status": "addressed_with_conservative_interpretation",
        "manuscript_action": "Keep outside-OD findings as shortcut/dataset-artifact warnings, not causal non-anatomical claims.",
    },
]

reviewer_matrix = pd.DataFrame(reviewer_rows)
save_csv_md(reviewer_matrix, NOTEBOOK14_REVIEWER_RESPONSE_MATRIX_CSV)

evidence_summary_rows = [
    {
        "evidence_block": "HYGD quality definition",
        "main_result": (
            "Quality tertiles are explicitly defined from the selected HYGD quality artifact; thresholds, counts, missingness, and label distribution are now tabulated."
        ),
        "main_table_or_figure": str(HYGD_QUALITY_SCORE_DEFINITION_CSV),
        "recommended_location": "Methods + Supplement",
    },
    {
        "evidence_block": "HYGD OD/OC proxy quality",
        "main_result": (
            f"Strict-valid OD/OC proxy images: {int(base['strict_anatomical_proxy_valid'].sum())}/{int(len(base))}; "
            f"patient-level CDR proxy GON+ minus GON- = "
            f"{cdr_diff['mean_difference_positive_minus_negative']:.4f} "
            f"[{cdr_diff['ci95_low']:.4f}, {cdr_diff['ci95_high']:.4f}]."
        ),
        "main_table_or_figure": str(HYGD_PROXY_QUALITY_EVIDENCE_CSV),
        "recommended_location": "Main text brief result + Supplement details",
    },
    {
        "evidence_block": "Patient aggregation robustness",
        "main_result": (
            "Patient-level endpoints were recalculated using mean, median, max, and quality-weighted mean aggregation."
        ),
        "main_table_or_figure": str(HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV),
        "recommended_location": "Supplement; one sentence in Results",
    },
    {
        "evidence_block": "Expanded core AQPR endpoints",
        "main_result": (
            "Notebook 15 supplies all-HYGD saliency sanity, deletion/insertion AUC, perturbation method-stability, and final claim boundaries."
            if exists_nonempty(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV)
            else "Pending Notebook 15 expanded claim-boundary output."
        ),
        "main_table_or_figure": str(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV),
        "recommended_location": "Main Results claim-boundary table/figure",
    },
    {
        "evidence_block": "SMDG per-source calibration",
        "main_result": (
            "Notebook 13 supplies per-source raw/scaled Brier/ECE and reliability diagrams."
            if exists_nonempty(NOTEBOOK13_CALIBRATION_HANDOFF_JSON)
            else "Pending Notebook 13 calibration handoff."
        ),
        "main_table_or_figure": str(NOTEBOOK13_CALIBRATION_HANDOFF_JSON),
        "recommended_location": "Main or Supplement calibration paragraph/table",
    },
]

evidence_summary = pd.DataFrame(evidence_summary_rows)
save_csv_md(evidence_summary, NOTEBOOK14_MANUSCRIPT_EVIDENCE_SUMMARY_CSV)

main_text_candidate_rows = [
    {
        "candidate": "quality_definition_sentence",
        "text": (
            "HYGD quality stratification used the selected HYGD quality-score artifact; images were divided into low, medium, and high tertiles, with thresholds and counts reported in the supplement. "
            "Quality was used only for reliability stratification, not for classifier training, threshold selection, or calibration fitting."
        ),
    },
    {
        "candidate": "proxy_claim_boundary_sentence",
        "text": (
            f"The OD/OC proxy was strict-valid in {int(base['strict_anatomical_proxy_valid'].sum())}/{int(len(base))} HYGD images and produced higher patient-level {cdr_col} in GON+ than GON- patients "
            f"(mean difference {cdr_diff['mean_difference_positive_minus_negative']:.3f}; 95% CI {cdr_diff['ci95_low']:.3f}–{cdr_diff['ci95_high']:.3f}), "
            "supporting its use as an audit region while not replacing manual HYGD segmentation ground truth."
        ),
    },
    {
        "candidate": "patient_aggregation_sentence",
        "text": (
            "The primary patient-level endpoint used the unweighted mean of image-level probabilities to avoid giving greater influence to patients with more images; robustness checks using median, max, and quality-weighted mean aggregation are provided in the supplement."
        ),
    },
]

main_text_candidate_table = pd.DataFrame(main_text_candidate_rows)
save_csv_md(main_text_candidate_table, NOTEBOOK14_MAIN_TEXT_CANDIDATE_TABLE_CSV)

supplementary_candidate_rows = [
    {
        "supplementary_item": "Quality-score definition and tertile thresholds",
        "file": str(HYGD_QUALITY_SCORE_DEFINITION_CSV),
        "purpose": "Responds to reviewer request for quality-score origin and thresholding method.",
    },
    {
        "supplementary_item": "Proxy strict-valid and CDR face-validity summary",
        "file": str(HYGD_PROXY_QUALITY_EVIDENCE_CSV),
        "purpose": "Bounds OD/OC proxy interpretation without claiming manual HYGD mask validation.",
    },
    {
        "supplementary_item": "Patient aggregation robustness",
        "file": str(HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_CSV),
        "purpose": "Tests whether patient-level endpoints depend on mean pooling.",
    },
    {
        "supplementary_item": "Expanded core AQPR all-HYGD claim boundaries",
        "file": str(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV),
        "purpose": "Addresses reviewer concern about the previous 60-image subset.",
    },
    {
        "supplementary_item": "SMDG per-source calibration",
        "file": str(NOTEBOOK13_CALIBRATION_HANDOFF_JSON),
        "purpose": "Addresses reviewer request for per-source Brier/ECE and reliability diagrams.",
    },
]

supplementary_candidate_table = pd.DataFrame(supplementary_candidate_rows)
save_csv_md(supplementary_candidate_table, NOTEBOOK14_SUPPLEMENTARY_CANDIDATE_TABLE_CSV)

to_notebook16_rows = [
    {
        "priority": 1,
        "action": "Integrate Notebook 15 expanded all-HYGD claim-boundary table into main Results.",
        "source": str(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV),
        "status": "ready" if exists_nonempty(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV) else "pending",
    },
    {
        "priority": 2,
        "action": "Integrate Notebook 13 SMDG per-source calibration table/figures.",
        "source": str(NOTEBOOK13_CALIBRATION_HANDOFF_JSON),
        "status": "ready" if exists_nonempty(NOTEBOOK13_CALIBRATION_HANDOFF_JSON) else "pending",
    },
    {
        "priority": 3,
        "action": "Add HYGD quality-score definition and thresholds to supplement.",
        "source": str(HYGD_QUALITY_SCORE_DEFINITION_CSV),
        "status": "ready",
    },
    {
        "priority": 4,
        "action": "Add proxy strict-valid/CDR evidence and claim boundary.",
        "source": str(HYGD_PROXY_QUALITY_EVIDENCE_CSV),
        "status": "ready",
    },
    {
        "priority": 5,
        "action": "Add patient aggregation robustness to supplement.",
        "source": str(HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV),
        "status": "ready" if not aggregation_robustness_compact.empty else "missing_prediction_artifact",
    },
]

to_notebook16_actions = pd.DataFrame(to_notebook16_rows)
save_csv_md(to_notebook16_actions, NOTEBOOK14_TO_NOTEBOOK16_ACTION_ITEMS_CSV)

# -------------------------------------------------------------------------
# Handoff JSON/TXT
# -------------------------------------------------------------------------

handoff = {
    "notebook": "14_hygd_smdg_domain_shift_audit_reviewer_ready",
    "cell": "Cell 10",
    "status": "PASS",
    "purpose": (
        "Expanded Notebook 14 reviewer handoff for quality definition, OD/OC proxy-QC, "
        "patient aggregation robustness, and Notebook 16 manuscript integration."
    ),
    "hygd_manifest": {
        "n_images": int(len(base)),
        "n_unique_images": int(base["image_id"].nunique()) if "image_id" in base.columns else int(len(base)),
        "n_patients": int(base["patient_id"].nunique()) if "patient_id" in base.columns else None,
        "n_gon_positive_images": int((base["gon_binary"] == 1).sum()),
        "n_gon_negative_images": int((base["gon_binary"] == 0).sum()),
    },
    "quality_definition": {
        "source_path": str(quality_source_path) if quality_source_path is not None else None,
        "source_kind": quality_source_kind,
        "manifest_has_score_column": bool(quality_manifest_has_score),
        "manifest_has_tertile_column": bool(quality_manifest_has_tertile),
        "tertile_origin": (
            "upstream_quality_manifest"
            if quality_manifest_has_tertile
            else "computed_in_notebook14_from_quality_score_rank_qcut"
            if len(valid_quality) > 0
            else "unknown_no_quality_score_available"
        ),
        "n_missing_quality_scores": int(base["quality_score"].isna().sum()),
        "quality_score_min": float(valid_quality.min()) if len(valid_quality) else None,
        "quality_score_max": float(valid_quality.max()) if len(valid_quality) else None,
        "quality_score_mean": float(valid_quality.mean()) if len(valid_quality) else None,
        "quality_score_std": float(valid_quality.std(ddof=1)) if len(valid_quality) > 1 else None,
    },
    "proxy_quality": {
        "n_strict_valid": int(base["strict_anatomical_proxy_valid"].sum()),
        "n_proxy_failures": int((~base["strict_anatomical_proxy_valid"]).sum()),
        "strict_valid_fraction": float(base["strict_anatomical_proxy_valid"].mean()),
        "cdr_proxy_column": cdr_col,
        "patient_cdr_gon_positive_minus_gon_negative": cdr_diff,
        "claim_boundary": "OD/OC masks are proxy audit regions, not manual HYGD OD/OC ground truth.",
    },
    "patient_aggregation_robustness": {
        "prediction_source_role": pred_role,
        "prediction_source_path": str(pred_path) if pred_path is not None else None,
        "status": "PASS" if not pred_df.empty else "MISSING_UPSTREAM_PREDICTION_ARTIFACT",
        "aggregation_methods": ["mean", "median", "max", "quality_weighted_mean"],
        "primary_aggregation": "unweighted_mean",
        "compact_rows": int(len(aggregation_robustness_compact)),
    },
    "notebook13_calibration_handoff_available": bool(exists_nonempty(NOTEBOOK13_CALIBRATION_HANDOFF_JSON)),
    "notebook15_expanded_claim_boundary_available": bool(exists_nonempty(NOTEBOOK15_EXPANDED_CLAIM_BOUNDARY_CSV)),
    "output_tables": {
        "quality_score_definition": str(HYGD_QUALITY_SCORE_DEFINITION_CSV),
        "quality_tertile_thresholds": str(HYGD_QUALITY_TERTILE_THRESHOLDS_CSV),
        "proxy_quality_evidence": str(HYGD_PROXY_QUALITY_EVIDENCE_CSV),
        "proxy_cdr_patient_difference": str(HYGD_PROXY_CDR_PATIENT_DIFF_CSV),
        "proxy_failure_audit": str(HYGD_PROXY_FAILURE_AUDIT_CSV),
        "patient_aggregation_robustness_long": str(HYGD_PATIENT_AGGREGATION_ROBUSTNESS_LONG_CSV),
        "patient_aggregation_robustness_compact": str(HYGD_PATIENT_AGGREGATION_ROBUSTNESS_COMPACT_CSV),
        "reviewer_response_matrix": str(NOTEBOOK14_REVIEWER_RESPONSE_MATRIX_CSV),
        "manuscript_evidence_summary": str(NOTEBOOK14_MANUSCRIPT_EVIDENCE_SUMMARY_CSV),
        "main_text_candidate_table": str(NOTEBOOK14_MAIN_TEXT_CANDIDATE_TABLE_CSV),
        "supplementary_candidate_table": str(NOTEBOOK14_SUPPLEMENTARY_CANDIDATE_TABLE_CSV),
        "notebook16_action_items": str(NOTEBOOK14_TO_NOTEBOOK16_ACTION_ITEMS_CSV),
    },
    "integration_priority_for_v32": [
        "Use Notebook 15 expanded all-HYGD claim boundaries in Results.",
        "Use Notebook 13 SMDG per-source calibration in Results/Supplement.",
        "Use Notebook 14 quality/proxy/patient aggregation outputs in Methods/Supplement.",
    ],
}

with open(NOTEBOOK14_FINAL_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

with open(CELL10_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(handoff, f, indent=2)

with open(NOTEBOOK14_FINAL_HANDOFF_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 14 final handoff for Notebook 16 / manuscript v3.2\n")
    f.write("=" * 78 + "\n\n")
    f.write(f"Status: {handoff['status']}\n")
    f.write(f"HYGD images: {handoff['hygd_manifest']['n_images']}\n")
    f.write(f"HYGD patients: {handoff['hygd_manifest']['n_patients']}\n")
    f.write(f"Strict-valid proxy: {handoff['proxy_quality']['n_strict_valid']} / {handoff['hygd_manifest']['n_images']}\n")
    f.write(
        "Patient-level CDR proxy GON+ minus GON-: "
        f"{cdr_diff['mean_difference_positive_minus_negative']:.6f} "
        f"[{cdr_diff['ci95_low']:.6f}, {cdr_diff['ci95_high']:.6f}]\n\n"
    )
    f.write("Quality tertile origin:\n")
    f.write(f"  {handoff['quality_definition']['tertile_origin']}\n\n")
    f.write("Notebook 16 priorities:\n")
    for item in handoff["integration_priority_for_v32"]:
        f.write(f"  - {item}\n")

# -------------------------------------------------------------------------
# Console report
# -------------------------------------------------------------------------

print("\nHYGD quality-score definition")
display(quality_definition)

print("\nHYGD quality-tertile thresholds")
display(quality_thresholds)

print("\nHYGD proxy-quality evidence summary")
display(proxy_quality_evidence)

print("\nPatient-level CDR proxy difference")
display(cdr_diff_table)

print("\nPatient aggregation robustness compact")
display(aggregation_robustness_compact)

print("\nReviewer-response evidence matrix")
display(reviewer_matrix)

print("\nNotebook 16 action items")
display(to_notebook16_actions)

print("\nCell 10 completed.")
print(json.dumps(handoff, indent=2))

In [ ]:

# Cell 11: v3.6 reviewer extension — proxy morphometric baseline versus end-to-end HYGD models
#
# Reviewer target:
# - Contextualize end-to-end HYGD classifiers against segmentation-first / morphology-only evidence.
# - Use proxy CDR and simple proxy morphometrics as diagnostic baselines.
# - Avoid tuning or model selection on HYGD. Any shallow model below is a fixed, patient-grouped
#   cross-validated contextual baseline and is not used to select AQPR components.
#
# Outputs:
# - notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv
# - notebook14_v36_proxy_morphometric_oof_predictions.csv
# - notebook14_v36_hygd_end_to_end_prediction_audit.csv

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

try:
    from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, brier_score_loss
    from sklearn.model_selection import GroupKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print(f"sklearn unavailable warning: {exc}")

required_vars = ["TABLE_DIR", "RESULTS_14_DIR", "HYGD_UNIFIED_BASE_CSV", "REVIEWER_SOURCE_MAP_CSV"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

TABLE_DIR = Path(TABLE_DIR)
RESULTS_14_DIR = Path(RESULTS_14_DIR)
HANDOFF_DIR = RESULTS_14_DIR / "handoff"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

V36_CDR_BASELINE_CSV = TABLE_DIR / "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv"
V36_CDR_BASELINE_MD = TABLE_DIR / "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.md"
V36_CDR_OOF_CSV = TABLE_DIR / "notebook14_v36_proxy_morphometric_oof_predictions.csv"
V36_E2E_AUDIT_CSV = TABLE_DIR / "notebook14_v36_hygd_end_to_end_prediction_audit.csv"
V36_CELL11_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_v36_cell11_proxy_morphometric_baseline_summary.json"


def safe_read_csv(path_like):
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        print(f"Could not read CSV {path}: {exc}")
        return pd.DataFrame()


def save_csv_md(df, csv_path, md_path=None, max_rows=80):
    df.to_csv(csv_path, index=False)
    if md_path is not None:
        try:
            df.head(max_rows).to_markdown(md_path, index=False)
        except Exception:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(df.head(max_rows).to_string(index=False))
    return str(csv_path)


def find_col(df, exact_names=None, contains_any=None, contains_all=None, exclude_any=None):
    if df is None or df.empty:
        return None
    cols = list(df.columns)
    lower_map = {str(c).lower(): c for c in cols}
    if exact_names:
        for name in exact_names:
            if str(name).lower() in lower_map:
                return lower_map[str(name).lower()]
    for c in cols:
        lc = str(c).lower()
        if exclude_any and any(tok.lower() in lc for tok in exclude_any):
            continue
        if contains_all and not all(tok.lower() in lc for tok in contains_all):
            continue
        if contains_any and not any(tok.lower() in lc for tok in contains_any):
            continue
        return c
    return None


def canonical_model_name(x):
    s = str(x)
    sl = s.lower().replace("-", "").replace("_", "").replace(" ", "")
    if "efficientnet" in sl and "b0" in sl:
        return "EfficientNetB0"
    if "convnext" in sl and "tiny" in sl:
        return "ConvNeXtTiny"
    if "densenet" in sl and "121" in sl:
        return "DenseNet121"
    if "swintiny" in sl or ("swin" in sl and "tiny" in sl):
        return "SwinTiny"
    return s


def metric_bundle(y_true, y_score):
    y_true = np.asarray(y_true).astype(float)
    y_score = np.asarray(y_score).astype(float)
    mask = np.isfinite(y_true) & np.isfinite(y_score)
    y_true = y_true[mask].astype(int)
    y_score = y_score[mask]
    out = {
        "n_units": int(len(y_true)),
        "n_positive": int(np.sum(y_true == 1)),
        "n_negative": int(np.sum(y_true == 0)),
        "auroc": np.nan,
        "auprc": np.nan,
        "brier": np.nan,
    }
    if len(y_true) == 0:
        return out
    if len(np.unique(y_true)) >= 2:
        try:
            out["auroc"] = float(roc_auc_score(y_true, y_score))
        except Exception:
            out["auroc"] = np.nan
        try:
            out["auprc"] = float(average_precision_score(y_true, y_score))
        except Exception:
            out["auprc"] = np.nan
    if np.nanmin(y_score) >= 0 and np.nanmax(y_score) <= 1:
        try:
            out["brier"] = float(brier_score_loss(y_true, y_score))
        except Exception:
            out["brier"] = np.nan
    return out


def get_selected_path(source_map, role):
    block = source_map[source_map["role"].astype(str).eq(role)].copy()
    if block.empty:
        return None
    # Prefer selected rows, otherwise first existing row for robustness.
    if "selected" in block.columns:
        selected = block[block["selected"].astype(str).str.lower().isin(["true", "1", "yes"])]
        if not selected.empty:
            block = selected
    for _, row in block.iterrows():
        path = Path(str(row.get("path", "")))
        if path.exists():
            return path
    return None


def standardize_hygd_predictions(pred_df, base):
    """
    Return long-form prediction rows with:
    image_id, patient_id, gon_binary, model_name, score
    The function supports both long and wide prediction artifacts.
    """
    if pred_df is None or pred_df.empty:
        return pd.DataFrame(columns=["image_id", "patient_id", "gon_binary", "model_name", "score", "score_column"])

    df = pred_df.copy()
    image_col = find_col(df, ["image_id", "sample_id", "image", "fundus_id", "file_id"], contains_any=["image_id", "sample_id"])
    patient_col = find_col(df, ["patient_id", "subject_id", "patient"], contains_any=["patient", "subject"])
    label_col = find_col(
        df,
        ["gon_binary", "label_binary", "y_true", "target", "glaucoma_binary", "true_label"],
        contains_any=["gon", "label", "target", "glaucoma", "y_true"],
        exclude_any=["pred", "prob", "score"]
    )

    if image_col and image_col != "image_id":
        df = df.rename(columns={image_col: "image_id"})
    if patient_col and patient_col != "patient_id":
        df = df.rename(columns={patient_col: "patient_id"})
    if label_col and label_col != "gon_binary":
        df = df.rename(columns={label_col: "gon_binary"})

    if "image_id" in df.columns:
        df["image_id"] = df["image_id"].astype(str)
    else:
        return pd.DataFrame(columns=["image_id", "patient_id", "gon_binary", "model_name", "score", "score_column"])

    # Fill patient/label from base if missing.
    base_keep = base[["image_id", "patient_id", "gon_binary"]].drop_duplicates("image_id").copy()
    base_keep["image_id"] = base_keep["image_id"].astype(str)
    for c in ["patient_id", "gon_binary"]:
        if c not in df.columns:
            df = df.merge(base_keep[["image_id", c]], on="image_id", how="left")
        else:
            supplement = base_keep[["image_id", c]].rename(columns={c: f"{c}_base"})
            df = df.merge(supplement, on="image_id", how="left")
            df[c] = df[c].where(df[c].notna(), df[f"{c}_base"])
            df = df.drop(columns=[f"{c}_base"])

    long_rows = []

    model_col = find_col(df, ["model_name", "architecture", "model", "backbone"], contains_any=["model", "architecture", "backbone"])
    score_col = find_col(
        df,
        ["prob_gon", "prob_glaucoma", "p_glaucoma", "score_gon", "score", "probability", "y_score"],
        contains_any=["prob", "score", "p_glaucoma", "y_score"],
        exclude_any=["random", "mask", "delta", "auc", "ci"]
    )

    # Long format.
    if model_col and score_col:
        tmp = df[["image_id", "patient_id", "gon_binary", model_col, score_col]].copy()
        tmp = tmp.rename(columns={model_col: "model_name", score_col: "score"})
        tmp["model_name"] = tmp["model_name"].map(canonical_model_name)
        tmp["score_column"] = score_col
        long_rows.append(tmp)

    # Wide format.
    model_tokens = {
        "EfficientNetB0": ["efficientnetb0", "efficientnet_b0", "efficientnet"],
        "ConvNeXtTiny": ["convnexttiny", "convnext_tiny", "convnext"],
        "DenseNet121": ["densenet121", "densenet_121", "densenet"],
        "SwinTiny": ["swintiny", "swin_tiny"],
    }

    for col in df.columns:
        lc = str(col).lower().replace("-", "_")
        if not any(tok in lc for tok in ["prob", "score", "logit", "pred"]):
            continue
        if any(tok in lc for tok in ["random", "mask", "delta", "auc", "ci", "label"]):
            continue
        for model, toks in model_tokens.items():
            if any(tok in lc.replace("_", "") or tok in lc for tok in toks):
                tmp = df[["image_id", "patient_id", "gon_binary", col]].copy()
                tmp = tmp.rename(columns={col: "score"})
                tmp["model_name"] = model
                tmp["score_column"] = str(col)
                long_rows.append(tmp)
                break

    if not long_rows:
        return pd.DataFrame(columns=["image_id", "patient_id", "gon_binary", "model_name", "score", "score_column"])

    out = pd.concat(long_rows, ignore_index=True)
    out["score"] = pd.to_numeric(out["score"], errors="coerce")
    out["gon_binary"] = pd.to_numeric(out["gon_binary"], errors="coerce")
    out["patient_id"] = out["patient_id"].astype(str)
    out = out[np.isfinite(out["score"]) & out["gon_binary"].isin([0, 1])].copy()

    # Deduplicate if the same source was detected twice.
    out = out.drop_duplicates(["image_id", "model_name", "score_column"]).reset_index(drop=True)
    return out


base = safe_read_csv(HYGD_UNIFIED_BASE_CSV)
base["image_id"] = base["image_id"].astype(str)
base["patient_id"] = base["patient_id"].astype(str)
base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")

source_map = safe_read_csv(REVIEWER_SOURCE_MAP_CSV)
prediction_path = get_selected_path(source_map, "hygd_prediction_source")
prediction_df = safe_read_csv(prediction_path) if prediction_path is not None else pd.DataFrame()
pred_long = standardize_hygd_predictions(prediction_df, base)
pred_long.to_csv(V36_E2E_AUDIT_CSV, index=False)

rows = []

# -------------------------------------------------------------------------
# Morphometric baselines from OD/OC proxy.
# -------------------------------------------------------------------------

strict_flag = (
    base["strict_anatomical_proxy_valid"].astype(str).str.lower().isin(["true", "1", "yes", "y"])
    if "strict_anatomical_proxy_valid" in base.columns
    else pd.Series(True, index=base.index)
)

cdr_col = None
for c in ["area_cdr_proxy", "cup_to_disc_area_ratio", "cdr_proxy", "vertical_cdr_proxy"]:
    if c in base.columns:
        cdr_col = c
        break

morph_cols = [c for c in [
    cdr_col,
    "od_area_512",
    "oc_area_512",
    "rim_area_512",
    "quality_score",
    "od_confidence_mean",
    "oc_confidence_mean",
] if c and c in base.columns]

for col in morph_cols:
    df = base[strict_flag].copy()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df[df["gon_binary"].isin([0, 1]) & df[col].notna()].copy()
    if df.empty:
        continue

    mb = metric_bundle(df["gon_binary"], df[col])
    rows.append({
        "comparison_family": "proxy_morphometric_univariate",
        "model_or_feature": col,
        "level": "image",
        "score_definition": f"{col} used directly as a monotonic morphology score",
        "n_units": mb["n_units"],
        "n_positive": mb["n_positive"],
        "n_negative": mb["n_negative"],
        "auroc": mb["auroc"],
        "auprc": mb["auprc"],
        "brier": np.nan,
        "selection_or_training": "none; diagnostic ranking baseline only",
        "manuscript_interpretation": "Segmentation-first contextual baseline; not a replacement for AQPR because AQPR audits localization, functional dependence, calibration, quality, and transportability.",
    })

    pat = (
        df.groupby("patient_id")
        .agg(gon_binary=("gon_binary", "max"), score=(col, "mean"), n_images=("image_id", "nunique"))
        .reset_index()
    )
    mbp = metric_bundle(pat["gon_binary"], pat["score"])
    rows.append({
        "comparison_family": "proxy_morphometric_univariate",
        "model_or_feature": col,
        "level": "patient_mean",
        "score_definition": f"mean patient-level {col}",
        "n_units": mbp["n_units"],
        "n_positive": mbp["n_positive"],
        "n_negative": mbp["n_negative"],
        "auroc": mbp["auroc"],
        "auprc": mbp["auprc"],
        "brier": np.nan,
        "selection_or_training": "none; diagnostic ranking baseline only",
        "manuscript_interpretation": "Patient-level segmentation-first contextual baseline.",
    })

# Fixed shallow contextual baseline, patient-grouped CV, only if enough rows exist.
oof_df = pd.DataFrame()
if SKLEARN_AVAILABLE and cdr_col is not None:
    feature_cols = [c for c in [cdr_col, "od_area_512", "oc_area_512", "rim_area_512", "quality_score"] if c in base.columns]
    shallow = base[strict_flag].copy()
    for c in feature_cols:
        shallow[c] = pd.to_numeric(shallow[c], errors="coerce")
    shallow = shallow[shallow["gon_binary"].isin([0, 1])].copy()
    shallow = shallow.dropna(subset=feature_cols + ["patient_id", "gon_binary"])

    if shallow["patient_id"].nunique() >= 10 and shallow["gon_binary"].nunique() == 2 and len(feature_cols) >= 1:
        patient_label = shallow.groupby("patient_id")["gon_binary"].max()
        n_pos_pat = int((patient_label == 1).sum())
        n_neg_pat = int((patient_label == 0).sum())
        n_splits = int(min(5, n_pos_pat, n_neg_pat))
        if n_splits >= 2:
            oof_prob = np.full(len(shallow), np.nan)
            groups = shallow["patient_id"].astype(str).values
            gkf = GroupKFold(n_splits=n_splits)
            X = shallow[feature_cols].astype(float).values
            y = shallow["gon_binary"].astype(int).values

            for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups)):
                if len(np.unique(y[tr])) < 2:
                    continue
                clf = make_pipeline(
                    StandardScaler(),
                    LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced")
                )
                clf.fit(X[tr], y[tr])
                oof_prob[te] = clf.predict_proba(X[te])[:, 1]

            shallow = shallow.copy()
            shallow["morphometric_logreg_oof_prob"] = oof_prob
            oof_df = shallow[["image_id", "patient_id", "gon_binary", "morphometric_logreg_oof_prob"] + feature_cols].copy()
            oof_df.to_csv(V36_CDR_OOF_CSV, index=False)

            valid = shallow[np.isfinite(shallow["morphometric_logreg_oof_prob"])].copy()
            mbi = metric_bundle(valid["gon_binary"], valid["morphometric_logreg_oof_prob"])
            rows.append({
                "comparison_family": "proxy_morphometric_groupcv_shallow_model",
                "model_or_feature": "CDR+proxy_morphometrics+quality_logreg",
                "level": "image_oof_groupcv",
                "score_definition": f"fixed LogisticRegression with patient-grouped {n_splits}-fold OOF predictions; features={','.join(feature_cols)}",
                "n_units": mbi["n_units"],
                "n_positive": mbi["n_positive"],
                "n_negative": mbi["n_negative"],
                "auroc": mbi["auroc"],
                "auprc": mbi["auprc"],
                "brier": mbi["brier"],
                "selection_or_training": "patient-grouped CV contextual baseline; no hyperparameter search; not used for AQPR model selection",
                "manuscript_interpretation": "Shows how far simple proxy morphometrics go before applying AQPR to end-to-end classifiers.",
            })

            pat = (
                valid.groupby("patient_id")
                .agg(gon_binary=("gon_binary", "max"), score=("morphometric_logreg_oof_prob", "mean"), n_images=("image_id", "nunique"))
                .reset_index()
            )
            mbp = metric_bundle(pat["gon_binary"], pat["score"])
            rows.append({
                "comparison_family": "proxy_morphometric_groupcv_shallow_model",
                "model_or_feature": "CDR+proxy_morphometrics+quality_logreg",
                "level": "patient_mean_oof_groupcv",
                "score_definition": f"patient mean of patient-grouped {n_splits}-fold OOF probabilities",
                "n_units": mbp["n_units"],
                "n_positive": mbp["n_positive"],
                "n_negative": mbp["n_negative"],
                "auroc": mbp["auroc"],
                "auprc": mbp["auprc"],
                "brier": mbp["brier"],
                "selection_or_training": "patient-grouped CV contextual baseline; no hyperparameter search",
                "manuscript_interpretation": "Contextual segmentation-first baseline versus end-to-end audited classifiers.",
            })
        else:
            pd.DataFrame([{"status": "SKIPPED_INSUFFICIENT_PATIENT_CLASS_COUNTS"}]).to_csv(V36_CDR_OOF_CSV, index=False)
    else:
        pd.DataFrame([{"status": "SKIPPED_INSUFFICIENT_FEATURES_OR_ROWS"}]).to_csv(V36_CDR_OOF_CSV, index=False)
else:
    pd.DataFrame([{"status": "SKIPPED_SKLEARN_OR_CDR_UNAVAILABLE"}]).to_csv(V36_CDR_OOF_CSV, index=False)


# -------------------------------------------------------------------------
# End-to-end classifier comparison rows from available HYGD predictions.
# -------------------------------------------------------------------------

if not pred_long.empty:
    # Limit to plausible probability scores.
    pred_eval = pred_long.copy()
    pred_eval = pred_eval[pred_eval["score"].between(0, 1, inclusive="both")].copy()

    for model_name, g in pred_eval.groupby("model_name"):
        if g.empty:
            continue
        mbi = metric_bundle(g["gon_binary"], g["score"])
        rows.append({
            "comparison_family": "end_to_end_classifier",
            "model_or_feature": model_name,
            "level": "image",
            "score_definition": "frozen classifier probability from selected HYGD prediction source",
            "n_units": mbi["n_units"],
            "n_positive": mbi["n_positive"],
            "n_negative": mbi["n_negative"],
            "auroc": mbi["auroc"],
            "auprc": mbi["auprc"],
            "brier": mbi["brier"],
            "selection_or_training": "frozen classifier; no HYGD tuning",
            "manuscript_interpretation": "End-to-end comparator audited by AQPR.",
        })

        pat = (
            g.groupby("patient_id")
            .agg(gon_binary=("gon_binary", "max"), score=("score", "mean"), n_images=("image_id", "nunique"))
            .reset_index()
        )
        mbp = metric_bundle(pat["gon_binary"], pat["score"])
        rows.append({
            "comparison_family": "end_to_end_classifier",
            "model_or_feature": model_name,
            "level": "patient_mean",
            "score_definition": "patient mean of frozen classifier probabilities",
            "n_units": mbp["n_units"],
            "n_positive": mbp["n_positive"],
            "n_negative": mbp["n_negative"],
            "auroc": mbp["auroc"],
            "auprc": mbp["auprc"],
            "brier": mbp["brier"],
            "selection_or_training": "frozen classifier; no HYGD tuning",
            "manuscript_interpretation": "End-to-end comparator audited by AQPR.",
        })
else:
    # Fallback: if raw HYGD prediction rows are not parseable in this resume session,
    # still add the already generated frozen primary-model patient aggregation metrics
    # from Notebook 14 Cell 10. This makes the CDR/morphometric-vs-end-to-end
    # comparison usable for the reviewer while remaining transparent that these are
    # patient-level summary metrics rather than re-parsed image-level prediction rows.
    fallback_added = False
    patient_labels_for_counts = (
        base[strict_flag & base["gon_binary"].isin([0, 1])]
        .groupby("patient_id")["gon_binary"]
        .max()
    )
    fallback_n_patients = int(patient_labels_for_counts.shape[0])
    fallback_n_positive = int((patient_labels_for_counts == 1).sum())
    fallback_n_negative = int((patient_labels_for_counts == 0).sum())

    fallback_metric_paths = [
        TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.csv",
        TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_long.csv",
        RESULTS_14_DIR / "tables" / "notebook14_hygd_patient_aggregation_robustness_compact.csv",
    ]

    for fp in fallback_metric_paths:
        fp = Path(fp)
        if not fp.exists() or fp.stat().st_size == 0:
            continue
        agg_df = safe_read_csv(fp)
        if agg_df.empty or "auroc" not in agg_df.columns:
            continue

        model_col = "model_name" if "model_name" in agg_df.columns else None
        aggregation_col = "aggregation_method" if "aggregation_method" in agg_df.columns else None

        for _, rr in agg_df.iterrows():
            model_name = canonical_model_name(rr.get(model_col, PRIMARY_MODEL_NAME if "PRIMARY_MODEL_NAME" in globals() else "primary_model"))
            aggregation_name = str(rr.get(aggregation_col, "patient_summary"))
            auroc_value = pd.to_numeric(pd.Series([rr.get("auroc", np.nan)]), errors="coerce").iloc[0]
            brier_value = pd.to_numeric(pd.Series([rr.get("brier", np.nan)]), errors="coerce").iloc[0] if "brier" in agg_df.columns else np.nan
            n_pat_value = int(rr.get("n_patients", fallback_n_patients)) if pd.notna(rr.get("n_patients", np.nan)) else fallback_n_patients

            rows.append({
                "comparison_family": "end_to_end_classifier",
                "model_or_feature": model_name,
                "level": f"patient_{aggregation_name}",
                "score_definition": "frozen end-to-end classifier patient-level metric from Notebook 14 aggregation robustness summary",
                "n_units": n_pat_value,
                "n_positive": fallback_n_positive,
                "n_negative": fallback_n_negative,
                "auroc": float(auroc_value) if pd.notna(auroc_value) else np.nan,
                "auprc": np.nan,
                "brier": float(brier_value) if pd.notna(brier_value) else np.nan,
                "selection_or_training": "frozen classifier; no HYGD tuning; fallback summary metric because raw prediction source was not parseable in this extension cell",
                "manuscript_interpretation": "Primary end-to-end comparator for contextualizing proxy CDR/morphometric baselines; AQPR still evaluates localization, functional dependence, calibration, quality, and transportability.",
            })
            fallback_added = True

        if fallback_added:
            break

    if not fallback_added:
        rows.append({
            "comparison_family": "end_to_end_classifier",
            "model_or_feature": "unavailable",
            "level": "image/patient",
            "score_definition": "HYGD prediction source not found or not parseable by Cell 11, and no patient aggregation fallback table was found",
            "n_units": 0,
            "n_positive": 0,
            "n_negative": 0,
            "auroc": np.nan,
            "auprc": np.nan,
            "brier": np.nan,
            "selection_or_training": "not available",
            "manuscript_interpretation": "Use Notebook 15 claim-boundary / main model metrics if the selected prediction source is not parseable here.",
        })

baseline_table = pd.DataFrame(rows)

# Make display ordering stable.
order_map = {
    "proxy_morphometric_univariate": 0,
    "proxy_morphometric_groupcv_shallow_model": 1,
    "end_to_end_classifier": 2,
}
baseline_table["order"] = baseline_table["comparison_family"].map(order_map).fillna(99)
baseline_table = baseline_table.sort_values(["order", "model_or_feature", "level"]).drop(columns=["order"]).reset_index(drop=True)

save_csv_md(baseline_table, V36_CDR_BASELINE_CSV, V36_CDR_BASELINE_MD, max_rows=120)

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "14_hygd_smdg_domain_shift_audit",
    "cell": "Cell 11",
    "status": "PASS" if not baseline_table.empty else "PASS_WITH_EMPTY_OUTPUT",
    "cdr_column_used": cdr_col,
    "morphometric_feature_columns": morph_cols,
    "prediction_path_used": str(prediction_path) if prediction_path is not None else "",
    "n_standardized_prediction_rows": int(len(pred_long)),
    "end_to_end_fallback_rows": int(((baseline_table["comparison_family"] == "end_to_end_classifier") & (baseline_table["model_or_feature"] != "unavailable")).sum()) if "comparison_family" in baseline_table.columns and "model_or_feature" in baseline_table.columns else 0,
    "baseline_rows": int(len(baseline_table)),
    "outputs": {
        "baseline_csv": str(V36_CDR_BASELINE_CSV),
        "oof_predictions_csv": str(V36_CDR_OOF_CSV),
        "end_to_end_prediction_audit_csv": str(V36_E2E_AUDIT_CSV),
    },
    "interpretation_guardrail": "Proxy-morphometric baselines contextualize segmentation-first information. They do not replace AQPR and were not used for model selection.",
}
with open(V36_CELL11_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Cell 11 v3.6 proxy morphometric baseline summary:")
print(json.dumps(summary, indent=2))
print("\nBaseline/comparator table:")
print(baseline_table.to_string(index=False, max_rows=80))


In [ ]:

# Cell 12: v3.6 reviewer extension — outside-OD root-cause forensic analysis
#
# Reviewer target:
# - Deepen the outside-OD shortcut analysis without making causal claims.
# - Quantify simple non-OD acquisition/artifact cues: border darkness, corner brightness,
#   vignetting proxy, FOV fraction, peripheral statistics, and corner high-contrast proxy.
# - Create optional border/corner-sanitized inputs for future inference if needed.
#
# Outputs:
# - notebook14_v36_outside_od_root_cause_features.csv
# - notebook14_v36_outside_od_root_cause_univariate_auc.csv
# - notebook14_v36_outside_od_root_cause_top_features.csv
# - notebook14_v36_border_corner_sanitized_input_manifest.csv

from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

try:
    from PIL import Image
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    from sklearn.metrics import roc_auc_score
    SKLEARN_METRIC_AVAILABLE = True
except Exception as exc:
    SKLEARN_METRIC_AVAILABLE = False
    print(f"sklearn metric warning: {exc}")

required_vars = ["TABLE_DIR", "FIGURE_DATA_DIR", "RESULTS_14_DIR", "HYGD_UNIFIED_BASE_CSV"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

TABLE_DIR = Path(TABLE_DIR)
FIGURE_DATA_DIR = Path(FIGURE_DATA_DIR)
RESULTS_14_DIR = Path(RESULTS_14_DIR)
HANDOFF_DIR = RESULTS_14_DIR / "handoff"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DATA_DIR.mkdir(parents=True, exist_ok=True)
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

V36_OUTSIDE_FEATURES_CSV = TABLE_DIR / "notebook14_v36_outside_od_root_cause_features.csv"
V36_OUTSIDE_AUC_CSV = TABLE_DIR / "notebook14_v36_outside_od_root_cause_univariate_auc.csv"
V36_OUTSIDE_TOP_FEATURES_CSV = TABLE_DIR / "notebook14_v36_outside_od_root_cause_top_features.csv"
V36_OUTSIDE_TOP_FEATURES_MD = TABLE_DIR / "notebook14_v36_outside_od_root_cause_top_features.md"
V36_SANITIZED_MANIFEST_CSV = TABLE_DIR / "notebook14_v36_border_corner_sanitized_input_manifest.csv"
V36_OUTSIDE_FORENSIC_FIG = FIGURE_DATA_DIR / "notebook14_v36_outside_od_root_cause_top_feature_auc.png"
V36_CELL12_SUMMARY_JSON = RESULTS_14_DIR / "notebook14_v36_cell12_outside_od_root_cause_summary.json"

# Keep this bounded. It is a forensic root-cause probe, not a new model-training stage.
ROOT_CAUSE_OUTPUT_SIZE = 384
BORDER_FRACTION = 0.08
CORNER_FRACTION = 0.18
CENTER_FRACTION = 0.50
SANITIZED_INPUT_ROOT = RESULTS_14_DIR / "outside_od_border_corner_sanitized_inputs_384"
SANITIZED_INPUT_ROOT.mkdir(parents=True, exist_ok=True)


def safe_read_csv(path_like):
    path = Path(path_like)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, low_memory=False)
    except Exception as exc:
        print(f"Could not read CSV {path}: {exc}")
        return pd.DataFrame()


def save_csv_md(df, csv_path, md_path=None, max_rows=80):
    df.to_csv(csv_path, index=False)
    if md_path is not None:
        try:
            df.head(max_rows).to_markdown(md_path, index=False)
        except Exception:
            with open(md_path, "w", encoding="utf-8") as f:
                f.write(df.head(max_rows).to_string(index=False))
    return str(csv_path)


def read_rgb(path):
    path = Path(path)
    if CV2_AVAILABLE:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(path)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if PIL_AVAILABLE:
        return np.asarray(Image.open(path).convert("RGB"))
    raise RuntimeError("Neither cv2 nor PIL is available for image reading.")


def resize_rgb(rgb, size):
    if CV2_AVAILABLE:
        return cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    img = Image.fromarray(rgb.astype(np.uint8)).resize((size, size), Image.Resampling.LANCZOS)
    return np.asarray(img)


def save_rgb(path, rgb):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if CV2_AVAILABLE:
        cv2.imwrite(str(path), cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2BGR))
    else:
        Image.fromarray(rgb.astype(np.uint8)).save(path)
    return str(path)


def compute_fov_mask(rgb):
    gray = rgb.astype(np.float32).mean(axis=2)
    return gray > 10


def mask_from_slices(h, w, kind):
    yy, xx = np.mgrid[0:h, 0:w]
    mask = np.zeros((h, w), dtype=bool)

    bw = max(1, int(round(BORDER_FRACTION * min(h, w))))
    cw = max(1, int(round(CORNER_FRACTION * min(h, w))))

    if kind == "border":
        mask[:bw, :] = True
        mask[-bw:, :] = True
        mask[:, :bw] = True
        mask[:, -bw:] = True
    elif kind == "corners":
        mask[:cw, :cw] = True
        mask[:cw, -cw:] = True
        mask[-cw:, :cw] = True
        mask[-cw:, -cw:] = True
    elif kind == "center":
        ch = int(round(CENTER_FRACTION * h / 2))
        cw2 = int(round(CENTER_FRACTION * w / 2))
        cy, cx = h // 2, w // 2
        mask[max(0, cy-ch):min(h, cy+ch), max(0, cx-cw2):min(w, cx+cw2)] = True
    elif kind == "periphery":
        center = mask_from_slices(h, w, "center")
        mask = ~center
    return mask


def safe_mean(arr, mask):
    if mask is None:
        return np.nan
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(arr[mask]))


def safe_std(arr, mask):
    if mask is None:
        return np.nan
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return np.nan
    return float(np.std(arr[mask]))


def edge_proxy(gray):
    gray = gray.astype(np.float32)
    if CV2_AVAILABLE:
        g = cv2.GaussianBlur(gray.astype(np.uint8), (3, 3), 0)
        edges = cv2.Canny(g, 50, 150)
        return edges > 0
    gy, gx = np.gradient(gray)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    thr = np.nanpercentile(mag, 95)
    return mag >= thr


def feature_auc_table(features, label_col="gon_binary", patient_col="patient_id"):
    rows = []
    if features.empty or not SKLEARN_METRIC_AVAILABLE:
        return pd.DataFrame(columns=[
            "feature", "level", "n_units", "n_positive", "n_negative",
            "auroc", "informative_auc", "higher_feature_associated_with"
        ])

    exclude = {
        "image_id", "patient_id", "gon_binary", "image_path", "status",
        "quality_tertile", "split", "strict_anatomical_proxy_valid",
    }
    numeric_cols = []
    for c in features.columns:
        if c in exclude:
            continue
        s = pd.to_numeric(features[c], errors="coerce")
        if s.notna().sum() >= 20 and s.nunique(dropna=True) > 1:
            numeric_cols.append(c)

    for c in numeric_cols:
        df = features[[label_col, patient_col, c]].copy()
        df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
        df[c] = pd.to_numeric(df[c], errors="coerce")
        df = df[df[label_col].isin([0, 1]) & df[c].notna()].copy()

        for level in ["image", "patient_mean"]:
            if level == "patient_mean":
                eval_df = (
                    df.groupby(patient_col)
                    .agg(gon_binary=(label_col, "max"), score=(c, "mean"))
                    .reset_index()
                )
            else:
                eval_df = df.rename(columns={c: "score"})[[label_col, "score"]].rename(columns={label_col: "gon_binary"})

            y = eval_df["gon_binary"].astype(int).values
            score = eval_df["score"].astype(float).values
            if len(np.unique(y)) < 2:
                auc = np.nan
            else:
                try:
                    auc = float(roc_auc_score(y, score))
                except Exception:
                    auc = np.nan

            informative = max(auc, 1.0 - auc) if np.isfinite(auc) else np.nan
            direction = "GON+" if np.isfinite(auc) and auc >= 0.5 else "GON-"

            rows.append({
                "feature": c,
                "level": level,
                "n_units": int(len(y)),
                "n_positive": int(np.sum(y == 1)),
                "n_negative": int(np.sum(y == 0)),
                "auroc": auc,
                "informative_auc": informative,
                "higher_feature_associated_with": direction,
                "interpretation": "Univariate artifact/acquisition cue association; not a causal explanation of outside-OD classifier behavior.",
            })
    return pd.DataFrame(rows)


base = safe_read_csv(HYGD_UNIFIED_BASE_CSV)
base["image_id"] = base["image_id"].astype(str)
base["patient_id"] = base["patient_id"].astype(str)
base["gon_binary"] = pd.to_numeric(base["gon_binary"], errors="coerce")

if "image_path" not in base.columns:
    base["image_path"] = ""

strict_flag = (
    base["strict_anatomical_proxy_valid"].astype(str).str.lower().isin(["true", "1", "yes", "y"])
    if "strict_anatomical_proxy_valid" in base.columns
    else pd.Series(True, index=base.index)
)

work = base[base["gon_binary"].isin([0, 1]) & strict_flag].copy()
work["image_path"] = work["image_path"].fillna("").astype(str)

feature_rows = []
error_rows = []
sanitized_rows = []

if not (CV2_AVAILABLE or PIL_AVAILABLE):
    error_rows.append({"status": "SKIPPED_NO_IMAGE_LIBRARY"})
else:
    for _, row in work.iterrows():
        image_path = Path(str(row.get("image_path", "")))
        if not image_path.exists():
            error_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "image_path": str(image_path),
                "status": "missing_image",
            })
            continue
        try:
            rgb0 = read_rgb(image_path)
            rgb = resize_rgb(rgb0, ROOT_CAUSE_OUTPUT_SIZE).astype(np.uint8)
            h, w = rgb.shape[:2]
            gray = rgb.astype(np.float32).mean(axis=2)
            saturation = (rgb.astype(np.float32).max(axis=2) - rgb.astype(np.float32).min(axis=2))
            fov = compute_fov_mask(rgb)
            border = mask_from_slices(h, w, "border")
            corners = mask_from_slices(h, w, "corners")
            center = mask_from_slices(h, w, "center")
            periphery = mask_from_slices(h, w, "periphery")
            edges = edge_proxy(gray)

            border_fov = border & fov
            corner_fov = corners & fov
            center_fov = center & fov
            periphery_fov = periphery & fov

            # Sanitized image: replace corners and thin border with FOV mean color.
            sanitize_mask = (border | corners) & fov
            mean_color = rgb[fov].mean(axis=0) if fov.sum() > 0 else rgb.reshape(-1, 3).mean(axis=0)
            sanitized = rgb.copy().astype(np.float32)
            sanitized[sanitize_mask] = mean_color
            sanitized = np.clip(sanitized, 0, 255).astype(np.uint8)
            san_path = SANITIZED_INPUT_ROOT / f"{row['image_id']}_border_corner_sanitized.png"
            save_rgb(san_path, sanitized)

            feature_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "gon_binary": int(row.get("gon_binary")),
                "split": row.get("split", ""),
                "quality_tertile": row.get("quality_tertile", ""),
                "strict_anatomical_proxy_valid": row.get("strict_anatomical_proxy_valid", ""),
                "image_path": str(image_path),
                "fov_area_fraction": float(fov.mean()),
                "border_gray_mean": safe_mean(gray, border),
                "border_gray_std": safe_std(gray, border),
                "center_gray_mean": safe_mean(gray, center_fov),
                "periphery_gray_mean": safe_mean(gray, periphery_fov),
                "corner_gray_mean": safe_mean(gray, corners),
                "corner_gray_std": safe_std(gray, corners),
                "vignetting_center_minus_border": safe_mean(gray, center_fov) - safe_mean(gray, border_fov),
                "periphery_minus_center_gray": safe_mean(gray, periphery_fov) - safe_mean(gray, center_fov),
                "border_black_fraction": float(np.mean(gray[border] < 15)) if border.sum() > 0 else np.nan,
                "corner_black_fraction": float(np.mean(gray[corners] < 15)) if corners.sum() > 0 else np.nan,
                "corner_bright_fraction": float(np.mean(gray[corners] > 220)) if corners.sum() > 0 else np.nan,
                "corner_high_saturation_fraction": float(np.mean(saturation[corners] > 45)) if corners.sum() > 0 else np.nan,
                "corner_edge_fraction": float(np.mean(edges[corners])) if corners.sum() > 0 else np.nan,
                "border_edge_fraction": float(np.mean(edges[border])) if border.sum() > 0 else np.nan,
                "left_right_border_gray_asymmetry": safe_mean(gray, np.s_[:, :max(1, int(0.08*w))]) if False else np.nan,
            })

            # More explicit asymmetry calculations.
            r = feature_rows[-1]
            bw = max(1, int(round(BORDER_FRACTION * min(h, w))))
            top = np.zeros((h, w), dtype=bool); top[:bw, :] = True
            bottom = np.zeros((h, w), dtype=bool); bottom[-bw:, :] = True
            left = np.zeros((h, w), dtype=bool); left[:, :bw] = True
            right = np.zeros((h, w), dtype=bool); right[:, -bw:] = True
            r["top_bottom_border_gray_asymmetry"] = safe_mean(gray, top) - safe_mean(gray, bottom)
            r["left_right_border_gray_asymmetry"] = safe_mean(gray, left) - safe_mean(gray, right)

            sanitized_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "gon_binary": int(row.get("gon_binary")),
                "source_image_path": str(image_path),
                "border_corner_sanitized_image_path": str(san_path),
                "output_size": int(ROOT_CAUSE_OUTPUT_SIZE),
                "sanitized_region": "border_plus_corners_inside_estimated_fov",
                "sanitized_fraction_of_fov": float(sanitize_mask.sum() / max(1, fov.sum())),
                "purpose": "Optional future inference input to test whether border/corner artifact removal changes outside-OD behavior.",
            })
        except Exception as exc:
            error_rows.append({
                "image_id": row.get("image_id", ""),
                "patient_id": row.get("patient_id", ""),
                "image_path": str(image_path),
                "status": repr(exc),
            })

features = pd.DataFrame(feature_rows)
features.to_csv(V36_OUTSIDE_FEATURES_CSV, index=False)

san_manifest = pd.DataFrame(sanitized_rows)
san_manifest.to_csv(V36_SANITIZED_MANIFEST_CSV, index=False)

auc_table = feature_auc_table(features)
auc_table = auc_table.sort_values(["level", "informative_auc"], ascending=[True, False]).reset_index(drop=True)
auc_table.to_csv(V36_OUTSIDE_AUC_CSV, index=False)

top_features = (
    auc_table[auc_table["level"].eq("patient_mean")]
    .sort_values("informative_auc", ascending=False)
    .head(12)
    .reset_index(drop=True)
    if not auc_table.empty
    else pd.DataFrame()
)
save_csv_md(top_features, V36_OUTSIDE_TOP_FEATURES_CSV, V36_OUTSIDE_TOP_FEATURES_MD, max_rows=20)

# Compact figure for supplement.
try:
    import matplotlib.pyplot as plt
    if not top_features.empty:
        plot_df = top_features.sort_values("informative_auc", ascending=True).tail(10)
        fig, ax = plt.subplots(figsize=(7.0, 4.0))
        ax.barh(plot_df["feature"].astype(str), plot_df["informative_auc"].astype(float))
        ax.axvline(0.5, linestyle="--", linewidth=1)
        ax.set_xlabel("Informative AUROC, max(AUC, 1-AUC)")
        ax.set_title("HYGD non-OD artifact/acquisition cue association")
        fig.tight_layout()
        fig.savefig(V36_OUTSIDE_FORENSIC_FIG, dpi=220, bbox_inches="tight")
        plt.close(fig)
except Exception as exc:
    print(f"Could not generate outside-OD root-cause figure: {exc}")

summary = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "14_hygd_smdg_domain_shift_audit",
    "cell": "Cell 12",
    "status": "PASS" if not features.empty else "PASS_WITH_NO_FEATURES",
    "n_feature_rows": int(len(features)),
    "n_patients": int(features["patient_id"].nunique()) if not features.empty else 0,
    "n_errors": int(len(error_rows)),
    "top_patient_mean_features": top_features[["feature", "auroc", "informative_auc", "higher_feature_associated_with"]].head(8).to_dict("records") if not top_features.empty else [],
    "outputs": {
        "features_csv": str(V36_OUTSIDE_FEATURES_CSV),
        "univariate_auc_csv": str(V36_OUTSIDE_AUC_CSV),
        "top_features_csv": str(V36_OUTSIDE_TOP_FEATURES_CSV),
        "sanitized_manifest_csv": str(V36_SANITIZED_MANIFEST_CSV),
        "figure_png": str(V36_OUTSIDE_FORENSIC_FIG) if V36_OUTSIDE_FORENSIC_FIG.exists() else "",
    },
    "interpretation_guardrail": "These are univariate non-OD cue associations and sanitized input construction for root-cause forensics. They are not causal proof of what the outside-OD model uses.",
}
with open(V36_CELL12_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Cell 12 v3.6 outside-OD root-cause summary:")
print(json.dumps(summary, indent=2))
if not top_features.empty:
    print("\nTop patient-level non-OD cue associations:")
    print(top_features[["feature", "level", "n_units", "auroc", "informative_auc", "higher_feature_associated_with"]].to_string(index=False))
if error_rows:
    print("\nFirst image-feature extraction errors:")
    print(pd.DataFrame(error_rows).head(20).to_string(index=False))


In [ ]:

# Cell 13: v3.6 reviewer-question handoff for Notebook 16
#
# Purpose:
# - Consolidate the new Notebook 14 outputs needed for manuscript v3.6.
# - Make explicit which reviewer questions are answered by data, which are limitations,
#   and which require optional future work.

from pathlib import Path
import json
import pandas as pd
import numpy as np

required_vars = ["TABLE_DIR", "RESULTS_14_DIR"]
missing_vars = [v for v in required_vars if v not in globals()]
if missing_vars:
    raise RuntimeError(f"Run previous cells first. Missing variables: {missing_vars}")

TABLE_DIR = Path(TABLE_DIR)
RESULTS_14_DIR = Path(RESULTS_14_DIR)
HANDOFF_DIR = RESULTS_14_DIR / "handoff"
HANDOFF_DIR.mkdir(parents=True, exist_ok=True)

V36_CDR_BASELINE_CSV = TABLE_DIR / "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv"
V36_OUTSIDE_TOP_FEATURES_CSV = TABLE_DIR / "notebook14_v36_outside_od_root_cause_top_features.csv"
V36_OUTSIDE_AUC_CSV = TABLE_DIR / "notebook14_v36_outside_od_root_cause_univariate_auc.csv"
V36_SANITIZED_MANIFEST_CSV = TABLE_DIR / "notebook14_v36_border_corner_sanitized_input_manifest.csv"

V36_REVIEWER_HANDOFF_CSV = TABLE_DIR / "notebook14_v36_reviewer_question_handoff.csv"
V36_MANUSCRIPT_SNIPPETS_CSV = TABLE_DIR / "notebook14_v36_manuscript_snippets_for_notebook16.csv"
V36_FINAL_HANDOFF_JSON = HANDOFF_DIR / "notebook14_v36_handoff_for_notebook16.json"
V36_FINAL_HANDOFF_TXT = RESULTS_14_DIR / "notebook14_v36_FINAL_HANDOFF_FOR_NOTEBOOK16.txt"


def csv_ready(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return False, 0, []
    try:
        df = pd.read_csv(path, low_memory=False)
        return (len(df) > 0), int(len(df)), list(df.columns)
    except Exception:
        return False, 0, []


outputs = {
    "proxy_cdr_baseline": V36_CDR_BASELINE_CSV,
    "outside_od_root_cause_top_features": V36_OUTSIDE_TOP_FEATURES_CSV,
    "outside_od_root_cause_all_feature_auc": V36_OUTSIDE_AUC_CSV,
    "border_corner_sanitized_inputs": V36_SANITIZED_MANIFEST_CSV,
}

output_status = {}
for key, path in outputs.items():
    ready, n_rows, columns = csv_ready(path)
    output_status[key] = {
        "path": str(path),
        "ready": bool(ready),
        "n_rows": int(n_rows),
        "columns": columns[:80],
    }

handoff_rows = [
    {
        "reviewer_question": "Could you compare proxy CDR-based / segmentation-first morphometrics with frozen end-to-end classifiers on HYGD?",
        "notebook14_response": "Added proxy CDR and proxy-morphometric diagnostic baselines, including univariate image/patient AUROC and optional fixed patient-grouped CV logistic baseline when features are available.",
        "status": "answered_with_new_analysis" if output_status["proxy_cdr_baseline"]["ready"] else "pending_output",
        "primary_output": str(V36_CDR_BASELINE_CSV),
        "manuscript_use": "One compact sentence/table in Supplement. Keep main text conservative: CDR provides an anatomical contextual baseline but does not replace AQPR.",
    },
    {
        "reviewer_question": "The outside-OD-only signal is high. Which non-OD cues may contribute?",
        "notebook14_response": "Added hand-crafted non-OD acquisition/artifact cue analysis: border/corner brightness, vignetting proxy, FOV fraction, edge/corner marker proxies, and peripheral statistics, summarized by image- and patient-level univariate AUROC.",
        "status": "answered_with_new_forensic_analysis" if output_status["outside_od_root_cause_top_features"]["ready"] else "pending_output",
        "primary_output": str(V36_OUTSIDE_TOP_FEATURES_CSV),
        "manuscript_use": "Report as root-cause forensics and shortcut warning, not causal proof. Mention top cues if informative AUROC is meaningfully above 0.5.",
    },
    {
        "reviewer_question": "Have you tested masking text markers or vignetting explicitly?",
        "notebook14_response": "Constructed border/corner-sanitized inputs for optional future inference. Current v3.6 uses univariate cue association and sanitized-input manifest; it does not claim causal removal unless model inference is later run on sanitized images.",
        "status": "input_manifest_ready_not_causal_inference" if output_status["border_corner_sanitized_inputs"]["ready"] else "pending_output",
        "primary_output": str(V36_SANITIZED_MANIFEST_CSV),
        "manuscript_use": "Mention only if needed in Supplement as optional forensic material; avoid claiming performance after masking unless a downstream inference rerun is performed.",
    },
    {
        "reviewer_question": "Would you add manual HYGD OD/OC annotations?",
        "notebook14_response": "Not performed. Existing evidence remains cross-dataset segmenter validation, strict-valid QC, and proxy CDR face-validity. Manual target-domain masks remain future work.",
        "status": "address_as_limitation",
        "primary_output": "",
        "manuscript_use": "Limitations: no ophthalmologist-verified HYGD OD/OC subset.",
    },
    {
        "reviewer_question": "Did you test isotonic/source-aware calibration?",
        "notebook14_response": "Not added here. AQPR intentionally evaluates frozen AIROGS-light temperature scaling as a transportability diagnostic; source-aware recalibration is an intervention for future deployment adaptation.",
        "status": "address_as_scope_boundary",
        "primary_output": "",
        "manuscript_use": "Discussion/limitations, not a new experiment.",
    },
]

handoff = pd.DataFrame(handoff_rows)
handoff.to_csv(V36_REVIEWER_HANDOFF_CSV, index=False)

snippets = pd.DataFrame([
    {
        "section": "Results/Supplement",
        "candidate_text": "Proxy CDR and other OD/OC morphometrics were evaluated as segmentation-first contextual baselines on HYGD. These baselines quantify the standalone value of anatomy-derived measurements but do not replace AQPR, which audits localization, functional dependence, calibration, quality, patient aggregation, and transportability of frozen end-to-end classifiers.",
        "source_output": str(V36_CDR_BASELINE_CSV),
    },
    {
        "section": "Results/Supplement",
        "candidate_text": "Outside-OD root-cause forensics quantified simple non-OD acquisition/artifact cues, including border darkness, corner brightness, vignetting proxies, FOV fraction, and high-contrast corner structure. These analyses are interpreted as shortcut-risk indicators rather than causal proof of the specific features used by the outside-OD control.",
        "source_output": str(V36_OUTSIDE_TOP_FEATURES_CSV),
    },
    {
        "section": "Limitations",
        "candidate_text": "No ophthalmologist-verified HYGD OD/OC subset, alternative segmenter swap, or source-aware recalibration experiment was included. These remain natural extensions; AQPR is compatible with such additional slices and interventions when metadata and annotations are available.",
        "source_output": "",
    },
])
snippets.to_csv(V36_MANUSCRIPT_SNIPPETS_CSV, index=False)

final_handoff = {
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "14_hygd_smdg_domain_shift_audit",
    "version": "v3.6_reviewer_minor_revision_extension",
    "status": "READY_FOR_NOTEBOOK16" if all(
        output_status[k]["ready"] for k in ["proxy_cdr_baseline", "outside_od_root_cause_top_features"]
    ) else "PARTIAL_READY_CHECK_OUTPUTS",
    "outputs": output_status,
    "reviewer_handoff_csv": str(V36_REVIEWER_HANDOFF_CSV),
    "manuscript_snippets_csv": str(V36_MANUSCRIPT_SNIPPETS_CSV),
    "interpretation_guardrails": [
        "Proxy CDR baseline is contextual, not used for model selection.",
        "Outside-OD root-cause features are univariate association probes, not causal isolation of model mechanisms.",
        "Border/corner-sanitized images require separate model inference before any performance claim.",
        "Manual HYGD OD/OC validation and alternative segmenter swaps remain limitations/future work."
    ],
}
with open(V36_FINAL_HANDOFF_JSON, "w", encoding="utf-8") as f:
    json.dump(final_handoff, f, indent=2)

with open(V36_FINAL_HANDOFF_TXT, "w", encoding="utf-8") as f:
    f.write("Notebook 14 v3.6 handoff for Notebook 16\n")
    f.write("=" * 60 + "\n\n")
    f.write(json.dumps(final_handoff, indent=2))

print("Notebook 14 v3.6 handoff summary:")
print(json.dumps(final_handoff, indent=2))
print("\nReviewer-question handoff:")
print(handoff.to_string(index=False))
